# Classical AI Reasoning Agent



## Section 1: Setup
TEST_PATH is the private test file. If it does not exist, Section 12 builds it from ProverQA (public first-order
logic data with labels verified by the Prover9 prover). Section 12 also builds a logic-only public test from the
held-out public data and reports the accuracy on both files. RUN_EXPERIMENTS = False skips Section 10.

In [1]:
!pip install -q datasets pandas huggingface_hub

import json, ast, os, re, time, random, collections
from collections import defaultdict
import pandas as pd

SEED = 0
random.seed(SEED)

# Test files. Every test input is formal logic (knowledge base formulas + query formula).
TEST_PATH = "private_test.csv"                  # private test file (CSV / JSON / JSONL / parquet)
PRIVATE_LABELS_PATH = "private_test_labels.csv" # answers for the private test, if available
USE_PROVERQA_AS_PRIVATE_TEST = True             # if TEST_PATH is missing, build it from ProverQA (public FOL data)
PROVERQA_PER_LEVEL = 100                        # ProverQA questions per level (easy / medium / hard)
PUBLIC_TEST_PATH = "public_test.csv"            # logic-only test built from held-out public data
PUBLIC_LABELS_PATH = "public_test_labels.csv"
SUBMISSION_PATH = "submission.csv"              # output: id,label (predictions for the private test)

RUN_EXPERIMENTS = True                         # True: also run the Section 10 experiments (several minutes)
RESULTS = {}                                    # evaluation results are collected here

print("Setup done | TEST_PATH:", TEST_PATH, "| RUN_EXPERIMENTS:", RUN_EXPERIMENTS)

Setup done | TEST_PATH: private_test.csv | RUN_EXPERIMENTS: True


## Section 2: Data Exploration and Splits
### 2a: Loading
RuleTaker (tasksource/ruletaker), FOLIO v2 (yale-nlp/FOLIO, gated: requires a Hugging Face token, otherwise the
public FOLIO v0.0 file on GitHub is used) and CounterBench (CounterBench/CounterBench, four JSON files with two
schemas). If a dataset cannot be loaded, only the cells that need it are skipped.

In [2]:
# FOLIO is gated: store a READ token in Colab "Secrets" under the name HF_TOKEN.
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in to Hugging Face")
except Exception as e:
    print("No Hugging Face login (FOLIO falls back to GitHub v0.0):", type(e).__name__)

Logged in to Hugging Face


In [3]:
def try_load(fn, name):
    """Try to load a dataset. On failure the notebook does not stop; it only prints a warning."""
    try:
        out = fn()
        print(f"[ok]   {name}")
        return out
    except Exception as e:
        print(f"[warn] {name} could not be loaded: {type(e).__name__}: {str(e)[:150]}")
        return None


def hf_splits(repo):
    from datasets import load_dataset
    ds = load_dataset(repo)
    return {split: ds[split].to_pandas() for split in ds}


def load_counterbench():
    from huggingface_hub import hf_hub_download
    repo = "CounterBench/CounterBench"

    def read_json(fname):
        path = hf_hub_download(repo, fname, repo_type="dataset")
        with open(path, encoding="utf-8") as f:
            try:
                return json.load(f)
            except json.JSONDecodeError:
                f.seek(0)
                return [json.loads(line) for line in f if line.strip()]

    parts = []
    for q_file, m_file, tag in [("data_balanced_alpha_V1.json", "meta_model_alpha_V1.json", "alpha"),
                                ("data_balanced_backdoor_V2.json", "meta_model_backdoor_V2.json", "backdoor")]:
        questions = pd.json_normalize(read_json(q_file))
        models = pd.DataFrame(read_json(m_file))
        merged = questions.merge(models, left_on="meta.model_id", right_on="model_id", how="left")
        parts.append(merged.assign(src=tag))
    return pd.concat(parts, ignore_index=True)


ruletaker = try_load(lambda: hf_splits("tasksource/ruletaker"), "RuleTaker")
folio = try_load(lambda: hf_splits("yale-nlp/FOLIO"), "FOLIO (Hugging Face v2)")
if folio is None:
    url = "https://raw.githubusercontent.com/Yale-LILY/FOLIO/main/data/v0.0/folio-validation.jsonl"
    folio = try_load(lambda: {"validation": pd.read_json(url, lines=True)}, "FOLIO (GitHub v0.0 fallback)")
counterbench = try_load(load_counterbench, "CounterBench")

for name, d in [("RuleTaker", ruletaker), ("FOLIO", folio)]:
    if d is not None:
        print(f"{name:12s}", {split: len(df) for split, df in d.items()})
if counterbench is not None:
    print(f"{'CounterBench':12s}", len(counterbench), "rows")

HAVE_RT = isinstance(ruletaker, dict) and "test" in ruletaker
HAVE_FOLIO = isinstance(folio, dict) and "validation" in folio          # the GitHub fallback has validation only
HAVE_CB = counterbench is not None
print("Data:", {"RuleTaker": HAVE_RT, "FOLIO": HAVE_FOLIO, "CounterBench": HAVE_CB})

[ok]   RuleTaker
[ok]   FOLIO (Hugging Face v2)
[ok]   CounterBench
RuleTaker    {'train': 480152, 'test': 151911, 'dev': 75872}
FOLIO        {'train': 1001, 'validation': 203}
CounterBench 1200 rows
Data: {'RuleTaker': True, 'FOLIO': True, 'CounterBench': True}


### 2b: Tune and report splits


In [4]:
# Tune split  = data used for exploration and for choosing settings.
# Report split = data used only to report accuracy. The two never share a context, story or problem.
CB_HELDOUT_FRACTION = 0.5        # CounterBench: share of problems held out for reporting


def counterbench_split(df, heldout=CB_HELDOUT_FRACTION, seed=SEED):
    """Every CounterBench problem appears 5 times with different names (nonsense_0..4). All copies of a problem go
    to the same side (otherwise the held-out split would contain seen problems), and every type is split evenly."""
    qid = pd.to_numeric(df["question_id"], errors="coerce") if "question_id" in df.columns else None
    if qid is not None and "src" in df.columns and qid.notna().all():  # question_id may be loaded as text
        group = df["src"].astype(str) + "_" + (qid.astype(int) % 200).astype(str)
    else:                                                               # no grouping information: one group per question
        group = pd.Series(range(len(df)), index=df.index).astype(str)
    split = pd.Series("tune", index=df.index)
    rng = random.Random(seed)
    for t in sorted(df["type"].unique()):
        groups = sorted(group[df["type"] == t].unique())
        rng.shuffle(groups)
        held = set(groups[:round(len(groups) * heldout)])
        split[(df["type"] == t) & group.isin(held)] = "heldout"
    return split, group


# ---- RuleTaker: tune = dev split (or train); report = test split ----
RT_TUNE_DF = RT_REPORT_DF = None
RT_TUNE_NAME = None
if HAVE_RT:
    rt_all = {k: v[~v["config"].astype(str).str.contains("NatLang")] for k, v in ruletaker.items()}
    RT_REPORT_DF = rt_all["test"]
    RT_TUNE_NAME = next((s for s in ("dev", "validation", "valid", "train") if s in rt_all), None)
    if RT_TUNE_NAME:
        RT_TUNE_DF = rt_all[RT_TUNE_NAME]
    else:                                         # only a test split is available: split it by context
        contexts = sorted(RT_REPORT_DF["context"].unique())
        random.Random(SEED).shuffle(contexts)
        tune_ctx = set(contexts[: len(contexts) // 2])
        RT_TUNE_DF = RT_REPORT_DF[RT_REPORT_DF["context"].isin(tune_ctx)]
        RT_REPORT_DF = RT_REPORT_DF[~RT_REPORT_DF["context"].isin(tune_ctx)]
        RT_TUNE_NAME = "half of test (by context)"
    shared = set(RT_TUNE_DF["context"]) & set(RT_REPORT_DF["context"])
    if shared:                                    # never tune on a context that is also reported
        RT_TUNE_DF = RT_TUNE_DF[~RT_TUNE_DF["context"].isin(shared)]
    RT_OVERLAP = len(shared)

# ---- FOLIO: tune = train; report = validation ----
FOLIO_OVERLAP = 0
if HAVE_FOLIO and "train" in folio:
    tr, va = folio["train"], folio["validation"]
    if "story_id" in tr.columns and "story_id" in va.columns:
        shared = set(tr["story_id"]) & set(va["story_id"])
        same = tr["story_id"].isin(shared)
    else:
        key = lambda df: df["premises-FOL"].map(str)
        shared = set(key(tr)) & set(key(va))
        same = key(tr).isin(shared)
    FOLIO_OVERLAP = int(same.sum())
    if FOLIO_OVERLAP:                             # never tune on a story that is also reported
        folio["train"] = tr[~same]

# ---- CounterBench: tune half / held-out half, split by problem ----
if HAVE_CB:
    counterbench["split"], counterbench["problem"] = counterbench_split(counterbench)
    tune_p = set(counterbench.loc[counterbench.split == "tune", "problem"])
    held_p = set(counterbench.loc[counterbench.split == "heldout", "problem"])
    CB_OVERLAP = len(tune_p & held_p)

rows = []
if HAVE_RT:
    rows.append({"dataset": "RuleTaker", "tune split": RT_TUNE_NAME, "tune rows": len(RT_TUNE_DF),
                 "report split": "test", "report rows": len(RT_REPORT_DF),
                 "shared contexts removed from tune": RT_OVERLAP})
if HAVE_FOLIO:
    rows.append({"dataset": "FOLIO", "tune split": "train" if "train" in folio else "-",
                 "tune rows": len(folio.get("train", [])), "report split": "validation",
                 "report rows": len(folio["validation"]), "shared contexts removed from tune": FOLIO_OVERLAP})
if HAVE_CB:
    rows.append({"dataset": "CounterBench", "tune split": "tune half", "tune rows": int((counterbench.split == "tune").sum()),
                 "report split": "held-out half", "report rows": int((counterbench.split == "heldout").sum()),
                 "shared contexts removed from tune": CB_OVERLAP})
if rows:
    print(pd.DataFrame(rows).to_string(index=False))
    print("\nOverlap check: tune and report splits share no context / story / problem.")
else:
    print("No public data loaded.")

     dataset tune split  tune rows  report split  report rows  shared contexts removed from tune
   RuleTaker        dev      56463          test       113101                                  0
       FOLIO      train       1001    validation          203                                  0
CounterBench  tune half        600 held-out half          600                                  0

Overlap check: tune and report splits share no context / story / problem.


### 2c: Exploration (tune splits only)
Size, label balance and structure of each dataset. Of the report splits only the size is shown.

In [5]:
if HAVE_RT or HAVE_FOLIO or HAVE_CB:
    # Exploration uses the tune splits only; of the report splits only the size is shown.
    _RULE = re.compile(r"\bthen\b|^All\b|\b(things|people|someone|something)\b", re.I)
    _OPS = {"∀": "forall", "∃": "exists", "∧": "and", "∨": "or", "→": "implies", "↔": "iff", "⊕": "xor", "¬": "not"}


    def _sentences(text):
        return [s.strip() for s in re.split(r"(?<=\.)\s+", str(text)) if s.strip()]


    def _formula_list(v):
        return list(v) if not isinstance(v, str) else [l for l in v.split("\n") if l.strip()]


    if HAVE_RT:
        d = RT_TUNE_DF
        print(f"===== RuleTaker  (tune split: {RT_TUNE_NAME}, {len(d)} questions; report split: test, {len(RT_REPORT_DF)} questions)")
        print("Depth configs :", d["config"].value_counts().sort_index().to_dict())
        print("Labels        :", d["label"].value_counts().to_dict())
        ctx = d.drop_duplicates("context")["context"]
        sents = ctx.map(_sentences)
        n_rules = sents.map(lambda ss: sum(bool(_RULE.search(s)) for s in ss))
        n_facts = sents.map(len) - n_rules
        neg = sum(" not " in f" {s} " for ss in sents for s in ss) / max(1, sum(map(len, sents)))
        print(f"Contexts      : {len(ctx)} | facts per context: mean {n_facts.mean():.1f}, max {n_facts.max()} | "
              f"rules per context: mean {n_rules.mean():.1f}, max {n_rules.max()} | sentences with 'not': {100 * neg:.1f}%")
        absent = []                                   # 'X is not Y.' where Y never occurs in the context
        for r in d.itertuples():
            m = re.match(r"^(?:The )?(\w+) is not (\w+)\.$", str(r.question).strip())
            if m and not re.search(rf"\b{re.escape(m.group(2))}\b", str(r.context), re.I):
                absent.append(r.label)
        if absent:
            c = collections.Counter(absent)
            print(f"Closed-world evidence: {len(absent)} negative questions about an attribute absent from the context; "
                  f"labels {dict(c)}")

    if HAVE_FOLIO:
        tr = folio.get("train")
        print(f"\n===== FOLIO  (tune split: train, {0 if tr is None else len(tr)} questions; "
              f"report split: validation, {len(folio['validation'])} questions)")
        if tr is not None and len(tr):
            print("Labels        :", tr["label"].value_counts().to_dict())
            if "story_id" in tr.columns:
                print("Stories       :", tr["story_id"].nunique())
            prem = tr["premises-FOL"].map(_formula_list)
            print(f"Premises      : mean {prem.map(len).mean():.1f}, max {prem.map(len).max()} per question")
            text = "\n".join("\n".join(p) for p in prem) + "\n" + "\n".join(tr["conclusion-FOL"].map(str))
            print("Operators     :", {name: text.count(sym) for sym, name in _OPS.items()})
            non_horn = prem.map(lambda ps: any(s in f for f in ps for s in "∨⊕↔∃")).mean()
            print(f"Questions whose premises use ∨, ⊕, ↔ or ∃ (not expressible as Horn rules): {100 * non_horn:.1f}%")

    if HAVE_CB:
        t = counterbench[counterbench.split == "tune"]
        print(f"\n===== CounterBench  (tune split: {len(t)} questions; report split: held-out, "
              f"{int((counterbench.split == 'heldout').sum())} questions)")
        print("Types         :", t["type"].value_counts().to_dict())
        print("Answers       :", t["answer"].value_counts().to_dict())
        n_vars = t["given_info"].map(lambda g: len(set(re.findall(r"\b[A-Z][a-z]+\b", str(g))) - {"We"}))
        print(f"Variables per causal graph: mean {n_vars.mean():.1f}, max {n_vars.max()}")
        print("Copies per problem (renamed variables):", t.groupby("problem").size().value_counts().to_dict())
        ex = t.iloc[0]
        print("Example given_info:", ex["given_info"])
        print("Example question  :", ex["question"], "->", ex["answer"])
else:
    print('Skipped: dataset exploration (data missing)')

===== RuleTaker  (tune split: dev, 56463 questions; report split: test, 113101 questions)
Depth configs : {'depth-0': 8718, 'depth-1': 8870, 'depth-2': 8906, 'depth-3': 8896, 'depth-3ext': 11587, 'depth-5': 9486}
Labels        : {'entailment': 28431, 'not entailment': 28032}
Contexts      : 6012 | facts per context: mean 8.4, max 16 | rules per context: mean 4.8, max 9 | sentences with 'not': 13.9%

===== FOLIO  (tune split: train, 1001 questions; report split: validation, 203 questions)
Labels        : {'True': 388, 'Uncertain': 324, 'False': 289}
Stories       : 340
Premises      : mean 5.3, max 10 per question
Operators     : {'forall': 3240, 'exists': 1031, 'and': 5635, 'or': 529, 'implies': 3186, 'iff': 10, 'xor': 569, 'not': 1942}
Questions whose premises use ∨, ⊕, ↔ or ∃ (not expressible as Horn rules): 74.4%

===== CounterBench  (tune split: 600 questions; report split: held-out, 600 questions)
Types         : {'basic': 125, 'conditional': 125, 'joint': 125, 'nested': 125, 'bac

In [6]:
if HAVE_CB:
    # CounterBench text patterns (tune split only): which sentence and question templates occur.
    cb_tune = counterbench[counterbench.split == "tune"]
    for t in sorted(cb_tune["type"].unique()):
        ex = cb_tune[cb_tune["type"] == t].iloc[0]
        print(f"[{t}] GIVEN: {ex['given_info']}\n       QUESTION: {ex['question']}  ->  {ex['answer']}")


    def to_pattern(text, mapping):
        """Replace the variable names (Ziklo, Blaf, ...) by their role: X (cause), Y (effect), V (other)."""
        if isinstance(mapping, dict):
            for role, name in mapping.items():
                if name:
                    tag = role if role in ("X", "Y") else "V"
                    text = re.sub(rf"\b{re.escape(str(name))}\b", tag, text)
        return re.sub(r"^We know that\s+", "", str(text).strip())


    clause_pats, question_pats = collections.Counter(), collections.Counter()
    for _, ex in cb_tune.iterrows():
        mapping = ex["variable_mapping"] if "variable_mapping" in cb_tune.columns else None
        g = to_pattern(ex["given_info"], mapping)
        for part in re.split(r",\s*(?:and\s+)?|\.\s+|\.$", g):
            if part.strip():
                clause_pats[part.strip()] += 1
        question_pats[(ex["type"], to_pattern(ex["question"], mapping))] += 1

    print("\nMost frequent clause patterns in given_info:")
    for p, c in clause_pats.most_common(15):
        print(f"{c:6d}  {p}")
    print("\nQuestion patterns (with type):")
    for (t, p), c in question_pats.most_common(10):
        print(f"{c:6d}  [{t}]  {p}")
else:
    print('Skipped: CounterBench patterns (data missing)')

[backdoor] GIVEN: We know that Trune causes Ziklo, Ziklo causes Blaf, Blaf or Trune cause Vork, and Vork causes Lumbo. Trune~Bern(0.7). We observed Blaf occur.
       QUESTION: Would Lumbo occur if not Ziklo instead of Ziklo?  ->  yes
[basic] GIVEN: We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.
       QUESTION: Would Lumbo occur if not Ziklo instead of Ziklo?  ->  no
[conditional] GIVEN: We know that Ziklo or Blaf causes Trune, Trune causes Vork, Vork causes Lumbo. We observed Blaf.
       QUESTION: Would Lumbo occur if not Ziklo instead of Ziklo?  ->  yes
[joint] GIVEN: We know that Ziklo causes Blaf and Vork, Blaf causes not Trune, Trune or Vork causes Lumbo.
       QUESTION: Would Lumbo occur if not Ziklo and not Blaf?  ->  yes
[nested] GIVEN: We know that Ziklo causes Blaf, Trune and Vork, not Vork causes Sline, Sline causes Frim, Frim or Vork causes Qado, Qado causes Jurf, and Jurf causes Lumbo.
       QUESTION: Assume not Ziklo, and 

### 2d: Findings and design decisions



## Section 3: Internal Representation


Tuples are immutable and hashable, so facts and clauses can be stored in sets and dictionaries
(constant-time lookup, automatic de-duplication, caching).

In [7]:
def is_var(t):
    """A variable is a string starting with '?' (e.g. '?x1')."""
    return isinstance(t, str) and t.startswith("?")


def term_str(t):
    """Readable string for a term."""
    if isinstance(t, tuple):                                  # function term: ('f', (args))
        return f"{t[0]}({', '.join(term_str(a) for a in t[1])})"
    if is_var(t):                                             # '?x12' -> 'x'
        m = re.match(r"\?([A-Za-z]+)", t)
        return m.group(1) if m else "x" + t[1:].replace("'", "")
    return t                                                  # constant


def lit_str(lit):
    """Literal (positive, pred, args) -> 'Smart(bob)' ya '¬Smart(bob)'."""
    pos, pred, args = lit
    s = pred + (f"({', '.join(term_str(a) for a in args)})" if args else "")
    return s if pos else "¬" + s


def fml_str(f):
    """Readable logic notation for a formula (nested tuple)."""
    op = f[0]
    if op == "atom":
        return lit_str((True, f[1], f[2]))
    if op == "not":
        return "¬" + fml_str(f[1])
    if op in ("and", "or"):
        sym = " ∧ " if op == "and" else " ∨ "
        return "(" + sym.join(fml_str(g) for g in f[1:]) + ")"
    if op in ("imp", "iff", "xor"):
        sym = {"imp": " → ", "iff": " ↔ ", "xor": " ⊕ "}[op]
        return "(" + fml_str(f[1]) + sym + fml_str(f[2]) + ")"
    if op in ("forall", "exists"):
        return ("∀" if op == "forall" else "∃") + term_str(f[1]) + " " + fml_str(f[2])
    if op == "true":
        return "⊤"
    if op == "false":
        return "⊥"
    raise ValueError(f"Unknown formula type: {op}")

In [8]:
young_bob = ("atom", "Young", ("bob",))
kind_bob  = ("atom", "Kind", ("bob",))
rule1 = ("forall", "?x1", ("imp", ("atom", "Young", ("?x1",)),
                                  ("atom", "Smart", ("?x1",))))
rule2 = ("forall", "?x2", ("imp", ("and", ("atom", "Smart", ("?x2",)), ("atom", "Kind", ("?x2",))),
                                  ("atom", "Successful", ("?x2",))))
query = ("atom", "Successful", ("bob",))

for f in [young_bob, kind_bob, rule1, rule2, query]:
    print(f"{fml_str(f):45s}  <- {f}")

facts = {young_bob, kind_bob, young_bob}        # the duplicate is removed automatically
print("\nFacts in the set:", len(facts))
print("Negative literal:", lit_str((False, "Smart", ("dave",))))
print("Variable?", is_var("?x1"), "| Constant?", not is_var("bob"))

Young(bob)                                     <- ('atom', 'Young', ('bob',))
Kind(bob)                                      <- ('atom', 'Kind', ('bob',))
∀x (Young(x) → Smart(x))                       <- ('forall', '?x1', ('imp', ('atom', 'Young', ('?x1',)), ('atom', 'Smart', ('?x1',))))
∀x ((Smart(x) ∧ Kind(x)) → Successful(x))      <- ('forall', '?x2', ('imp', ('and', ('atom', 'Smart', ('?x2',)), ('atom', 'Kind', ('?x2',))), ('atom', 'Successful', ('?x2',))))
Successful(bob)                                <- ('atom', 'Successful', ('bob',))

Facts in the set: 2
Negative literal: ¬Smart(dave)
Variable? True | Constant? True


## Section 4: Formal Logic Parser
The notation of the test data is not specified, so the parser accepts many dialects and maps all of them to
the same internal form.



**Conventions.** Precedence `↔ < → < ⊕ < ∨ < ∧ < ¬` (→ is right-associative). `∀x (...)` covers only the bracket
unless x is used after it; otherwise the scope extends as far right as possible. Free single-letter variables
in the knowledge base are universally quantified; in Prolog/TPTP every capitalised argument is a variable.
Constants and predicates are case-insensitive. Parsing is strict first; only if that fails is a tolerant mode
used (missing operator read as ∧, bracket repair). A formula that cannot be parsed is dropped and recorded,
the rest of the knowledge base is kept.

In [9]:
import re, json, ast


class ParseError(Exception):
    """The formula could not be parsed (invalid syntax)."""


# LaTeX command -> Unicode symbol
_LATEX = {
    r"\forall": "∀", r"\exists": "∃",
    r"\neg": "¬", r"\lnot": "¬", r"\sim": "¬",
    r"\land": "∧", r"\wedge": "∧",
    r"\lor": "∨", r"\vee": "∨",
    r"\rightarrow": "→", r"\to": "→", r"\implies": "→", r"\Rightarrow": "→", r"\supset": "→",
    r"\leftarrow": "←", r"\Leftarrow": "←",
    r"\leftrightarrow": "↔", r"\iff": "↔", r"\Leftrightarrow": "↔", r"\equiv": "↔",
    r"\oplus": "⊕", r"\veebar": "⊕", r"\xor": "⊕",
    r"\top": "⊤", r"\bot": "⊥", r"\neq": "≠", r"\ne": "≠",
}
_LATEX_RE = re.compile(r"\\[A-Za-z]+")

_NUMBERING = re.compile(r"^\s*(?:\(?\d+[.):]\s+|\d+[.)](?=[^\d\s])|(?:premise|fact|rule|axiom|p(?=\d))\s*\d*\s*:\s*)",
                        re.I)
_LABEL = re.compile(r"^\s*(?:query|question|conclusion|statement|hypothesis|goal|prove|kb|context)\s*[:=]\s*",
                    re.I)
_RT_TRIPLE = re.compile(r'\(\s*"([^"]+)"\s+"([^"]+)"\s+"([^"]+)"\s+"([+-])"\s*\)')
_RT_VARS = {"something", "someone", "it", "they", "people", "things"}


def _latex_to_unicode(s):
    s = s.replace("$", "")
    s = re.sub(r"\\(left|right|big|Big)\b", "", s)
    s = re.sub(r"\\[,;:! ]", " ", s)

    def repl(m):
        cmd = m.group(0)
        if cmd in _LATEX:
            return " " + _LATEX[cmd] + " "
        raise ParseError(f"Unknown LaTeX command {cmd}")
    return _LATEX_RE.sub(repl, s)


def _ruletaker_triples(s):
    """RuleTaker native triple format: ("Bob" "is" "big" "+")  ->  Big(bob)."""
    def lit(m):
        subj, rel, obj, sign = m.groups()
        subj = "x" if subj.lower() in _RT_VARS else subj
        if rel.lower() == "is":
            atom = f"{obj.capitalize()}({subj})"
        else:
            atom = f"{rel.capitalize()}({subj}, {obj})"
        return ("¬" if sign == "-" else "") + atom
    out = _RT_TRIPLE.sub(lit, s)
    return re.sub(r"\)\s+(?=[¬A-Z])", ") ∧ ", out)      # (A) (B) -> (A) ∧ (B)


# FLD (Formal Logic Deduction) notation: {A}{a} = A(a), {A}x = A(x), {A} = proposition A,
# (x): = ∀x, (Ex): = ∃x, v = ∨.
_FLD = re.compile(r"\{[A-Z]{1,3}\}")


def _fld_to_standard(s):
    s = re.sub(r"\(E([a-z])\)\s*:", r"∃\1 ", s)                 # (Ex):  -> ∃x
    s = re.sub(r"\(([a-z])\)\s*:", r"∀\1 ", s)                  # (x):   -> ∀x
    s = re.sub(r"\{([A-Z]+)\}\{([a-z]+)\}", r"\1(\2)", s)       # {A}{a} -> A(a)
    s = re.sub(r"\{([A-Z]+)\}([a-z])\b", r"\1(\2)", s)           # {A}x   -> A(x)
    s = re.sub(r"\{([A-Z]+)\}", r"\1", s)                       # {A}    -> A
    return re.sub(r"(?<=[\s)])v(?=[\s(¬])", "∨", s)              # ' v '  -> ∨


# ---------- further dialects: z3py, Pyke (Logic-LM), SMT-LIB / S-expression, set notation, Lean ----------
_KEYWORDS = {"not", "and", "or", "xor", "implies", "iff", "if", "then", "true", "false", "forall", "exists",
             "all", "some", "every", "each", "fun"}


def _match_paren(s, i):
    """s[i] == '(' -> index of the matching ')' (-1 if none)."""
    depth = 0
    for j in range(i, len(s)):
        if s[j] == "(":
            depth += 1
        elif s[j] == ")":
            depth -= 1
            if depth == 0:
                return j
    return -1


def _split_top(s):
    """'a, f(b, c), d' -> ['a', 'f(b, c)', 'd']  (splits on top-level commas only)."""
    parts, depth, cur = [], 0, ""
    for ch in s:
        if ch in "([":
            depth += 1
        elif ch in ")]":
            depth -= 1
        if ch == "," and depth == 0:
            parts.append(cur)
            cur = ""
        else:
            cur += ch
    return parts + [cur]


# z3py:  ForAll([x], Implies(Young(x), Smart(x)))   And(a, b)   Or(..)   Not(a)   Xor(a, b)   s.add(..)
_Z3_OPS = re.compile(r"\b(ForAll|Exists|Implies|And|Or|Not|Xor)\s*\(")
_Z3_DECL = re.compile(r"^\s*[\w\s,]+=\s*(?:Consts?|Functions?|DeclareSort|Bools?|Ints?|Reals?|EnumSort|"
                      r"Datatype|Solver)\s*\(|^\s*\w+\s*=\s*\w+\.create\(\)|^\s*(?:from|import)\s")


def _z3_to_infix(e):
    e = e.strip()
    m = _Z3_OPS.match(e)
    if not m or _match_paren(e, m.end() - 1) != len(e) - 1:
        return _z3_calls(e)                                   # '(And(..) ↔ Or(..))': convert the inner calls
    op, args = m.group(1), [a.strip() for a in _split_top(e[m.end():-1])]
    conv = [_z3_to_infix(a) for a in args]
    if op in ("ForAll", "Exists"):
        q = "∀" if op == "ForAll" else "∃"
        vs = [v.strip() for v in args[0].strip("[] ").split(",") if v.strip()]
        return "".join(f"{q}{v} " for v in vs) + f"({_z3_to_infix(','.join(args[1:]))})"
    if op == "Implies":
        return f"({conv[0]} → {conv[1]})"
    if op == "Not":
        return f"¬({conv[0]})"
    sym = {"And": " ∧ ", "Or": " ∨ ", "Xor": " ⊕ "}[op]
    return "(" + sym.join(conv) + ")"


def _z3_rewrite(s):
    """Rewrite every z3 call (also nested) in infix form; strip wrappers such as s.add(..) / prove(..)."""
    m = re.match(r"^\s*(?:\w+\.add|prove|solve)\s*\((.*)\)\s*$", s, re.S)
    if m:
        s = m.group(1)
    s = re.sub(r"(?<![\w)])(\w+)\s*==\s*(\w+)(?![\w(])", r"\1 = \2", s)      # x == y   : equality (terms)
    s = re.sub(r"(?<![\w)])(\w+)\s*!=\s*(\w+)(?![\w(])", r"\1 ≠ \2", s)
    s = s.replace("==", "↔").replace("!=", "⊕")                            # A == B   : formulas -> iff / xor
    return _z3_calls(s)


def _z3_calls(s):
    out, i = "", 0
    while True:
        m = _Z3_OPS.search(s, i)
        if not m:
            return out + s[i:]
        j = _match_paren(s, m.end() - 1)
        if j < 0:
            return out + s[i:]
        out += s[i:m.start()] + _z3_to_infix(s[m.start():j + 1])
        i = j + 1


# Pyke (Logic-LM):  Young($x, True) >>> Smart($x, False)    Young(Bob, True) ::: Bob is young.
_PYKE = re.compile(r">>>|,\s*(?:True|False)\s*\)|\$[a-z]\w*\s*[,)]")


def _pyke_to_standard(s):
    s = re.sub(r"\$([a-z]\w*)", r"\1", s)
    s = s.replace(">>>", "→").replace("&&", "∧").replace("||", "∨")
    s = re.sub(r"(\w+)\(([^()]*?),\s*False\s*\)", r"¬\1(\2)", s)
    return re.sub(r"(\w+)\(([^()]*?),\s*True\s*\)", r"\1(\2)", s)


# SMT-LIB / S-expression:  (assert (forall ((x U)) (=> (Young x) (Smart x))))   (Young bob)
_SEXPR_OPS = {"and", "or", "not", "xor", "implies", "iff", "=>", "<=>", "="}


def _sexpr_parse(s):
    toks = re.findall(r"\(|\)|[^\s()]+", s)
    pos = 0

    def node():
        nonlocal pos
        t = toks[pos]
        pos += 1
        if t != "(":
            return t
        out = []
        while toks[pos] != ")":
            out.append(node())
        pos += 1
        return out
    tree = node()
    if pos != len(toks):
        raise ParseError("Extra text after the S-expression")
    return tree


def _looks_sexpr(s):
    if not s.startswith("(") or re.search(r"[,∧∨→¬↔⊕∀∃&|~!]|->|:-|\w\(", s):
        return False
    try:
        tree = _sexpr_parse(s)
    except Exception:
        return False

    def ok(t):                                  # '(A and B)' is infix, not an S-expression
        if isinstance(t, str):
            return True
        if not t or not isinstance(t[0], str):
            return t == [] or all(ok(x) for x in t)
        return not any(isinstance(x, str) and x.lower() in _SEXPR_OPS for x in t[1:]) and all(ok(x) for x in t[1:])
    return isinstance(tree, list) and ok(tree)


def _sexpr_to_infix(t):
    if isinstance(t, str):
        return {"true": "⊤", "false": "⊥"}.get(t.lower(), t)
    if not t:
        return ""
    head = t[0] if isinstance(t[0], str) else None
    h = (head or "").lower()
    args = t[1:]
    if h == "assert":
        return _sexpr_to_infix(args[0])
    if h.startswith(("declare-", "define-", "set-", "get-")) or h in ("check-sat", "exit", "push", "pop", "reset"):
        return ""
    if h in ("forall", "exists"):
        b = args[0]
        vs = [b] if isinstance(b, str) else [x if isinstance(x, str) else x[0] for x in b]
        q = "∀" if h == "forall" else "∃"
        return "".join(f"{q}{v} " for v in vs) + f"({_sexpr_to_infix(args[1])})"
    c = [_sexpr_to_infix(a) for a in args]
    if h in ("=>", "implies"):
        out = c[-1]
        for a in reversed(c[:-1]):
            out = f"({a} → {out})"
        return out
    if h in ("and", "or", "xor"):
        return "(" + {"and": " ∧ ", "or": " ∨ ", "xor": " ⊕ "}[h].join(c) + ")"
    if h == "not":
        return f"¬({c[0]})"
    if h in ("iff", "<=>") or (h == "=" and all(isinstance(a, list) or a[:1].isupper() for a in args)):
        return f"({c[0]} ↔ {c[1]})"
    if h == "=":
        return f"{c[0]} = {c[1]}"
    if head is None:
        raise ParseError("The head of an S-expression must be a name")
    return f"{head}({', '.join(c)})" if c else head


# Set notation:  bob ∈ Young   (a, b) ∈ Likes   bob ∉ Young   Young ⊆ Smart   ∀x ∈ Dog (..)
_ELEM = r"(\([^()]*\)|[\w']+)"


def _set_to_standard(s):
    def bounded(s):
        m = re.search(r"([∀∃])\s*([a-z]\w*)\s*∈\s*([A-Za-z]\w*)\s*([,:.]?)\s*", s)
        if not m:
            return s
        q, v, S, sep = m.groups()
        rest = s[m.end():]
        if not sep and rest.startswith("("):
            j = _match_paren(rest, 0)
            body, tail = (rest[:j + 1], rest[j + 1:]) if j >= 0 else (rest, "")
        else:
            body, tail = rest, ""
        link = "→" if q == "∀" else "∧"
        return s[:m.start()] + f"{q}{v} ({S}({v}) {link} {bounded(body)})" + bounded(tail)
    s = bounded(s)
    s = re.sub(rf"{_ELEM}\s*∉\s*([A-Za-z]\w*)", lambda m: f"¬{m.group(2)}({m.group(1).strip('()')})", s)
    s = re.sub(rf"{_ELEM}\s*∈\s*([A-Za-z]\w*)", lambda m: f"{m.group(2)}({m.group(1).strip('()')})", s)
    return re.sub(r"\b([A-Za-z]\w*)\s*⊆\s*([A-Za-z]\w*)", r"∀v (\1(v) → \2(v))", s)


# Lean / type theory:  ∀ x, Young x → Smart x     ∀ (x : Person), ..     ∃ x y, Likes x y
def _lean_to_standard(s):
    def binder(m):
        vs = re.sub(r"\(\s*([\w\s]+?)\s*:\s*\w+\s*\)", r"\1", m.group(2))
        vs = re.sub(r":\s*\w+\s*$", "", vs.strip())
        return "".join(f"{m.group(1)}{v}." for v in vs.split())
    s = re.sub(r"([∀∃])\s*((?:\(\s*[\w\s]+:\s*\w+\s*\)\s*|[^\W\d]\w*\s*)+(?::\s*\w+\s*)?),", binder, s)
    toks = re.findall(r"\w(?:[\w'’+]|-(?![>-])(?=\w))*|\s+|.", s)
    out, i, n = [], 0, len(toks)
    while i < n:
        t = toks[i]
        prev = next((x for x in reversed(out) if not x.isspace()), "")
        is_id = re.match(r"[^\W\d]", t) and t.lower() not in _KEYWORDS
        if is_id and prev not in ("∀", "∃"):
            j, args = i + 1, []
            while j + 1 < n and toks[j].isspace() and re.match(r"\w", toks[j + 1]) and \
                    toks[j + 1].lower() not in _KEYWORDS:
                args.append(toks[j + 1])
                j += 2
            if args:
                out.append(f"{t}({', '.join(args)})")
                i = j
                continue
        out.append(t)
        i += 1
    return "".join(out)


def _more_dialects(s):
    """Rewrite the additional dialects into the standard syntax. Each rewrite fires only on its own markers."""
    if ":::" in s:                                                  # Pyke/Logic-LM comment
        s = s.split(":::")[0].strip()
    if re.fullmatch(r"(?:predicates|facts|rules|query|premises|conclusion)\s*:?", s, re.I) or \
            re.fullmatch(r"\w+\([^()]*\bbool\s*\)", s) or _Z3_DECL.match(s):
        return ""                                                   # declaration / header line
    if s.startswith("(") and _looks_sexpr(s):
        return _sexpr_to_infix(_sexpr_parse(s))
    if _Z3_OPS.search(s) or re.match(r"^\s*(?:\w+\.add|prove|solve)\s*\(", s):
        s = _z3_rewrite(s)
    if _PYKE.search(s):
        s = _pyke_to_standard(s)
    if re.search(r"[∈∉⊆]", s):
        s = _set_to_standard(s)
    if not re.search(r"\w\(", s) and re.search(r"[^\W\d]\w*[ \t]+\w", s) and \
            not re.search(r"\bcauses?\b", s):
        s = _lean_to_standard(s)
    return s


def normalize_formula_text(s):
    """Rewrite one formula string from any supported dialect into the common syntax."""
    s = s.strip()
    if '"' in s and _RT_TRIPLE.search(s):
        s = _ruletaker_triples(s)
    else:
        if "{" in s and "\\" not in s and _FLD.search(s):
            s = _fld_to_standard(s)
        s = _more_dialects(s)
    s = re.sub(r"^\s*\d*\.?\d+\s*::\s*", "", s)          # ProbLog probability: 0.8::P
    s = _NUMBERING.sub("", s)                            # "1. P(a)", "Premise 2: ..."
    s = _LABEL.sub("", s)                                # "Query: P(a)"
    m = re.match(r"^\s*(?:fof|cnf|tff|thf)\s*\(\s*[^,]+,\s*[^,]+,(.*)\)\s*\.?\s*$", s, re.S)
    if m:                                                # TPTP: fof(name, role, FORMULA).
        s = m.group(1)
    m = re.match(r"^\s*query\s*\((.*)\)\s*\.?\s*$", s, re.S)
    if m:                                                # ProbLog: query(P).
        s = m.group(1)
    s = re.sub(r"^\s*\?-\s*", "", s)                     # Prolog query: ?- P.
    if "\\" in s and _LATEX_RE.search(s.replace("\\+", "").replace("\\/", "")):
        s = _latex_to_unicode(s)
    s = re.sub(r"!\s*\[([^\]]*)\]\s*:", lambda m: "∀" + m.group(1) + " ", s)    # TPTP ![X,Y]:
    s = re.sub(r"\?\s*\[([^\]]*)\]\s*:", lambda m: "∃" + m.group(1) + " ", s)   # TPTP ?[X]:
    s = re.sub(r"\$true\b", "⊤", s)
    s = re.sub(r"\$false\b", "⊥", s)
    s = s.replace("$", "")                               # LaTeX math mode: $P(a)$
    s = re.sub(r"\s+", " ", s).strip()
    if s.endswith("."):
        s = s[:-1].rstrip()
    return s


_QUANT_TAIL = re.compile(r"(?:[∀∃]|\b(?:forall|exists|all|some)\b)\s*[\w\s,]*$", re.I)


def _split_on_periods(text):
    """'P(a). Q(a). ∀x. (R(x))'  ->  ['P(a)', 'Q(a)', '∀x. (R(x))']
    Split only on top-level '.' (not inside brackets, and not on the dot of '∀x.')."""
    parts, depth, start = [], 0, 0
    for i, ch in enumerate(text):
        if ch in "([":
            depth += 1
        elif ch in ")]":
            depth = max(depth - 1, 0)
        elif ch == "." and depth == 0:
            nxt = text[i + 1] if i + 1 < len(text) else " "
            before = text[start:i].strip()
            if (nxt.isspace() and not _QUANT_TAIL.search(before)
                    and not re.fullmatch(r"\(?\d+\)?", before)):    # '1. P(a)' is numbering
                parts.append(text[start:i])
                start = i + 1
    parts.append(text[start:])
    return [p.strip() for p in parts if p.strip()]


def split_kb(kb):
    """Knowledge base in any form (list, JSON list, string separated by newlines, ';' or '.')
    -> list of formula strings."""
    if kb is None:
        return []
    if not isinstance(kb, str):
        try:
            items = [str(x) for x in kb]
        except TypeError:
            items = [str(kb)]
    else:
        text = kb.strip()
        items = None
        if text.startswith("["):
            for loader in (json.loads, ast.literal_eval):
                try:
                    val = loader(text)
                    if isinstance(val, (list, tuple)):
                        items = [str(x) for x in val]
                        break
                except Exception:
                    pass
        if items is None:
            items = [text]
    out = []
    for item in items:
        for line in re.split(r"[\n;]+", item):
            out.extend(_split_on_periods(line))
    return [s for s in out if s.strip()]

In [10]:
# (kind, text) tokens. Longer symbols are checked first, so that '<->' is not read as '<' + '-' + '>'.
_SYMBOLS = [
    ("<==>", "IFF"), ("<=>", "IFF"), ("<->", "IFF"), ("<~>", "XOR"),
    ("==>", "IMP"), ("=>", "IMP"), ("->", "IMP"), ("—>", "IMP"), ("–>", "IMP"), ("-->", "IMP"),
    ("<-", "RIMP"), (":-", "RIMP"), ("<=", "RIMP"),
    ("&&", "AND"), ("||", "OR"), ("/\\", "AND"), ("\\/", "OR"), ("\\+", "NOT"),
    ("!=", "NEQ"), ("\\=", "NEQ"),
    ("∀", "ALL"), ("∃", "EX"),
    ("¬", "NOT"), ("~", "NOT"), ("∼", "NOT"), ("!", "NOT"), ("-", "NOT"), ("−", "NOT"),
    ("∧", "AND"), ("&", "AND"), ("^", "AND"), ("·", "AND"),
    ("∨", "OR"), ("|", "OR"),
    ("→", "IMP"), ("⇒", "IMP"), ("⊃", "IMP"), ("⟶", "IMP"),
    ("←", "RIMP"), ("⇐", "RIMP"),
    ("↔", "IFF"), ("⟷", "IFF"), ("⇔", "IFF"), ("≡", "IFF"), ("⟺", "IFF"),
    ("⊕", "XOR"), ("⊻", "XOR"),
    ("⊤", "TRUE"), ("⊥", "FALSE"),
    ("≠", "NEQ"), ("==", "EQ"), ("=", "EQ"),
    ("(", "LP"), (")", "RP"), ("[", "LP"), ("]", "RP"), ("{", "LP"), ("}", "RP"),
    (",", "COMMA"), (":", "COLON"), (".", "DOT"),
]

# Words that are operators (case-insensitive). Quantifier words are handled separately by the parser.
_WORD_OPS = {"not": "NOT", "and": "AND", "or": "OR", "xor": "XOR", "implies": "IMP",
             "iff": "IFF", "if": "IF", "then": "THEN", "true": "TRUE", "false": "FALSE",
             "forall": "ALL", "exists": "EX"}
_QUANT_WORDS = {"all": "ALL", "some": "EX", "every": "ALL", "each": "ALL"}

# Identifier: starts with a letter, digit or _; may contain hyphens (gpt-3), plus (c++), apostrophes and
# a dot between two word characters (u.s). The '-' of '->' is never part of an identifier.
_IDENT = re.compile(r"\w(?:[\w'’+]|-(?![>-])(?=\w)|\.(?=\w))*")
_QUOTED = re.compile(r"'([^']*)'|\"([^\"]*)\"")


def tokenize(s):
    """Text -> tokens, plus the 'glued' set: indices of ID tokens directly followed (without a space)
    by '(', as in 'P(x)'. This distinguishes '∀x y (..)' from '∀x P(x)'."""
    toks, glued, i, n = [], set(), 0, len(s)
    while i < n:
        c = s[i]
        if c.isspace():
            i += 1
            continue
        m = _QUOTED.match(s, i)
        if m:                                           # 'Bob' ya "the bald eagle" -> constant
            toks.append(("QID", m.group(1) if m.group(1) is not None else m.group(2)))
            i = m.end()
            continue
        m = _IDENT.match(s, i)
        if m:
            word = m.group(0)
            low = word.lower()
            if low in _WORD_OPS:
                toks.append((_WORD_OPS[low], word))
            elif low in _QUANT_WORDS:
                toks.append(("QWORD", word))            # may be 'all x' or the name 'all'
            else:
                if m.end() < n and s[m.end()] == "(":
                    glued.add(len(toks))
                toks.append(("ID", word))
            i = m.end()
            continue
        for sym, kind in _SYMBOLS:
            if s.startswith(sym, i):
                toks.append((kind, sym))
                i += len(sym)
                break
        else:
            raise ParseError(f"Unknown symbol {c!r} at position {i} in: {s}")
    toks.append(("EOF", ""))
    return toks, glued

In [11]:
print("1) Dialect normalizer:")
for s in ["fof(a2, axiom, ![X]: (young(X) => smart(X))).",
          r"$\forall x\, (Young(x) \rightarrow Smart(x))$",
          "0.9::smart(X) :- young(X).",
          "?- smart(bob).",
          "Premise 2: ∀x (Young(x) → Smart(x))",
          '((("someone" "is" "young" "+")) -> ("someone" "is" "smart" "+"))']:
    print(f"   {s:62s} ->  {normalize_formula_text(s)}")

print("\n2) KB splitter:")
for kb in ["Young(bob). Kind(bob). ∀x. (Young(x) → Smart(x)).",
           "1. Young(bob)\n2. ∀x (Young(x) → Smart(x))",
           '["Young(bob)", "Kind(bob)"]',
           "Young(bob); Kind(bob)"]:
    print(f"   {kb!r:58s} ->  {split_kb(kb)}")

print("\n3) Tokenizer:")
toks, glued = tokenize("∀x y (Likes(x, y) -> ¬Hates(gpt-3, y))")
print("  ", [t for t in toks])
print("   glued (name directly followed by '('):", sorted(glued))

print("\n4) Unknown symbol raises a clear error:")
try:
    tokenize("Young(bob) % Kind(bob)")
except ParseError as e:
    print("   ParseError:", e)

1) Dialect normalizer:
   fof(a2, axiom, ![X]: (young(X) => smart(X))).                  ->  ∀X (young(X) => smart(X))
   $\forall x\, (Young(x) \rightarrow Smart(x))$                  ->  ∀ x (Young(x) → Smart(x))
   0.9::smart(X) :- young(X).                                     ->  smart(X) :- young(X)
   ?- smart(bob).                                                 ->  smart(bob)
   Premise 2: ∀x (Young(x) → Smart(x))                            ->  ∀x (Young(x) → Smart(x))
   ((("someone" "is" "young" "+")) -> ("someone" "is" "smart" "+")) ->  ((Young(x)) -> Smart(x))

2) KB splitter:
   'Young(bob). Kind(bob). ∀x. (Young(x) → Smart(x)).'        ->  ['Young(bob)', 'Kind(bob)', '∀x. (Young(x) → Smart(x))']
   '1. Young(bob)\n2. ∀x (Young(x) → Smart(x))'               ->  ['1. Young(bob)', '2. ∀x (Young(x) → Smart(x))']
   '["Young(bob)", "Kind(bob)"]'                              ->  ['Young(bob)', 'Kind(bob)']
   'Young(bob); Kind(bob)'                                    ->  ['Youn

In [12]:
_SINGLE_LETTER_VARS = set("xyzuvwXYZUVW")


class _Parser:
    """Grammar (from weakest to strongest binding):
         iff  :=  imp  ( ↔ imp )*
         imp  :=  xor  ( → imp  |  ← imp )?          (→ right-associative)
         xor  :=  disj ( ⊕ disj )*
         disj :=  conj ( ∨ conj )*
         conj :=  unary ( (∧ | ,) unary )*
         unary:=  ¬ unary | quantifier | ( formula ) | if A then B | ⊤ | ⊥ | atom
       Quantifier scope: '∀x (...)' covers only the bracket unless x is used after it; otherwise it extends right."""

    def __init__(self, text, counter, upper_vars=False, tolerant=False):
        self.toks, self.glued = tokenize(text)
        self.p = 0
        self.scope = []              # [(source_name, unique_var)]
        self.counter = counter       # globally unique variable numbers
        self.upper_vars = upper_vars # Prolog/TPTP: a capitalised name is a variable
        self.tolerant = tolerant     # read a missing operator ('P(a) Q(a)') as '∧'
        self.free = {}               # variables without a quantifier

    # ---- helpers ----
    def peek(self, k=0):
        return self.toks[min(self.p + k, len(self.toks) - 1)][0]

    def take(self, kind=None):
        k, v = self.toks[self.p]
        if kind and k != kind:
            raise ParseError(f"Expected {kind}, got {k} {v!r}")
        self.p += 1
        return v

    def new_var(self, name):
        self.counter[0] += 1
        return f"?{name}{self.counter[0]}"

    # ---- grammar ----
    def parse(self):
        f = self.iff()
        while self.peek() == "DOT":
            self.take()
        if self.peek() != "EOF":
            raise ParseError(f"Unexpected {self.toks[self.p]} after a complete formula")
        return f

    def iff(self):
        left = self.imp()
        while self.peek() == "IFF":
            self.take()
            left = ("iff", left, self.imp())
        return left

    def imp(self):
        left = self.xor()
        if self.peek() == "IMP":
            self.take()
            return ("imp", left, self.imp())
        if self.peek() == "RIMP":                      # B ← A   ==   A → B
            self.take()
            return ("imp", self.imp(), left)
        return left

    def xor(self):
        left = self.disj()
        while self.peek() == "XOR":
            self.take()
            left = ("xor", left, self.disj())
        return left

    def disj(self):
        parts = [self.conj()]
        while self.peek() == "OR":
            self.take()
            parts.append(self.conj())
        return parts[0] if len(parts) == 1 else ("or", *parts)

    def _starts_formula(self):
        return self.peek() in ("ID", "QID", "NOT", "LP", "ALL", "EX", "QWORD", "TRUE", "FALSE")

    def conj(self):
        parts = [self.unary()]
        while True:
            if self.peek() in ("AND", "COMMA"):
                self.take()
            elif self.tolerant and self.toks[self.p - 1][0] == "RP" and self._starts_formula():
                pass                                    # tolerant: 'P(a) Q(a)' -> P(a) ∧ Q(a)
            else:
                break
            parts.append(self.unary())
        return parts[0] if len(parts) == 1 else ("and", *parts)

    @staticmethod
    def _looks_like_var(name):
        return len(name) <= 2 and name[0].isalpha()

    def _is_quantifier_here(self):
        if self.peek() in ("ALL", "EX"):
            return True
        # 'all x (...)', 'some y P(y)': a quantifier word followed by a short variable name
        return (self.peek() == "QWORD" and self.peek(1) == "ID"
                and self._looks_like_var(self.toks[self.p + 1][1]))

    @staticmethod
    def _q_kind(word):
        return "forall" if word.lower() in ("∀", "forall", "all", "every", "each") else "exists"

    def _var_list(self):
        names = [self.take("ID")]
        while True:                                     # ∀x,y   ∀x y
            if self.peek() == "COMMA" and self.peek(1) == "ID":
                self.take()
                names.append(self.take("ID"))
            elif (self.peek() == "ID" and self._looks_like_var(self.toks[self.p][1])
                  and self.p not in self.glued):
                names.append(self.take("ID"))
            else:
                return names

    def _group_end(self):
        """If the current token is '(', the index of its matching ')', otherwise None."""
        if self.peek() != "LP":
            return None
        depth, j = 0, self.p
        while self.toks[j][0] != "EOF":
            depth += {"LP": 1, "RP": -1}.get(self.toks[j][0], 0)
            if depth == 0:
                return j
            j += 1
        return None

    def _quantified_body(self, q, names):
        uniq = []
        for nm in names:
            v = self.new_var(nm.lower())                # '?x12'  (X and x are both shown as 'x')
            self.scope.append((nm, v))
            uniq.append(v)
        # Scope rule:
        #  the variable is not used after '∀x (...)'        -> the scope is the bracket only
        #       ¬∃x (P(x)) ∧ Q(a)   =   (¬∃x P(x)) ∧ Q(a)
        #  otherwise (no bracket, or the variable is used later) -> the scope extends as far right as possible
        #       ∀x P(x) → Q(x)      =   ∀x (P(x) → Q(x))
        end = self._group_end()
        used_later = end is not None and any(
            k == "ID" and v in names for k, v in self.toks[end + 1:])
        if end is not None and not used_later:
            body = self.unary()                         # the bracketed part only
        else:
            body = self.iff()                           # maximal scope
        for _ in names:
            self.scope.pop()
        for v in reversed(uniq):
            body = (q, v, body)
        return body

    def quantifier(self):
        q = self._q_kind(self.take())
        if self.peek() != "ID":                         # '∃(P(x) ∧ Q(x))': the variable was omitted
            before = set(self.free)
            body = self.unary()
            for nm in [n for n in self.free if n not in before]:
                body = (q, self.free.pop(nm), body)     # the body's free variables belong to this quantifier
            return body
        names = self._var_list()
        if self.peek() in ("COLON", "DOT"):             # ∀x: P(x)   ∀x. P(x)
            self.take()
        return self._quantified_body(q, names)

    def _bracketed_quantifier(self):
        """(∀x)(P(x) → Q(x))  or  (∀x, y) ... : a quantifier enclosed in brackets."""
        if self.peek(1) not in ("ALL", "EX", "QWORD"):
            return None
        j = self.p + 2
        while self.toks[j][0] in ("ID", "COMMA"):
            j += 1
        if self.toks[j][0] != "RP" or j == self.p + 2:
            return None
        self.take("LP")
        q = self._q_kind(self.take())
        names = self._var_list()
        self.take("RP")
        return self._quantified_body(q, names)

    def unary(self):
        k = self.peek()
        if k == "NOT":
            self.take()
            return ("not", self.unary())
        if self._is_quantifier_here():
            return self.quantifier()
        if k == "LP":
            f = self._bracketed_quantifier()
            if f is not None:
                return f
            self.take()
            f = self.iff()
            self.take("RP")
            return f
        if k == "IF":                                   # if A then B
            self.take()
            a = self.iff()
            self.take("THEN")
            return ("imp", a, self.iff())
        if k == "TRUE":
            self.take()
            return ("true",)
        if k == "FALSE":
            self.take()
            return ("false",)
        return self.atom()

    # ---- terms and atoms ----
    def _name(self):
        if self.peek() in ("ID", "QWORD", "TRUE", "FALSE"):   # 'all', 'some', 'false' may also be names
            return self.take()
        raise ParseError(f"Expected a name, got {self.toks[self.p]}")

    def _args(self):
        """Arguments after '(' up to ')'. A trailing comma ('f(a, b,)') is accepted.
        Tolerant mode: a missing ')' followed by an operator ('In(a, b ∧ ...') closes the list there."""
        args = [self.term()]
        while self.peek() == "COMMA":
            self.take()
            if self.peek() == "RP":
                break
            args.append(self.term())
        if self.tolerant and self.peek() in ("AND", "OR", "IMP", "IFF", "XOR", "EOF"):
            return tuple(args)
        self.take("RP")
        return tuple(args)

    def term(self):
        if self.peek() == "QID":                        # 'Bob': a quoted name is always a constant
            return re.sub(r"\s+", "_", self.take().strip()).lower()
        name = self._name()
        while self.peek() == "ID" and self.peek(1) != "LP":
            name += "_" + self.take()                   # 'captain america' -> captain_america
        if self.peek() == "LP":                         # function term f(a, b)
            self.take()
            return (name.lower(), self._args())
        for src, v in reversed(self.scope):             # variable bound by a quantifier
            if src == name:
                return v
        if name in _SINGLE_LETTER_VARS or (self.upper_vars and (name[0].isupper() or name[0] == "_")):
            if name not in self.free:                   # variable without a quantifier
                self.free[name] = self.new_var(name.lower().lstrip("_") or "v")
            return self.free[name]
        return name.lower()                             # constant: Bob == bob

    def atom(self):
        # try equality first: a = b,  f(a) ≠ b
        save_p, save_free = self.p, dict(self.free)
        try:
            a = self.term()
            if self.peek() in ("EQ", "NEQ"):
                neg = self.take() in ("≠", "!=", "\\=")
                at = ("atom", "=", (a, self.term()))
                return ("not", at) if neg else at
        except ParseError:
            pass
        self.p, self.free = save_p, save_free           # not an equality: backtrack
        if self.peek() == "QID":
            raise ParseError("A quoted name cannot be a predicate")
        name = self._name()
        args = ()
        if self.peek() == "LP":
            self.take()
            if self.peek() == "RP":
                self.take()
            else:
                args = self._args()
        return ("atom", name[0].upper() + name[1:], args)   # young == Young


_VAR_COUNTER = [0]


def _repair_parens(text):
    """One bracket too many or too few: drop unmatched ')' and close open '('."""
    out, depth = [], 0
    for ch in text:
        if ch in "([":
            depth += 1
        elif ch in ")]":
            if depth == 0:
                continue
            depth -= 1
        out.append(ch)
    return "".join(out) + ")" * depth


def parse_formula(text, upper_vars=False, free_as="forall", stats=None):
    """Formula string -> AST.
    Strict parsing first; if it fails, tolerant parsing (missing operator = ∧, bracket repair).
    free_as: how variables without a quantifier are bound ('forall', 'exists' or 'free')."""
    s = normalize_formula_text(text)
    if not s:
        return ("true",)
    attempts = [(s, False), (s, True), (_repair_parens(s), True)]
    last = None
    for i, (src, tol) in enumerate(attempts):
        try:
            p = _Parser(src, _VAR_COUNTER, upper_vars=upper_vars, tolerant=tol)
            f = p.parse()
            if stats is not None and i > 0:
                stats["tolerant"] = stats.get("tolerant", 0) + 1
            for v in reversed(list(p.free.values())):
                f = (free_as, v, f)
            return f
        except ParseError as e:
            last = e
    raise last


_UPPER_VAR_MARKERS = re.compile(r":-|\?-|!\s*\[|\?\s*\[|\b(?:fof|cnf|tff)\s*\(")


def parse_kb(kb, query=None):
    """Parse a whole knowledge base (+ optional query).
    - the dialect is decided once from the whole text (Prolog/TPTP: capitalised names are variables)
    - a formula that cannot be parsed is dropped and recorded in the stats; the rest of the KB is kept."""
    texts = split_kb(kb)
    raw = " ".join(texts) + " " + (query or "")
    upper = bool(_UPPER_VAR_MARKERS.search(raw))
    stats = {"formulas": len(texts), "dropped": [], "tolerant": 0, "upper_vars": upper}
    formulas = []
    for t in texts:
        try:
            formulas.append(parse_formula(t, upper_vars=upper, stats=stats))
        except ParseError as e:
            stats["dropped"].append((t, str(e)))
    q = None
    if query is not None:                               # an unreadable query raises; the router handles it
        q = parse_formula(query, upper_vars=upper, free_as="exists" if upper else "forall", stats=stats)
    return formulas, q, stats

In [13]:
# Test 1: the same rule in 10 dialects must give the same AST
same_rule = {
    "Unicode":  "∀x (Young(x) ∧ ¬Kind(x) → Smart(x))",
    "ASCII":    "forall x (Young(x) & ~Kind(x) -> Smart(x))",
    "Words":    "FORALL x (Young(x) AND NOT Kind(x) IMPLIES Smart(x))",
    "Prover9":  "all x (Young(x) & -Kind(x) -> Smart(x)).",
    "Prolog":   "smart(X) :- young(X), \\+ kind(X).",
    "TPTP":     "fof(r1, axiom, ![X]: ((young(X) & ~kind(X)) => smart(X))).",
    "LaTeX":    r"\forall x (Young(x) \land \neg Kind(x) \rightarrow Smart(x))",
    "Textbook": "(∀x)((Young(x) · ∼Kind(x)) ⊃ Smart(x))",
    "If-then":  "forall x (if Young(x) and not Kind(x) then Smart(x))",
    "Free var": "Young(x) ∧ ¬Kind(x) → Smart(x)",
}
results = {}
for name, text in same_rule.items():
    upper = bool(_UPPER_VAR_MARKERS.search(text))
    results[name] = fml_str(parse_formula(text, upper_vars=upper))
    print(f"{name:9s} {results[name]}")
print("All dialects give the same AST?", len(set(results.values())) == 1)

# Test 2: conventions
print()
for text in ["Young(Bob)", "young(bob)",                # case: Bob == bob, young == Young
             "Superhero(captain america)",             # multi-word name
             "a ≠ b", "∀x y (Likes(x, y) → Knows(x, y))",
             "∀x Young(x) → Smart(x)",                 # maximal scope
             "A ∧ B ∨ C", "A → B → C", "A ↔ B ⊕ C"]:   # precedence
    print(f"{text:38s} ->  {fml_str(parse_formula(text))}")

# Test 3: strict vs tolerant parsing, and one broken formula in a KB
print()
formulas, q, stats = parse_kb(["Young(bob)", "∃x (Country(x) Nearby(rome, x))",
                               "%%% ???", "∀x (Young(x) → Smart(x)"], "Smart(bob)")
for f in formulas:
    print("  parsed :", fml_str(f))
print("  query  :", fml_str(q))
print("  stats  :", stats)

Unicode   ∀x ((Young(x) ∧ ¬Kind(x)) → Smart(x))
ASCII     ∀x ((Young(x) ∧ ¬Kind(x)) → Smart(x))
Words     ∀x ((Young(x) ∧ ¬Kind(x)) → Smart(x))
Prover9   ∀x ((Young(x) ∧ ¬Kind(x)) → Smart(x))
Prolog    ∀x ((Young(x) ∧ ¬Kind(x)) → Smart(x))
TPTP      ∀x ((Young(x) ∧ ¬Kind(x)) → Smart(x))
LaTeX     ∀x ((Young(x) ∧ ¬Kind(x)) → Smart(x))
Textbook  ∀x ((Young(x) ∧ ¬Kind(x)) → Smart(x))
If-then   ∀x ((Young(x) ∧ ¬Kind(x)) → Smart(x))
Free var  ∀x ((Young(x) ∧ ¬Kind(x)) → Smart(x))
All dialects give the same AST? True

Young(Bob)                             ->  Young(bob)
young(bob)                             ->  Young(bob)
Superhero(captain america)             ->  Superhero(captain_america)
a ≠ b                                  ->  ¬=(a, b)
∀x y (Likes(x, y) → Knows(x, y))       ->  ∀x ∀y (Likes(x, y) → Knows(x, y))
∀x Young(x) → Smart(x)                 ->  ∀x (Young(x) → Smart(x))
A ∧ B ∨ C                              ->  ((A ∧ B) ∨ C)
A → B → C                              ->  (A → (B

## Section 5: Reasoning Engines

### 5a: Unification


In [14]:
def walk(t, s):
    """Look up a variable in the substitution, following chains (?x -> ?y -> bob)."""
    while isinstance(t, str) and t in s:
        t = s[t]
    return t


def occurs(v, t, s):
    """Occurs check: does variable v occur inside term t? (prevents ?x = f(?x))"""
    t = walk(t, s)
    if t == v:
        return True
    if isinstance(t, tuple):
        return any(occurs(v, a, s) for a in t[1])
    return False


def unify(a, b, s):
    """Most general unifier of two terms (extending s), or None."""
    a, b = walk(a, s), walk(b, s)
    if a == b:
        return s
    if is_var(a):
        if occurs(a, b, s):
            return None
        s = dict(s)                      # copy on write: a failed attempt never changes s
        s[a] = b
        return s
    if is_var(b):
        return unify(b, a, s)
    if isinstance(a, tuple) and isinstance(b, tuple) and a[0] == b[0] and len(a[1]) == len(b[1]):
        for x, y in zip(a[1], b[1]):     # f(a1, a2) = f(b1, b2): unify argument by argument
            s = unify(x, y, s)
            if s is None:
                return None
        return s
    return None                          # different constants / different functions


def unify_args(xs, ys, s):
    """Unify two argument lists: Chases(?x, rabbit) vs Chases(bob, ?y)."""
    if len(xs) != len(ys):
        return None
    for x, y in zip(xs, ys):
        s = unify(x, y, s)
        if s is None:
            return None
    return s


def subst_term(t, s):
    t = walk(t, s)
    if isinstance(t, tuple):
        return (t[0], tuple(subst_term(a, s) for a in t[1]))
    return t


def subst_lit(lit, s):
    """Apply a substitution to a literal: Smart(?x) + {?x: bob} -> Smart(bob)."""
    return (lit[0], lit[1], tuple(subst_term(a, s) for a in lit[2]))


def rename_term(t, ren):
    """Plain renaming (does not follow chains like a substitution)."""
    if isinstance(t, tuple):
        return (t[0], tuple(rename_term(a, ren) for a in t[1]))
    return ren.get(t, t)


def rename_lit(lit, ren):
    return (lit[0], lit[1], tuple(rename_term(a, ren) for a in lit[2]))


def term_vars(t, acc):
    if isinstance(t, tuple):
        for a in t[1]:
            term_vars(a, acc)
    elif is_var(t):
        acc.append(t)
    return acc


def lit_vars(lit):
    """All variables of a literal, in order."""
    acc = []
    for a in lit[2]:
        term_vars(a, acc)
    return acc

### 5b: Forward Chaining with Negation as Failure (closed world)


In [15]:
from itertools import product


def _as_literal(f):
    """Atom or ¬Atom -> literal (True/False, pred, args); otherwise None."""
    if f[0] == 'atom':
        return (True, f[1], f[2])
    if f[0] == 'not' and f[1][0] == 'atom':
        return (False, f[1][1], f[1][2])
    if f[0] == 'not' and f[1][0] == 'not':
        return _as_literal(f[1][1])
    return None


def _literal_list(f):
    """A literal or a conjunction of literals -> list of literals; otherwise None."""
    if f[0] == 'and':
        out = []
        for g in f[1:]:
            sub = _literal_list(g)
            if sub is None:
                return None
            out += sub
        return out
    if f[0] == 'true':
        return []
    lit = _as_literal(f)
    return None if lit is None else [lit]


def _flat(lits):
    """In a rule base, arguments are constants or variables only (no function terms)."""
    return all(not isinstance(a, tuple) for l in lits for a in l[2])


def formula_to_rules(f):
    """Formula -> [(body_literals, head_literal), ...] rules, or None if the formula is not a rule-base
    formula (∨ in the head, ⊕, ↔, ∃ ...). A fact is a rule with an empty body."""
    while f[0] == 'forall':
        f = f[2]
    if f[0] == 'imp':
        heads = _literal_list(f[2])
        if heads is None:
            return None
        bodies = []
        alts = f[1][1:] if f[1][0] == 'or' else (f[1],)   # A ∨ B → C  ==  A→C, B→C
        for alt in alts:
            b = _literal_list(alt)
            if b is None:
                return None
            bodies.append(b)
        rules = [(b, h) for b in bodies for h in heads]
    else:
        lits = _literal_list(f)
        if lits is None:
            return None
        rules = [([], h) for h in lits]
    if not all(_flat(b + [h]) for b, h in rules):
        return None
    return rules


class RuleBase:
    """Facts + rules. On construction, stratified forward chaining computes the closure (all derivable facts)."""

    def __init__(self, rules, extra_constants=()):
        self.rules = rules                       # [(body, head), ...]
        self.domain = set(extra_constants)       # all constants (Herbrand domain)
        for b, h in rules:
            for l in b + [h]:
                self.domain.update(a for a in l[2] if not is_var(a))
        self.domain = sorted(self.domain)
        self._extended = {}                      # cache for queries that name new constants
        self.pos = {}                            # pred -> set(args): derived facts (closure)
        self.why = {}                            # (pred, args) -> (rule index, body) that derived it
        self.neg_derived = {}                    # negative conclusions of rules (for traces)
        self.stratified = True
        self.steps = 0                           # number of rule firings (efficiency metric)
        self._closure()

    # ---- matching -------------------------------------------------------
    def _holds(self, pred, args):
        """Is this fact in the closure? (set lookup, O(1))"""
        return args in self.pos.get(pred, ())

    def _match_body(self, body):
        """Match the positive body literals against known facts (unification with constants only).
        Yield the substitution of every match."""
        positives = [l for l in body if l[0]]

        def rec(i, s):
            if i == len(positives):
                yield s
                return
            _, pred, args = positives[i]
            pattern = tuple(s.get(a, a) for a in args)
            for fact in tuple(self.pos.get(pred, ())):
                s2 = s
                ok = True
                for p, v in zip(pattern, fact):
                    if is_var(p):
                        bound = s2.get(p)
                        if bound is None:
                            if s2 is s:
                                s2 = dict(s)
                            s2[p] = v
                        elif bound != v:
                            ok = False
                            break
                    elif p != v:
                        ok = False
                        break
                if ok and len(pattern) == len(fact):
                    yield from rec(i + 1, s2)

        yield from rec(0, {})

    def _fire(self, rule_idx, body, head):
        """Fire one rule in every possible way; return the newly derived facts."""
        new = []
        negatives = [l for l in body if not l[0]]
        for s in self._match_body(body):
            # a variable not bound by a positive literal ranges over the whole domain
            free = sorted({v for l in negatives + [head] for v in lit_vars(l) if v not in s})
            for vals in (product(self.domain, repeat=len(free)) if free else [()]):
                s2 = dict(s, **dict(zip(free, vals))) if free else s
                if any(self._holds(l[1], tuple(s2.get(a, a) for a in l[2])) for l in negatives):
                    continue                                    # NAF: P is already derived, so ¬P fails and the rule does not fire
                self.steps += 1
                hargs = tuple(s2.get(a, a) for a in head[2])
                key = (head[1], hargs)
                ground_body = [(l[0], l[1], tuple(s2.get(a, a) for a in l[2])) for l in body]
                if head[0]:
                    if not self._holds(*key):
                        new.append(key)
                        self.why.setdefault(key, (rule_idx, ground_body))
                else:
                    self.neg_derived.setdefault(key, (rule_idx, ground_body))
        return new

    # ---- stratification ---------------------------------------------------
    def _strata(self):
        """Assign every predicate a stratum: a predicate used under ¬ in a body must be complete in a lower
        stratum than the head. Returns None if there is a negative cycle."""
        preds = {h[1] for _, h in self.rules if h[0]} | {l[1] for b, _ in self.rules for l in b}
        level = {p: 0 for p in preds}
        for _ in range(len(preds) + 1):
            changed = False
            for body, head in self.rules:
                if not head[0]:
                    continue
                for l in body:
                    need = level[l[1]] + (0 if l[0] else 1)
                    if level[head[1]] < need:
                        level[head[1]] = need
                        changed = True
            if not changed:
                return level
        return None                                  # negative cycle: not stratifiable

    def _closure(self):
        level = self._strata()
        if level is None:
            self.stratified = False
            level = {h[1]: 0 for _, h in self.rules}
        groups = {}
        for i, (body, head) in enumerate(self.rules):
            if head[0]:
                groups.setdefault(level[head[1]], []).append(i)
        for lv in sorted(groups):
            idxs = groups[lv]
            for _ in range(10_000):                  # run to a fixpoint inside the stratum
                added = []
                for i in idxs:
                    body, head = self.rules[i]
                    for key in self._fire(i, body, head):
                        if not self._holds(*key):
                            self.pos.setdefault(key[0], set()).add(key[1])
                            added.append(key)
                if not added:
                    break
        for i, (body, head) in enumerate(self.rules):  # negative-head rules: recorded for traces only
            if not head[0]:
                self._fire(i, body, head)

    # ---- queries ------------------------------------------------------------
    def _for_constants(self, args):
        """If the query names a constant that is not in the KB (e.g. 'charlie'), extend the domain and
        recompute the closure; otherwise '∀x (¬Kind(x) → Nice(x))' would never apply to charlie."""
        new = frozenset(a for a in args if a not in self.domain)
        if not new:
            return self
        if new not in self._extended:
            self._extended[new] = RuleBase(self.rules, self.domain + sorted(new))
        return self._extended[new]

    def holds(self, lit):
        """Closed world: P holds if it is derived; ¬P holds if P is not derived."""
        pos, pred, args = lit
        rb = self._for_constants(args)
        return rb._holds(pred, args) == pos

    def explain(self, key, depth=0, seen=None):
        """Proof tree (reasoning trace) of a derived fact, as text lines."""
        seen = set() if seen is None else seen
        pad = '    ' * depth
        atom = lit_str((True,) + key)
        if key in seen:
            return [f"{pad}{atom}   (see above)"]
        seen.add(key)
        rule_idx, ground_body = self.why[key]
        body, head = self.rules[rule_idx]
        if not body:
            return [f"{pad}{atom}   [given fact]"]
        lines = [f"{pad}{atom}   [rule {rule_idx}: {' ∧ '.join(map(lit_str, body))} → {lit_str(head)}]"]
        for l in ground_body:
            if l[0]:
                lines += self.explain((l[1], l[2]), depth + 1, seen)
            else:
                lines.append(f"{pad}    {lit_str(l)}   [not derivable → true by negation-as-failure]")
        return lines

In [16]:
def build_rulebase(kb):
    """KB text -> parse -> rules -> RuleBase (the closure is computed immediately)."""
    formulas, _, stats = parse_kb(kb)
    rules = []
    for f in formulas:
        r = formula_to_rules(f)
        if r is None:
            raise ValueError(f"Not a rule-base formula: {fml_str(f)}")
        rules += r
    return RuleBase(rules)


# 1) Unification
print("1) Unification")
s = unify_args(("?x", "rabbit"), ("bob", "?y"), {})
print("   Chases(?x, rabbit) = Chases(bob, ?y)  ->", s)
print("   Chases(?x, rabbit) = Chases(bob, cat) ->", unify_args(("?x", "rabbit"), ("bob", "cat"), {}))
print("   ?x = f(?x)  (occurs check)            ->", unify("?x", ("f", ("?x",)), {}))

# 2) The example from the assignment, with its reasoning trace
print("\n2) Assignment example")
rb = build_rulebase(["Young(bob)", "Kind(bob)",
                     "∀x (Young(x) → Smart(x))", "∀x (Smart(x) ∧ Kind(x) → Successful(x))"])
print("   Successful(bob)?", rb.holds((True, "Successful", ("bob",))))
for line in rb.explain(("Successful", ("bob",))):
    print("     ", line)

# 3) A RuleTaker context with its gold labels
print("\n3) RuleTaker context 0 (closed world)")
rb = build_rulebase(["Quiet(anne)", "¬Young(anne)", "Kind(bob)", "Young(bob)", "Rough(dave)",
                     "Round(dave)", "Smart(dave)", "¬Young(dave)", "Quiet(fiona)", "¬Round(fiona)",
                     "∀x (Kind(x) ∧ Young(x) → ¬Smart(x))"])
tests = [("Kind(bob)", True), ("¬Smart(dave)", False), ("¬Smart(bob)", True), ("Smart(bob)", False),
         ("¬Quiet(dave)", True), ("Rough(anne)", False), ("¬Smart(fiona)", True), ("Kind(dave)", False)]
correct = 0
for q, gold in tests:
    lit = _literal_list(parse_formula(q))[0]
    ans = rb.holds(lit)
    correct += ans == gold
    print(f"   {q:14s} -> {ans!s:5s} (gold {gold})")
print(f"   {correct}/{len(tests)} correct | rule firings: {rb.steps}")

# 4) Negation as failure with a new constant
print("\n4) Negation as failure")
rb = build_rulebase(["Quiet(fiona)", "∀x (Quiet(x) ∧ ¬Kind(x) → Nice(x))", "∀x (¬Kind(x) → Lonely(x))"])
print("   Nice(fiona)?    ", rb.holds((True, "Nice", ("fiona",))))
for line in rb.explain(("Nice", ("fiona",))):
    print("     ", line)
print("   Lonely(charlie)?", rb.holds((True, "Lonely", ("charlie",))), " <- charlie does not occur in the KB")

# 5) Stratification
print("\n5) Stratification")
ok = build_rulebase(["Quiet(a)", "∀x (Quiet(x) ∧ ¬Kind(x) → Nice(x))"])
bad = build_rulebase(["Quiet(a)", "∀x (Quiet(x) ∧ ¬Kind(x) → Kind(x))"])
print("   Nice depends on ¬Kind      -> stratified:", ok.stratified)
print("   Kind depends on ¬Kind      -> stratified:", bad.stratified)

1) Unification
   Chases(?x, rabbit) = Chases(bob, ?y)  -> {'?x': 'bob', '?y': 'rabbit'}
   Chases(?x, rabbit) = Chases(bob, cat) -> None
   ?x = f(?x)  (occurs check)            -> None

2) Assignment example
   Successful(bob)? True
      Successful(bob)   [rule 3: Smart(x) ∧ Kind(x) → Successful(x)]
          Smart(bob)   [rule 2: Young(x) → Smart(x)]
              Young(bob)   [given fact]
          Kind(bob)   [given fact]

3) RuleTaker context 0 (closed world)
   Kind(bob)      -> True  (gold True)
   ¬Smart(dave)   -> False (gold False)
   ¬Smart(bob)    -> True  (gold True)
   Smart(bob)     -> False (gold False)
   ¬Quiet(dave)   -> True  (gold True)
   Rough(anne)    -> False (gold False)
   ¬Smart(fiona)  -> True  (gold True)
   Kind(dave)     -> False (gold False)
   8/8 correct | rule firings: 18

4) Negation as failure
   Nice(fiona)?     True
      Nice(fiona)   [rule 1: Quiet(x) ∧ ¬Kind(x) → Nice(x)]
          Quiet(fiona)   [given fact]
          ¬Kind(fiona)   [not de

### 5c: Backward Chaining
Goal-directed proof: `G` holds if it is a fact, or if a rule whose head unifies with `G` has a provable body.
Memoisation and a set of open goals prevent loops. This is the default closed-world engine (Experiment 1:
faster, and correct on non-stratified rule bases).

### 5d: Resolution Refutation (open world, first-order logic)
KB ⊨ q holds exactly when KB ∧ ¬q derives the empty clause □.
1. Clause form: remove →, ↔, ⊕; negation normal form; Skolemisation; CNF. Parts whose CNF would be large are
   replaced by a new atom (Tseitin-style definitional CNF), which preserves satisfiability.
2. Set of support: only descendants of ¬q are resolved.
3. Best-first selection: the lightest clause first, every fifth pick the oldest (fairness).
4. Pruning: tautologies, duplicates, unit subsumption; factoring; equality axioms when `=` occurs.
5. Deterministic budget of 50 given clauses (chosen on FOLIO train), with a 2-second safety net.

Open world: q proved → True, ¬q proved → False, otherwise Uncertain. The output is binary, so False and
Uncertain are both `False`.

In [17]:
class BackwardChainer:
    """Goal-directed proof: G holds if it is a fact, or if a rule whose head unifies with G has a
    provable body. Same negation-as-failure semantics as forward chaining."""

    def __init__(self, rules, max_depth=60):
        self.rules = rules
        self.max_depth = max_depth
        self.domain = sorted({a for b, h in rules for l in b + [h] for a in l[2] if not is_var(a)})
        self.facts = {(h[1], h[2]) for b, h in rules if not b and h[0] and not lit_vars(h)}
        self.by_head = {}
        for b, h in rules:
            if h[0]:
                self.by_head.setdefault(h[1], []).append((b, h))
        self.memo = {}                        # results of goals proved earlier
        self.steps = 0                        # number of sub-goals tried (efficiency metric)

    def prove(self, key, depth=0, active=frozenset()):
        if key in self.facts:
            return True
        if key in self.memo:
            return self.memo[key]
        if key in active or depth > self.max_depth:
            return False                      # loop: this path cannot prove the goal
        active = active | {key}
        pred, args = key
        result = False
        for body, head in self.by_head.get(pred, ()):
            s = {}
            ok = True
            for p, v in zip(head[2], args):
                if is_var(p):
                    if s.get(p, v) != v:
                        ok = False
                        break
                    s[p] = v
                elif p != v:
                    ok = False
                    break
            if ok and self._prove_body(body, s, depth, active):
                result = True
                break
        if not active - {key}:                # memoise only results that do not depend on open goals
            self.memo[key] = result
        return result

    def _prove_body(self, body, s, depth, active):
        if not body:
            return True
        lit, rest = body[0], body[1:]
        free = [v for v in dict.fromkeys(lit_vars(lit)) if v not in s]
        for vals in (product(self.domain, repeat=len(free)) if free else [()]):
            s2 = dict(s, **dict(zip(free, vals)))
            self.steps += 1
            g = (lit[1], tuple(s2.get(a, a) for a in lit[2]))
            ok = self.prove(g, depth + 1, active)
            if ok == lit[0] and self._prove_body(rest, s2, depth, active):
                return True
        return False

    def holds(self, lit):
        new = [a for a in lit[2] if a not in self.domain]
        if new:                               # add the query's new constants to the domain
            self.domain = sorted(set(self.domain) | set(new))
            self.memo = {}
        return self.prove((lit[1], lit[2])) == lit[0]

In [18]:
import heapq, itertools, time
from collections import deque

_SK = itertools.count()


def _elim(f):
    """Step 1: rewrite →, ↔, ⊕ with ¬, ∧, ∨ only."""
    op = f[0]
    if op == 'atom' or op in ('true', 'false'):
        return f
    if op == 'not':
        return ('not', _elim(f[1]))
    if op in ('and', 'or'):
        return (op, *[_elim(g) for g in f[1:]])
    if op == 'imp':
        return ('or', ('not', _elim(f[1])), _elim(f[2]))
    if op == 'iff':
        a, b = _elim(f[1]), _elim(f[2])
        return ('and', ('or', ('not', a), b), ('or', a, ('not', b)))
    if op == 'xor':
        a, b = _elim(f[1]), _elim(f[2])
        return ('and', ('or', a, b), ('or', ('not', a), ('not', b)))
    if op in ('forall', 'exists'):
        return (op, f[1], _elim(f[2]))
    raise ValueError(op)


def _nnf(f, neg=False):
    """Step 2: push ¬ inwards (De Morgan, ¬∀ = ∃¬): negation normal form."""
    op = f[0]
    if op == 'atom':
        return ('not', f) if neg else f
    if op == 'true':
        return ('false',) if neg else f
    if op == 'false':
        return ('true',) if neg else f
    if op == 'not':
        return _nnf(f[1], not neg)
    if op in ('and', 'or'):
        nop = {'and': 'or', 'or': 'and'}[op] if neg else op
        return (nop, *[_nnf(g, neg) for g in f[1:]])
    if op in ('forall', 'exists'):
        nop = {'forall': 'exists', 'exists': 'forall'}[op] if neg else op
        return (nop, f[1], _nnf(f[2], neg))
    raise ValueError(op)


def _skolem(f, univ, ren):
    """Step 3: remove ∃ (Skolem constant/function), give every ∀ a fresh variable, drop ∀."""
    op = f[0]
    if op == 'atom':
        return ('atom', f[1], tuple(_ren_term(a, ren) for a in f[2]))
    if op == 'not':
        return ('not', _skolem(f[1], univ, ren))
    if op in ('true', 'false'):
        return f
    if op in ('and', 'or'):
        return (op, *[_skolem(g, univ, ren) for g in f[1:]])
    if op == 'forall':
        v = f"?u{next(_SK)}"
        return _skolem(f[2], univ + (v,), dict(ren, **{f[1]: v}))
    if op == 'exists':
        n = next(_SK)
        sk = (f"sk{n}", tuple(univ)) if univ else f"skc{n}"
        return _skolem(f[2], univ, dict(ren, **{f[1]: sk}))
    raise ValueError(op)


def _ren_term(t, ren):
    if isinstance(t, tuple):
        return (t[0], tuple(_ren_term(a, ren) for a in t[1]))
    return ren.get(t, t)


_DEF = itertools.count()
CNF_LIMIT = 64          # a disjunction producing more clauses than this gets a new name (definition)


def _define(clauses, defs):
    """Tseitin-style naming: replace a large part S (list of clauses) by a new atom D(vars) and add
    'D → S' to defs. In NNF every part occurs positively, so one direction suffices.
    Satisfiability is preserved, so refutations stay sound."""
    vs = []
    for c in clauses:
        for l in c:
            for v in lit_vars(l):
                if v not in vs:
                    vs.append(v)
    d = (True, f"Def{next(_DEF)}", tuple(vs))
    for c in clauses:
        defs.append([(False, d[1], d[2])] + list(c))
    return d


def _cnf(f, defs, limit=CNF_LIMIT):
    """Step 4: distribute ∨ over ∧ (CNF) -> list of clauses.
    If distribution would produce more than `limit` clauses, that part is named instead (_define),
    so the CNF never grows exponentially."""
    op = f[0]
    if op == 'true':
        return []
    if op == 'false':
        return [[]]
    if op == 'atom':
        return [[(True, f[1], f[2])]]
    if op == 'not':
        return [[(False, f[1][1], f[1][2])]]
    if op == 'and':
        return [c for g in f[1:] for c in _cnf(g, defs, limit)]
    if op == 'or':
        subs = sorted((_cnf(g, defs, limit) for g in f[1:]), key=len)
        result = [[]]
        for sub in subs:
            if len(sub) > 1 and len(result) * len(sub) > limit:
                sub = [[_define(sub, defs)]]          # large part -> one new literal
            result = [a + b for a in result for b in sub]
            if len(result) > limit:
                result = [[_define(result, defs)]]
        return result
    raise ValueError(op)


def normalize_clause(lits):
    """Remove duplicate literals, return None for a tautology (P ∨ ¬P), rename variables to ?0, ?1, ..."""
    lits = set(lits)
    for l in lits:
        if (not l[0], l[1], l[2]) in lits:
            return None
    order = sorted(lits, key=lambda l: (l[1], l[0], len(l[2])))
    ren = {}
    for l in order:
        for v in lit_vars(l):
            if v not in ren:
                ren[v] = f"?{len(ren)}"
    if ren:
        lits = {rename_lit(l, ren) for l in lits}
    return frozenset(lits)


def to_clauses(f):
    """Formula -> clause form (steps 1-4), including the clauses of the definitions."""
    g = _skolem(_nnf(_elim(f)), (), {})
    defs = []
    out = []
    for c in _cnf(g, defs) + defs:
        n = normalize_clause(c)
        if n is not None:
            out.append(n)
    return out


def _rename_apart(clause, tag):
    """Rename variables apart so that two clauses never share a variable by accident."""
    ren = {}
    out = []
    for l in clause:
        for v in lit_vars(l):
            ren.setdefault(v, v + tag)
        out.append(rename_lit(l, ren))
    return out


def _weight(clause):
    """Clause weight (number of symbols). Lighter clauses first: the best-first heuristic."""
    def tw(t):
        return 1 + sum(tw(a) for a in t[1]) if isinstance(t, tuple) else 1
    return sum(1 + sum(tw(a) for a in l[2]) for l in clause)


def _match_lit(general, specific):
    """One-way matching (for subsumption): general·s == specific."""
    if general[0] != specific[0] or general[1] != specific[1] or len(general[2]) != len(specific[2]):
        return None
    s = {}
    stack = list(zip(general[2], specific[2]))
    while stack:
        g, t = stack.pop()
        if is_var(g):
            if g in s and s[g] != t:
                return None
            s[g] = t
        elif isinstance(g, tuple):
            if not (isinstance(t, tuple) and g[0] == t[0] and len(g[1]) == len(t[1])):
                return None
            stack.extend(zip(g[1], t[1]))
        elif g != t:
            return None
    return s

In [19]:
def _lit_key(l):
    """Sort key for literals. Set order differs between processes (hash seed); sorting keeps the
    proof search deterministic."""
    return (l[1], l[0], repr(l[2]))


class ResolutionProver:
    """KB ⊨ q  ⇔  the empty clause □ (a contradiction) can be derived from KB ∧ ¬q."""

    def __init__(self, axioms, time_limit=0.3, max_clauses=6000, age_ratio=5, max_given=None):
        self.axioms = axioms
        self.max_given = max_given      # deterministic budget: number of given clauses (machine-independent)
        self.time_limit = time_limit
        self.max_clauses = max_clauses
        self.age_ratio = age_ratio

    def refute(self, sos_clauses):
        """Start from the clauses of ¬query (set of support). Returns (proved, status, proof_lines)."""
        t0 = time.perf_counter()
        parents = {}
        index = {}                      # (sign, pred) -> clauses: fast lookup of resolution partners
        units = {}                      # unit clauses, for subsumption
        seen = set()
        heap = []
        tick = itertools.count()

        def add_usable(c):
            for l in c:
                index.setdefault((l[0], l[1]), []).append(c)
            if len(c) == 1:
                l = next(iter(c))
                units.setdefault((l[0], l[1]), []).append(l)

        def subsumed(c):
            for l in c:
                for u in units.get((l[0], l[1]), ()):
                    if _match_lit(u, l) is not None:
                        return True
            return False

        for c in self.axioms:
            if c not in seen:
                seen.add(c)
                parents[c] = ('axiom',)
                add_usable(c)
        for c in sos_clauses:
            if not c:
                return True, 'proved', ['¬query is itself contradictory (the query is a tautology)']
            if c not in seen:
                seen.add(c)
                parents[c] = ('goal',)
                heapq.heappush(heap, (_weight(c), next(tick), c))

        age_queue = deque(c for _, _, c in sorted(heap, key=lambda x: x[1]))  # oldest first
        picked = set()
        n_given = 0
        while heap or age_queue:
            if time.perf_counter() - t0 > self.time_limit:
                return False, 'timeout', []
            if len(seen) > self.max_clauses:
                return False, 'clause-limit', []
            if self.max_given is not None and n_given >= self.max_given:
                return False, 'step-limit', []
            # next given clause: usually the lightest, every 5th pick the oldest (fairness)
            given = None
            if n_given % self.age_ratio == self.age_ratio - 1:
                while age_queue and given is None:
                    c = age_queue.popleft()
                    if c not in picked:
                        given = c
            while given is None and heap:
                _, _, c = heapq.heappop(heap)
                if c not in picked:
                    given = c
            if given is None:
                break
            picked.add(given)
            n_given += 1
            if subsumed(given) and len(given) > 1:
                continue
            g_lits = sorted(given, key=_lit_key)        # fixed order: identical search on every run and machine
            add_usable(given)
            for i, lit in enumerate(g_lits):
                for other in tuple(index.get((not lit[0], lit[1]), ())):
                    o_lits = _rename_apart(sorted(other, key=_lit_key), "'")
                    for j, ol in enumerate(o_lits):
                        if ol[0] == lit[0] or ol[1] != lit[1]:
                            continue
                        s = unify_args(lit[2], ol[2], {})
                        if s is None:
                            continue
                        res = [subst_lit(l, s) for k, l in enumerate(g_lits) if k != i] + \
                              [subst_lit(l, s) for k, l in enumerate(o_lits) if k != j]
                        new = normalize_clause(res)
                        if new is None or new in seen:
                            continue
                        seen.add(new)
                        parents[new] = ('res', given, other)
                        if not new:
                            return True, 'proved', self._proof(new, parents)
                        if subsumed(new):
                            continue
                        heapq.heappush(heap, (_weight(new), next(tick), new))
                        age_queue.append(new)
            # factoring: merge two unifiable literals of the same sign in one clause
            for a, b in itertools.combinations(range(len(g_lits)), 2):
                la, lb = g_lits[a], g_lits[b]
                if la[0] == lb[0] and la[1] == lb[1]:
                    s = unify_args(la[2], lb[2], {})
                    if s is not None:
                        new = normalize_clause([subst_lit(l, s) for k, l in enumerate(g_lits) if k != b])
                        if new is not None and new not in seen:
                            seen.add(new)
                            parents[new] = ('factor', given)
                            heapq.heappush(heap, (_weight(new), next(tick), new))
                            age_queue.append(new)
        return False, 'saturated', []

    @staticmethod
    def _proof(empty, parents):
        """Walk back from □ and list only the clauses used in the proof, in order."""
        def cs(c):
            return '{' + ', '.join(sorted(map(lit_str, c))) + '}' if c else '□ (empty clause)'
        lines, order, stack, done = [], [], [empty], set()
        while stack:
            c = stack.pop()
            if c in done:
                continue
            done.add(c)
            order.append(c)
            p = parents[c]
            stack.extend(x for x in p[1:])
        for c in reversed(order):
            p = parents[c]
            if p[0] == 'axiom':
                lines.append(f"KB      : {cs(c)}")
            elif p[0] == 'goal':
                lines.append(f"¬query  : {cs(c)}")
            elif p[0] == 'res':
                lines.append(f"resolve : {cs(p[1])} + {cs(p[2])}  ⟹  {cs(c)}")
            else:
                lines.append(f"factor  : {cs(p[1])}  ⟹  {cs(c)}")
        return lines


def equality_axioms(preds_with_arity):
    """Equality axioms for a KB that uses '=': reflexivity, symmetry, transitivity, substitution."""
    ax = ["∀x (x = x)", "∀x ∀y (x = y → y = x)", "∀x ∀y ∀z (x = y ∧ y = z → x = z)"]
    for p, n in sorted(preds_with_arity):
        if p == '=' or n == 0:
            continue
        for k in range(n):
            xs = [f"a{i}" for i in range(n)]
            ys = list(xs)
            ys[k] = "b"
            q = ' '.join(f"∀{v}" for v in xs + ['b'])
            ax.append(f"{q} ({xs[k]} = b ∧ {p}({', '.join(xs)}) → {p}({', '.join(ys)}))")
    return ax

In [20]:
def kb_clauses(kb):
    formulas, _, _ = parse_kb(kb)
    return list(dict.fromkeys(c for f in formulas for c in to_clauses(f)))


def entails(kb, query, time_limit=0.3):
    """Open world: KB ⊨ query ?  -> (True/False, status, proof)"""
    prover = ResolutionProver(kb_clauses(kb), time_limit=time_limit)
    return prover.refute(to_clauses(("not", parse_formula(query))))


def show_clauses(text):
    print(f"   {text}")
    for c in to_clauses(parse_formula(text)):
        print("      {" + ", ".join(sorted(map(lit_str, c))) + "}")


# 1) Clause form (CNF): ⊕, ↔, ∃ and ∀∃ (Skolem function)
print("1) Clause form")
show_clauses("∀x (Drinks(x) ⊕ Jokes(x))")
show_clauses("Student(rina) ↔ Unaware(rina)")
show_clauses("∃x (Man(x) ∧ Rich(x))")
show_clauses("∀x ∃y Loves(x, y)")

# 2) Resolution proof (reasoning by cases + ⊕)
print("\n2) Resolution proof")
kb = ["∀x (Drinks(x) → Dependent(x))", "∀x (Drinks(x) ⊕ Jokes(x))",
      "∀x (Jokes(x) → ¬Unaware(x))", "Unaware(rina)"]
proved, status, proof = entails(kb, "Dependent(rina)")
print(f"   Dependent(rina)? {proved} [{status}]")
for line in proof:
    print("     ", line)

# 3) Open world: True / False / Uncertain
print("\n3) The three open-world answers")
kb = ["∀x (Bird(x) → Fly(x))", "Bird(tweety)", "¬Fly(rex)"]
for q in ["Fly(tweety)", "¬Fly(tweety)", "¬Bird(rex)", "Fly(polly)"]:
    yes = entails(kb, q)[0]
    no = entails(kb, f"¬({q})")[0]
    label = "True" if yes else ("False" if no else "Uncertain")
    print(f"   {q:12s} -> {label:9s} -> binary answer: {yes}")

# 4) Backward chaining vs forward chaining (same answers?)
print("\n4) Backward vs forward chaining")
kb = ["Quiet(anne)", "¬Young(anne)", "Kind(bob)", "Young(bob)", "Smart(dave)", "Quiet(fiona)",
      "∀x (Kind(x) ∧ Young(x) → ¬Smart(x))", "∀x (Quiet(x) ∧ ¬Smart(x) → Nice(x))"]
rules = [r for f in parse_kb(kb)[0] for r in formula_to_rules(f)]
fc, bc = RuleBase(rules), BackwardChainer(rules)
for q in ["Nice(anne)", "Nice(fiona)", "Nice(bob)", "¬Smart(bob)", "Nice(dave)"]:
    lit = _literal_list(parse_formula(q))[0]
    print(f"   {q:12s} FC={fc.holds(lit)!s:5s} BC={bc.holds(lit)!s:5s}")
print(f"   FC rule firings: {fc.steps} | BC sub-goals: {bc.steps}")

1) Clause form
   ∀x (Drinks(x) ⊕ Jokes(x))
      {Drinks(x0), Jokes(x0)}
      {¬Drinks(x0), ¬Jokes(x0)}
   Student(rina) ↔ Unaware(rina)
      {Unaware(rina), ¬Student(rina)}
      {Student(rina), ¬Unaware(rina)}
   ∃x (Man(x) ∧ Rich(x))
      {Man(skc1)}
      {Rich(skc1)}
   ∀x ∃y Loves(x, y)
      {Loves(x0, sk3(x0))}

2) Resolution proof
   Dependent(rina)? True [proved]
      ¬query  : {¬Dependent(rina)}
      KB      : {Dependent(x0), ¬Drinks(x0)}
      resolve : {¬Dependent(rina)} + {Dependent(x0), ¬Drinks(x0)}  ⟹  {¬Drinks(rina)}
      KB      : {Drinks(x0), Jokes(x0)}
      resolve : {¬Drinks(rina)} + {Drinks(x0), Jokes(x0)}  ⟹  {Jokes(rina)}
      KB      : {¬Jokes(x0), ¬Unaware(x0)}
      resolve : {Jokes(rina)} + {¬Jokes(x0), ¬Unaware(x0)}  ⟹  {¬Unaware(rina)}
      KB      : {Unaware(rina)}
      resolve : {¬Unaware(rina)} + {Unaware(rina)}  ⟹  □ (empty clause)

3) The three open-world answers
   Fly(tweety)  -> True      -> binary answer: True
   ¬Fly(tweety) -> False  

### 5e: Structural Causal Model (counterfactual queries)


In [21]:
import itertools, re


class CausalCycleError(ValueError):
    """The structural equations contain a cycle: the SCM has no unique solution, so do() is undefined."""

    def __init__(self, cycle):
        self.cycle = cycle                                  # ['Vork', 'Sline', 'Vork'] (effect <- cause order)
        super().__init__("causal cycle: " + " <- ".join(cycle))


class SCM:
    """Deterministic Structural Causal Model.
    equations : {var: formula}   every endogenous variable is a Boolean function of its parents
                (same AST as the parser: ('atom', name, ()), ('not', ..), ('and', ..), ('or', ..), ...)
    priors    : {root_var: P(root=1)}  (Bern(p); 0.5 if not given)
    Roots = variables without an equation (exogenous)."""

    def __init__(self, equations, priors=None):
        self.eq = dict(equations)
        names = set(self.eq)
        for f in self.eq.values():
            names |= _prop_atoms(f)
        self.vars = sorted(names)
        self.roots = [v for v in self.vars if v not in self.eq]
        self.priors = priors or {}
        self.order = self._topological_order()

    def _topological_order(self):
        order, done, path = [], set(), []

        def visit(v):
            if v in done:
                return
            if v in path:                                   # cycle: report the whole path
                cyc = path[path.index(v):] + [v]
                raise CausalCycleError(cyc)
            path.append(v)
            for p in sorted(_prop_atoms(self.eq.get(v, ("true",)))):
                visit(p)
            path.pop()
            done.add(v)
            order.append(v)

        for v in self.vars:
            visit(v)
        return order

    def evaluate(self, roots, do=None):
        """Root values + interventions (do) -> value of every variable.
        do(V=val): remove the equation of V (graph surgery) and fix V to val.
        An intervened variable outside the graph simply receives its value (no effect)."""
        do = do or {}
        val = {k: v for k, v in do.items() if k not in self.eq and k not in self.roots}
        for v in self.order:
            if v in do:
                val[v] = do[v]
            elif v in self.eq:
                val[v] = _eval(self.eq[v], val)
            else:
                val[v] = roots[v]
        return val

    def worlds(self, evidence):
        """Abduction: every root assignment consistent with the evidence, with its prior weight."""
        out = []
        for bits in itertools.product([False, True], repeat=len(self.roots)):
            roots = dict(zip(self.roots, bits))
            val = self.evaluate(roots)
            if all(val[k] == v for k, v in evidence.items()):
                w = 1.0
                for r, b in roots.items():
                    p = self.priors.get(r, 0.5)
                    w *= p if b else 1 - p
                out.append((roots, w))
        return out

    def counterfactual(self, target, evidence, do_steps, cf_evidence=None):
        """Pearl's three steps: 1) abduction (roots from evidence)  2) action (do)  3) prediction.
        evidence    : observations in the factual world
        do_steps    : list of interventions (more than one step for nested queries)
        cf_evidence : observations in the post-intervention world (conditional queries)
        Returns P(target = 1), weighted by the priors over consistent worlds, or None if no world remains."""
        if target not in self.vars:
            return None
        yes = total = 0.0
        for roots, w in self.worlds(evidence):
            do = {}
            for step in do_steps:
                do.update(step(self, roots, do) if callable(step) else step)
            val = self.evaluate(roots, do)
            if cf_evidence and any(val.get(k) != v for k, v in cf_evidence.items()):
                continue
            total += w
            yes += w if val[target] else 0.0
        return yes / total if total else None


def _prop_atoms(f, acc=None):
    acc = set() if acc is None else acc
    if f[0] == "atom":
        acc.add(f[1])
    elif f[0] not in ("true", "false"):
        for g in f[1:]:
            if isinstance(g, tuple):
                _prop_atoms(g, acc)
    return acc


def _eval(f, val):
    op = f[0]
    if op == "atom":
        return val[f[1]]
    if op == "not":
        return not _eval(f[1], val)
    if op == "and":
        return all(_eval(g, val) for g in f[1:])
    if op == "or":
        return any(_eval(g, val) for g in f[1:])
    if op == "imp":
        return (not _eval(f[1], val)) or _eval(f[2], val)
    if op == "iff":
        return _eval(f[1], val) == _eval(f[2], val)
    if op == "xor":
        return _eval(f[1], val) != _eval(f[2], val)
    if op == "true":
        return True
    if op == "false":
        return False
    raise ValueError(op)


def causal_answer(equations, priors, evidence, target, do, factual=None):
    """True/False answer to one counterfactual query (one policy, chosen on the CounterBench tune split).
    target  : (var, positive?)   e.g. ('Lumbo', True) = 'would Lumbo occur?'
    do      : {var: value}       interventions (more than one for joint / nested queries)
    factual : {var: value}       the actual world of 'instead of X' (X = 1)
    Policy (definitions of the CounterBench paper):
      - only roots (exogenous) are intervened and there are observations -> the observations hold in the
        post-intervention world (conditional: Y_x | Z_x = z)
      - otherwise observations + factual values hold in the actual world -> abduction (basic / backdoor)"""
    scm = SCM(equations, priors)
    known = set(scm.vars)
    evidence = {k: v for k, v in evidence.items() if k in known}       # drop names outside the graph
    factual = {k: v for k, v in (factual or {}).items() if k in known}
    # An observation of an intervened variable describes the factual world: do() overrides it, so it
    # cannot hold in the post-intervention world ('X = 0 was observed; what if X = 1?').
    factual.update({k: v for k, v in evidence.items() if k in do})
    evidence = {k: v for k, v in evidence.items() if k not in do}
    var, positive = target
    roots_intervened = all(v in scm.roots for v in do)
    if evidence and roots_intervened:
        p = scm.counterfactual(var, dict(factual or {}), [do], evidence)
        if p is None:                                   # no world is consistent with the observations
            p = scm.counterfactual(var, dict(factual or {}), [do])
    else:
        p = scm.counterfactual(var, dict(evidence, **(factual or {})), [do])
    if p is None:
        return False, p
    p_target = p if positive else 1 - p
    return p_target >= 0.5, p_target

In [22]:
_NAME = r"[A-Z][A-Za-z0-9_]*"
_CAUSE_VERB = re.compile(r"\b(?:together\s+)?causes?\b")


def _cb_literal(text):
    """'not Blaf' -> ('not', ('atom','Blaf',())),  'Blaf' -> ('atom','Blaf',())."""
    text = text.strip()
    neg = False
    while text.lower().startswith("not "):
        neg, text = not neg, text[4:].strip()
    m = re.fullmatch(_NAME, text)
    if not m:
        raise ValueError(f"Cannot read variable name: {text!r}")
    atom = ("atom", text, ())
    return ("not", atom) if neg else atom


def _cb_cause_expr(text):
    """'Blaf and not Trune' / 'Frim or Vork' / 'not Vork' -> formula."""
    text = re.sub(r"^\s*and\s+", "", text.strip())
    if " or " in text:
        return ("or", *[_cb_literal(t) for t in text.split(" or ")])
    if " and " in text:
        return ("and", *[_cb_literal(t) for t in text.split(" and ")])
    return _cb_literal(text)


def parse_causal_text(given_info):
    """CounterBench 'given_info' -> (equations, priors, evidence).
    'We know that A causes B, C, and D, not D causes E, E or B causes F.'
    'A~Bern(0.2).'  'We observed B and not C.'"""
    text = given_info.strip()
    priors = {}
    for m in re.finditer(rf"({_NAME})\s*~\s*Bern\(\s*([0-9.]+)\s*\)", text):
        priors[m.group(1)] = float(m.group(2))
    text = re.sub(rf"{_NAME}\s*~\s*Bern\(\s*[0-9.]+\s*\)\.?", " ", text)

    evidence = {}
    for m in re.finditer(r"We observed\s+([^.]*)", text):      # 'We observed A and not B', 'B not occurs'
        for part in m.group(1).split(" and "):
            part = part.strip()
            name = re.search(_NAME, part)
            if name:
                neg = part.startswith("not ") or bool(re.search(r"\bnot\s+occurs?\b", part))
                evidence[name.group(0)] = not neg
    text = re.sub(r"We observed[^.]*\.?", " ", text)

    m = re.search(r"We know that\s+(.*)", text, re.S)
    body = (m.group(1) if m else text).strip().rstrip(".")
    verbs = list(_CAUSE_VERB.finditer(body))
    equations = {}
    for i, v in enumerate(verbs):
        start = verbs[i - 1].end() if i else 0
        before = body[start:v.start()]
        cause = re.split(r",", before)[-1]                       # last part after the previous verb
        after_end = verbs[i + 1].start() if i + 1 < len(verbs) else len(body)
        after = body[v.end():after_end]
        effects_txt = after if i + 1 == len(verbs) else ",".join(after.split(",")[:-1])
        neg_effect = False
        effects_txt = effects_txt.strip()
        if effects_txt.startswith("not "):
            neg_effect, effects_txt = True, effects_txt[4:]
        expr = _cb_cause_expr(cause)
        for eff in re.split(r",\s*(?:and\s+)?|\s+and\s+", effects_txt):
            eff = eff.strip()
            if not eff:
                continue
            e_neg = neg_effect
            if eff.startswith("not "):
                e_neg, eff = True, eff[4:].strip()
            if not re.fullmatch(_NAME, eff):
                raise ValueError(f"Cannot read effect: {eff!r}")
            rhs = ("not", expr) if e_neg else expr
            if eff in equations:                                 # two cause clauses for one variable: OR
                equations[eff] = ("or", equations[eff], rhs)
            else:
                equations[eff] = rhs
    return equations, priors, evidence


def parse_causal_question(q):
    """Question -> (target, factual evidence, list of interventions).
    basic/conditional/backdoor : 'Would Y occur if not X instead of X?'
    joint                      : 'Would Y occur if not X and not V?'
    nested                     : 'Assume not X, and based on this assumption, further suppose not V. Would Y occur?'"""
    q = q.strip()
    m = re.match(rf"Would ({_NAME}) occur if (not )?({_NAME}) instead of (not )?({_NAME})\?", q)
    if m:
        y, neg_new, x, neg_old = m.group(1), bool(m.group(2)), m.group(3), bool(m.group(4))
        return {"kind": "simple", "target": y, "factual": {x: not neg_old}, "do": [{x: not neg_new}]}
    m = re.match(rf"Would ({_NAME}) occur if (.+)\?", q)
    if m:
        y, conds = m.group(1), m.group(2)
        do = {}
        for part in conds.split(" and "):
            lit = _cb_literal(part)
            if lit[0] == "not":
                do[lit[1][1]] = False
            else:
                do[lit[1]] = True
        return {"kind": "joint", "target": y, "factual": {}, "do": [do]}
    m = re.match(rf"Assume (not )?({_NAME}), and based on this assumption, further suppose (not )?({_NAME})\. "
                 rf"Would ({_NAME}) occur\?", q)
    if m:
        x, v, y = m.group(2), m.group(4), m.group(5)
        return {"kind": "nested", "target": y, "factual": {}, "x": (x, not m.group(1)), "v": (v, not m.group(3))}
    raise ValueError(f"Unknown question pattern: {q}")

# ---------- formal causal syntax ----------
# Each line of a causal knowledge base is recognised independently:
#   equation    : Blaf := ¬Ziklo   Blaf ≔ ¬Ziklo   Blaf = ¬Ziklo   Blaf := f(¬Ziklo)   Blaf ↔ ¬Ziklo
#                 Blaf ← ¬Ziklo   Blaf <- not Ziklo   ¬Ziklo → Blaf     Ziklo causes not Blaf
#   prior       : Blaf ~ Bern(0.2)   P(Blaf) = 0.2   P(Blaf=1) = 0.2
#   observation : Trune   ¬Trune   Trune = 1   Trune = 0
# Query:  do(¬Ziklo) → Lumbo    [do(¬Ziklo)] Lumbo    Lumbo | do(¬Ziklo)    P(Lumbo | do(Ziklo=0))
#         P(Lumbo=1 | do(Ziklo=0), Trune=1)    Lumbo_{¬Ziklo}    Lumbo_{Ziklo=0}    ¬Ziklo □→ Lumbo
#         do(¬X); do(¬V) → Y   (nested: both interventions)
_FNAME = r"[A-Za-z][A-Za-z0-9_]*"
_VAL = r"(?:0|1|true|false|T|F)"
_DO = re.compile(r"\bdo\s*\(([^()]*)\)")          # lower-case 'do(' only: a predicate such as 'Do(sam, x)' is not causal
_SUB = re.compile(rf"({_FNAME})\s*_\s*\{{([^{{}}]*(?:\([^()]*\)[^{{}}]*)*)\}}")       # Lumbo_{¬Ziklo}
_BOX = re.compile(r"□\s*→|◻\s*→|□->|\[\]\s*(?:->|→)|\bboxarrow\b")
_PRIOR = re.compile(rf"^(?:({_FNAME})\s*~\s*Bern(?:oulli)?\s*\(\s*([0-9.]+)\s*\)"
                    rf"|Pr?\s*\(\s*({_FNAME})\s*(?:=\s*(?:1|true|T))?\s*\)\s*=\s*([0-9.]+))$", re.I)
_ASSIGN = re.compile(rf"^({_FNAME})\s*(:=|≔|=(?![>=]))\s*(.+)$")
_CAUSES = re.compile(r"\bcauses?\s")


def is_causal_input(kb_texts, query):
    q = str(query)
    return bool(_DO.search(q) or _SUB.search(q) or _BOX.search(q) or re.match(r"\s*(Would|Assume)\b", q)) or \
        any(":=" in t or "≔" in t or re.search(r"~\s*Bern", t) or _CAUSES.search(t) for t in kb_texts)


def _canon(name):
    return name[0].upper() + name[1:]


def _truth(s):
    return s.strip().lower() in ("1", "true", "t")


def _is_var(f):
    return f[0] == "atom" and not f[2]


def _or(fs):
    return fs[0] if len(fs) == 1 else ("or", *fs)


def _add_eq(equations, v, rhs):
    """Two equations or cause clauses for one variable are combined with OR ('A causes V, B causes V')."""
    equations[v] = ("or", equations[v], rhs) if v in equations else rhs


def _parse_assignments(inner):
    """'¬Ziklo, Blaf=0' / '¬X ∧ ¬V' / 'X=0 and V=1' -> {var: value}."""
    out = {}
    for part in re.split(r",|;|∧|&|\band\b", inner):
        part = part.strip()
        if not part:
            continue
        m = re.fullmatch(rf"({_FNAME})\s*=\s*({_VAL})", part, re.I)
        if m:
            out[_canon(m.group(1))] = _truth(m.group(2))
            continue
        lit = _literal_list(parse_formula(part))
        if not lit or len(lit) != 1 or lit[0][2]:
            raise ValueError(f"Cannot read assignment: {part}")
        out[lit[0][1]] = lit[0][0]
    return out


_parse_do_list = _parse_assignments          # alias


def parse_causal_formal(kb_texts, query):
    """Formal causal KB + query -> (equations, priors, evidence, target, do)."""
    equations, priors, evidence, rules, cause_lines = {}, {}, {}, {}, []
    for t in kb_texts:
        t = t.strip().rstrip(".").strip()
        if not t:
            continue
        m = _PRIOR.match(t)
        if m:                                                          # prior
            name, p = (m.group(1), m.group(2)) if m.group(1) else (m.group(3), m.group(4))
            priors[_canon(name)] = float(p)
            continue
        if _CAUSES.search(t):                                          # 'Ziklo causes not Blaf'
            cause_lines.append(t)
            continue
        m = _ASSIGN.match(t)
        if m:                                                          # 'V := e' / 'V = e' / 'V = 1'
            name, rhs = _canon(m.group(1)), m.group(3).strip()
            fn = re.fullmatch(r"f[A-Za-z0-9_]*\s*\((.*)\)", rhs)        # 'f(¬Ziklo)' -> '¬Ziklo'
            rhs = fn.group(1) if fn else rhs
            if re.fullmatch(_VAL, rhs, re.I):
                evidence[name] = _truth(rhs)
            else:
                _add_eq(equations, name, parse_formula(rhs))
            continue
        f = parse_formula(t)
        if f[0] == "iff" and (_is_var(f[1]) or _is_var(f[2])):         # 'V ↔ e'
            v, rhs = (f[1], f[2]) if _is_var(f[1]) else (f[2], f[1])
            _add_eq(equations, v[1], rhs)
            continue
        if f[0] == "imp":                                              # 'e → V' / 'e → ¬V' / 'V ← e'
            head = f[2]
            pos = head[0] != "not"
            h = head if pos else head[1]
            if _is_var(h):
                rules.setdefault(h[1], ([], []))[0 if pos else 1].append(f[1])
                continue
        lits = _literal_list(f)
        if lits and all(not l[2] for l in lits):                       # observation
            for l in lits:
                evidence[l[1]] = l[0]
            continue
        raise ValueError(f"Cannot read causal KB line: {t}")
    if cause_lines:
        eq2, pri2, ev2 = parse_causal_text("We know that " + ", ".join(cause_lines) + ".")
        for v, rhs in eq2.items():
            _add_eq(equations, v, rhs)
        priors.update(pri2)
        evidence.update(ev2)
    for v, (pos, neg) in rules.items():                                # implications -> equation
        parts = ([_or(pos)] if pos else []) + ([("not", _or(neg))] if neg else [])
        _add_eq(equations, v, parts[0] if len(parts) == 1 else ("and", *parts))
    target, do, q_evidence = parse_causal_query(query)
    evidence.update(q_evidence)
    return equations, priors, evidence, target, do


def parse_causal_query(query):
    """Query -> (target (var, positive?), do {var: value}, observations {var: value})."""
    q = str(query).strip().rstrip(".?").strip()
    m = re.fullmatch(r"Pr?\s*\((.*)\)\s*(?:[<>]=?\s*[0-9.]+)?", q)     # P(Y | do(X=0))
    if m:
        q = m.group(1).strip()
    do = {}
    if _BOX.search(q):                                                  # '¬X □→ Y'
        ante, q = _BOX.split(q, 1)
        do.update(_parse_assignments(_DO.sub(lambda mm: mm.group(1), ante)))
    for mm in _SUB.finditer(q):                                         # 'Y_{¬X}'
        do.update(_parse_assignments(_DO.sub(lambda m2: m2.group(1), mm.group(2))))
    q = _SUB.sub(lambda mm: mm.group(1), q)
    for mm in _DO.finditer(q):                                          # 'do(¬X)'
        do.update(_parse_assignments(mm.group(1)))
    rest = _DO.sub(" ", q)
    rest = re.sub(r"[\[\];]|→|->|=>|⇒", " ", rest)
    left, _, right = rest.partition("|")
    observed = _parse_assignments(right) if right.strip(" ,") else {}
    left = left.strip(" ,")
    m = re.fullmatch(rf"({_FNAME})\s*=\s*({_VAL})", left, re.I)         # 'Y=1'
    if m:
        target = (_canon(m.group(1)), _truth(m.group(2)))
    else:
        lits = _literal_list(parse_formula(left)) if left else None
        if not lits or len(lits) != 1 or lits[0][2]:
            raise ValueError(f"Cannot read the target of the causal query: {query}")
        target = (lits[0][1], lits[0][0])
    if not do:
        raise ValueError(f"No intervention found in the causal query: {query}")
    return target, do, observed


def causal_predict(kb, query):
    """CounterBench-style input (English template ya formal) -> (answer, info)."""
    texts = split_kb(kb) if not isinstance(kb, str) or "We know that" not in kb else [kb]
    q = str(query).strip()
    if re.match(r"\s*(Would|Assume)\b", q):                           # English template
        eq, pri, ev = parse_causal_text(" ".join(texts))
        Q = parse_causal_question(q)
        if Q["kind"] == "nested":
            (x, xv), (v, vv) = Q["x"], Q["v"]
            return causal_answer(eq, pri, ev, (Q["target"], True), {x: xv, v: vv}, {x: not xv})
        factual = Q["factual"] if Q["kind"] == "simple" else {}
        return causal_answer(eq, pri, ev, (Q["target"], True), Q["do"][0], factual)
    eq, pri, ev, target, do = parse_causal_formal(texts, q)
    factual = {v: not val for v, val in do.items() if v not in eq}   # 'instead of': the actual value of an intervened root is the opposite
    return causal_answer(eq, pri, ev, target, do, factual)

In [23]:
# 1) A CounterBench-style question: text -> structural equations -> Pearl's three steps
given = "We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo."
question = "Would Lumbo occur if not Ziklo instead of Ziklo?"
eq, pri, ev = parse_causal_text(given)
print("1) Given    :", given)
print("   Equations:", {v: fml_str(f) for v, f in eq.items()})
print("   Question :", question, "->", parse_causal_question(question))
ans, p = causal_predict(given, question)
print(f"   Answer   : {'yes' if ans else 'no'} (P = {p})   gold: no")

# 2) Formal causal syntax
print("\n2) Formal syntax")
for kb, q in [(["Blaf := ¬Ziklo", "Vork := Blaf ∧ Trune", "Lumbo := Vork", "Trune"], "do(¬Ziklo) → Lumbo"),
              (["Ziklo := Blaf", "Trune := Ziklo ∨ Blaf", "Lumbo := Trune", "Blaf ~ Bern(0.2)", "Trune"],
               "do(Ziklo=0) → Lumbo"),
              (["Blaf = ¬Ziklo", "Lumbo = Blaf", "Ziklo = 1"], "P(Lumbo=1 | do(Ziklo=0))"),
              (["Ziklo → Blaf", "Blaf ∧ Trune → ¬Vork", "Trune"], "Vork_{¬Ziklo}")]:
    a, p = causal_predict(kb, q)
    print(f"   {kb} | {q}  ->  {a} (P = {p})")

# 3) A cyclic graph: the answer is undefined, so a clear error is raised
print("\n3) Causal cycle")
try:
    causal_predict(["Vork := ¬Sline", "Sline := Vork ∧ Blaf", "Lumbo := Vork", "Blaf"], "do(¬Blaf) → Lumbo")
except CausalCycleError as e:
    print("   ", type(e).__name__, "->", e)

1) Given    : We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.
   Equations: {'Blaf': 'Ziklo', 'Trune': 'Blaf', 'Vork': 'Trune', 'Lumbo': 'Vork'}
   Question : Would Lumbo occur if not Ziklo instead of Ziklo? -> {'kind': 'simple', 'target': 'Lumbo', 'factual': {'Ziklo': True}, 'do': [{'Ziklo': False}]}
   Answer   : no (P = 0.0)   gold: no

2) Formal syntax
   ['Blaf := ¬Ziklo', 'Vork := Blaf ∧ Trune', 'Lumbo := Vork', 'Trune'] | do(¬Ziklo) → Lumbo  ->  True (P = 1.0)
   ['Ziklo := Blaf', 'Trune := Ziklo ∨ Blaf', 'Lumbo := Trune', 'Blaf ~ Bern(0.2)', 'Trune'] | do(Ziklo=0) → Lumbo  ->  True (P = 1.0)
   ['Blaf = ¬Ziklo', 'Lumbo = Blaf', 'Ziklo = 1'] | P(Lumbo=1 | do(Ziklo=0))  ->  True (P = 1.0)
   ['Ziklo → Blaf', 'Blaf ∧ Trune → ¬Vork', 'Trune'] | Vork_{¬Ziklo}  ->  True (P = 1.0)

3) Causal cycle
    CausalCycleError -> causal cycle: Vork <- Sline <- Vork


## Section 6: Router and `predict()`

    

In [24]:
import difflib
from functools import lru_cache

CONFIG = {
    "semantics": "auto",       # 'auto' | 'cwa' | 'owa'
    "max_given": 50,           # resolution: deterministic budget (given clauses), chosen on FOLIO v2 train
    "time_limit": 2.0,         # resolution: safety net only (seconds)
    "max_clauses": 6000,       # resolution: memory safety net
    "cwa_engine": "backward",  # 'backward' | 'forward': engine for closed-world rule bases (Experiment 1)
    "align_query": True,       # map an unknown query symbol to a near-identical KB symbol (open world only)
    "align_threshold": 0.85,   # chosen on FOLIO v2 train
    "cwa_kb_vocab_only": True, # closed world only over the KB's own symbols; a query with an unknown symbol -> open world
    "query_free_vars": "auto", # 'auto' (literal query -> ∃, rule-like query -> ∀) | 'forall'  (Experiment 3, chosen on train)
    "default_answer": False,   # answer used when anything fails
}


class KnowledgeBase:
    """Everything about one KB: formulas, whether it is a rule base, closure, clauses.
    Each part is built on first use and cached."""

    def __init__(self, formulas, stats):
        self.formulas, self.stats = formulas, stats
        self.preds, self.consts = set(), set()
        for f in formulas:
            _symbols(f, self.preds, self.consts)
        self.has_equality = any(p == "=" for p, _ in self.preds)
        rules = []
        self.is_rulebase = not self.has_equality
        for f in (formulas if self.is_rulebase else []):
            r = formula_to_rules(f)
            if r is None:
                self.is_rulebase = False
                break
            rules += r
        self.rules = rules if self.is_rulebase else None
        self.neg_in_body = self.is_rulebase and any(not l[0] for b, _ in rules for l in b)
        # In RuleTaker-style rule bases every rule has at most one variable ("something ... it").
        self.max_rule_vars = max((len(set(v for l in b + [h] for v in lit_vars(l))) for b, h in rules),
                                 default=0) if self.is_rulebase else 99
        self._rb = self._axioms = self._bc = None

    @property
    def backward(self):
        if self._bc is None:
            self._bc = BackwardChainer(self.rules)
        return self._bc

    @property
    def rulebase(self):
        if self._rb is None:
            self._rb = RuleBase(self.rules)
        return self._rb

    @property
    def axioms(self):
        if self._axioms is None:
            ax = [c for f in self.formulas for c in to_clauses(f)]
            if self.has_equality:
                ax += [c for t in equality_axioms(self.preds) for c in to_clauses(parse_formula(t))]
            self._axioms = list(dict.fromkeys(ax))
        return self._axioms


def _symbols(f, preds, consts):
    """All (predicate, arity) pairs and constants of a formula."""
    op = f[0]
    if op == "atom":
        preds.add((f[1], len(f[2])))
        stack = list(f[2])
        while stack:
            t = stack.pop()
            if isinstance(t, tuple):
                stack.extend(t[1])
            elif not is_var(t):
                consts.add(t)
    elif op in ("forall", "exists"):
        _symbols(f[2], preds, consts)
    elif op not in ("true", "false"):
        for g in f[1:]:
            _symbols(g, preds, consts)


@lru_cache(maxsize=4096)
def _kb_cached(texts):
    formulas, _, stats = parse_kb(list(texts))
    return KnowledgeBase(formulas, stats)


def get_kb(kb):
    return _kb_cached(tuple(split_kb(kb)))


# ---------- query alignment (for annotation noise) ----------
def _key(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())


def _words(name):
    """'InSouthernChina' -> ['in', 'southern', 'china'],  'y1984' -> ['y', '1984']."""
    return [w.lower() for w in re.findall(r"[A-Z]?[a-z]+|[A-Z]+(?![a-z])|\d+", name)]


_NEGATION_PREFIXES = ("in", "un", "non", "dis", "il", "im", "ir", "not", "a")


def _same_words(a, b):
    """Accept only small differences: typos, plurals, one-letter prefixes (Name~Named, Studen~Student, y1984~1984).
    Reject changed words: Southern~Northern, Male~Female, Legal~Illegal."""
    wa, wb = _words(a), _words(b)
    for w in list(wa):
        if w in wb:
            wa.remove(w)
            wb.remove(w)
    if len(wa) != len(wb):                       # an extra word on one side: only a single letter is accepted
        extra = wa[len(wb):] + wb[len(wa):]
        if any(len(w) > 1 for w in extra):
            return False
        wa, wb = wa[:len(wb)], wb[:len(wa)]
    for x, y in zip(wa, wb):
        if any(x == n + y or y == n + x for n in _NEGATION_PREFIXES):
            return False                         # Active ~ Inactive: opposite meaning
        if not (x.startswith(y) or y.startswith(x)) and \
                difflib.SequenceMatcher(None, x, y).ratio() < 0.85:
            return False
    return True


def _closest(name, candidates, threshold):
    best, best_r = None, 0.0
    for c in candidates:
        if not _same_words(name, c):
            continue
        r = difflib.SequenceMatcher(None, _key(name), _key(c)).ratio()
        if r > best_r:
            best, best_r = c, r
    return best if best_r >= threshold else None


def _rename(f, pmap, cmap):
    op = f[0]
    if op == "atom":
        def rt(t):
            return (t[0], tuple(rt(a) for a in t[1])) if isinstance(t, tuple) else cmap.get(t, t)
        return ("atom", pmap.get((f[1], len(f[2])), f[1]), tuple(rt(a) for a in f[2]))
    if op in ("forall", "exists"):
        return (op, f[1], _rename(f[2], pmap, cmap))
    if op in ("true", "false"):
        return f
    return (op, *[_rename(g, pmap, cmap) for g in f[1:]])


def align_query(K, qf, threshold):
    """Replace a query predicate/constant that does not occur in the KB by the most similar KB symbol
    (same arity, similarity ≥ threshold), e.g. y1984 -> 1984, NameAfter -> NamedAfter."""
    qp, qc = set(), set()
    _symbols(qf, qp, qc)
    pmap = {}
    for p, n in qp - K.preds:
        b = _closest(p, [pp for pp, nn in K.preds if nn == n], threshold)
        if b:
            pmap[(p, n)] = b
    cmap = {}
    for c in qc - K.consts:
        b = _closest(c, K.consts, threshold)
        if b:
            cmap[c] = b
    return (_rename(qf, pmap, cmap), pmap, cmap) if (pmap or cmap) else (qf, {}, {})


def _has_op(f, ops):
    if f[0] in ops:
        return True
    if f[0] in ("atom", "true", "false"):
        return False
    if f[0] in ("forall", "exists"):
        return _has_op(f[2], ops)
    return any(_has_op(g, ops) for g in f[1:])


def _bind_free(f, force=None):
    """Free variables of a query (x, y without a quantifier).
    'ControlledBy(x, manager)'  -> ∃x : 'is there an x such that ...' (as in a Prolog query)
    'Mammal(x) → Animal(x)'     -> ∀x : a rule-like query, for every x"""
    free = []
    while f[0] == "free":
        free.append(f[1])
        f = f[2]
    rule_like = CONFIG["query_free_vars"] == "forall" or _has_op(f, ("imp", "iff"))
    q = force or ("forall" if rule_like else "exists")
    for v in reversed(free):
        f = (q, v, f)
    return f


def _ground_literals(qf):
    lits = _literal_list(qf)
    if lits is None or any(lit_vars(l) or any(isinstance(a, tuple) for a in l[2]) for l in lits):
        return None
    return lits


# ---------- main entry point ----------
def predict_detail(kb, query, semantics=None, want_trace=False):
    """Answer, engine, status and reasoning trace for one query. Never raises.
    want_trace=True also builds the forward-chaining proof tree (used for the traces in Section 11)."""
    semantics = semantics or CONFIG["semantics"]
    try:
        texts = split_kb(kb) if not (isinstance(kb, str) and "We know that" in kb) else [kb]
        if is_causal_input(texts, query):                                   # 1) counterfactual
            try:
                ans, p = causal_predict(kb, query)
                return {"answer": ans, "engine": "causal (SCM)", "status": f"P={p}", "trace": []}
            except CausalCycleError:
                raise                                                       # a genuine causal model whose answer is undefined
            except Exception:
                pass                                                        # looked causal but is not: use the logic routes

        K = get_kb(kb)
        up = K.stats.get("upper_vars", False) or bool(_UPPER_VAR_MARKERS.search(str(query)))
        qf = _bind_free(parse_formula(query, upper_vars=up, free_as="free"), "exists" if up else None)
        qlits = _ground_literals(qf)

        qp, qc = set(), set()
        _symbols(qf, qp, qc)
        # The closed world closes the KB's own vocabulary (its Herbrand base). If a query predicate or constant
        # does not occur in the KB, the KB says nothing about it -> open world.
        in_vocab = not (qp - K.preds) and not (qc - K.consts)
        use_cwa = semantics == "cwa" or (semantics == "auto" and K.is_rulebase and K.max_rule_vars <= 1 and
                                         (in_vocab or not CONFIG["cwa_kb_vocab_only"]))
        if K.is_rulebase and qlits is not None and use_cwa:                 # 2) closed world
            # Backward chaining explores only the goals the query needs (Experiment 1: faster, and correct on
            # non-stratified rule bases). Forward chaining computes the closure and proof trees (for traces).
            # Backward chaining is used only for rules with at most one variable.
            if (want_trace or CONFIG["cwa_engine"] == "forward" or K.max_rule_vars > 1) and \
                    (K.rulebase.stratified or K.max_rule_vars > 1):
                rb = K.rulebase
                ans = all(rb.holds(l) for l in qlits)
                return {"answer": ans, "engine": "forward-chaining (CWA)", "status": "closure",
                        "trace": _fc_trace(rb, qlits)}
            ans = all(K.backward.holds(l) for l in qlits)
            return {"answer": ans, "engine": "backward-chaining (CWA)", "status": "goal-directed", "trace": []}

        if K.is_rulebase and not K.neg_in_body and qlits is not None and all(l[0] for l in qlits):
            rb = K.rulebase                                                 # 3) Horn KB, positive query
            ans = all(rb.holds(l) for l in qlits)
            return {"answer": ans, "engine": "forward-chaining (Horn, OWA)", "status": "closure",
                    "trace": _fc_trace(rb, qlits) if want_trace else []}

        note = ""                                                           # 4) open world: resolution
        if CONFIG["align_query"]:
            qf, pmap, cmap = align_query(K, qf, CONFIG["align_threshold"])
            if pmap or cmap:
                note = f" aligned {pmap or ''}{cmap or ''}"
        prover = ResolutionProver(K.axioms, CONFIG["time_limit"], CONFIG["max_clauses"],
                                  max_given=CONFIG["max_given"])
        proved, status, proof = prover.refute(to_clauses(("not", qf)))
        return {"answer": proved, "engine": "resolution (OWA)", "status": status + note, "trace": proof}
    except Exception as e:                                                  # never crash
        return {"answer": CONFIG["default_answer"], "engine": "fallback",
                "status": f"error: {type(e).__name__}: {str(e)[:120]}", "trace": []}


def _fc_trace(rb, qlits):
    lines = []
    for l in qlits:
        r = rb._for_constants(l[2])
        key = (l[1], l[2])
        if l[0] and key in r.why:
            lines += r.explain(key)
        elif l[0]:
            lines.append(f"{lit_str(l)} is not derivable from the facts and rules")
        elif key in r.why:
            lines.append(f"{lit_str(l)} is FALSE because {lit_str((True,) + key)} is derivable:")
            lines += r.explain(key, 1)
        elif key in r.neg_derived:
            ri, body = r.neg_derived[key]
            lines.append(f"{lit_str(l)}   [rule {ri} with {' ∧ '.join(map(lit_str, body))}]")
        else:
            lines.append(f"{lit_str(l)}   [{lit_str((True,) + key)} is not derivable → "
                         f"true under the closed-world assumption]")
    return lines


def predict(kb, query, semantics=None):
    """True / False only."""
    return predict_detail(kb, query, semantics)["answer"]

## Section 7: Sanity Tests
Hand-checked cases for reasoning, semantics, dialects, input formats, robustness, causal queries and query
variables.

In [25]:
B = ["Young(bob)", "Kind(bob)", "∀x (Young(x) → Smart(x))", "∀x (Smart(x) ∧ Kind(x) → Successful(x))"]
TESTS = [
    # (category, name, KB, query, expected)
    ("Reasoning", "assignment example (2-step chain)", B, "Successful(bob)", True),
    ("Reasoning", "not derivable -> False", ["Young(bob)", "∀x (Young(x) → Smart(x))"], "Successful(bob)", False),
    ("Reasoning", "binary relation + unification", ["Chase(rabbit, tiger)", "∀x (Chase(x, tiger) → Rough(x))"], "Rough(rabbit)", True),
    ("Reasoning", "reasoning by cases (∨)", ["∀x (Drinks(x) → Dependent(x))", "∀x (Drinks(x) ∨ Jokes(x))", "¬Jokes(rina)"], "Dependent(rina)", True),
    ("Reasoning", "exclusive or", ["∀x (P(x) ⊕ Q(x))", "P(a)"], "¬Q(a)", True),
    ("Reasoning", "existential (Skolem)", ["∀x (Man(x) → Mortal(x))", "∃x Man(x)"], "∃y Mortal(y)", True),
    ("Reasoning", "transitivity", ["∀x ∀y ∀z (Taller(x, y) ∧ Taller(y, z) → Taller(x, z))", "Taller(a, b)", "Taller(b, c)"], "Taller(a, c)", True),
    ("Reasoning", "universal query", ["∀x (Young(x) → Smart(x))", "∀x (Smart(x) → Kind(x))"], "∀x (Young(x) → Kind(x))", True),
    ("Reasoning", "tautology, empty KB", [], "Young(bob) ∨ ¬Young(bob)", True),
    ("Reasoning", "equality", ["a = b", "Young(a)"], "Young(b)", True),
    ("Semantics", "closed world: unknown -> not", ["Quiet(anne)", "Big(bob)"], "¬Big(anne)", True),
    ("Semantics", "symbol outside KB -> open world", ["Quiet(anne)"], "¬Big(anne)", False),
    ("Semantics", "negation as failure in a body", ["Quiet(anne)", "∀x (Quiet(x) ∧ ¬Kind(x) → Young(x))"], "Young(anne)", True),
    ("Semantics", "new constant in query -> open world", ["Quiet(a)", "∀x (¬Kind(x) → Lonely(x))"], "Lonely(charlie)", False),
    ("Semantics", "open world: unknown is not entailed", ["∀x (P(x) ∨ Q(x))", "P(a)"], "Q(a)", False),
    ("Semantics", "modus tollens (open world)", ["∀x (Bird(x) ∨ Fish(x))", "∀x (Bird(x) → Fly(x))", "¬Fly(rex)"], "Fish(rex)", True),
    ("Semantics", "quantifier scope (bracket)", ["¬∃x (Job(x) ∧ Has(james, x)) ∧ ¬Works(james)"], "¬Works(james)", True),
    ("Dialect", "ASCII", ["Young(bob)", "forall x (Young(x) -> Smart(x))"], "Smart(bob)", True),
    ("Dialect", "words AND/IMPLIES", ["Young(bob) AND Kind(bob)", "FORALL x (Young(x) AND Kind(x) IMPLIES Happy(x))"], "Happy(bob)", True),
    ("Dialect", "Prover9", ["Young(bob).", "all x (Young(x) -> Smart(x))."], "Smart(bob)", True),
    ("Dialect", "Prolog + \\+", ["quiet(anne).", "young(X) :- quiet(X), \\+ kind(X)."], "young(anne)", True),
    ("Dialect", "TPTP", ["fof(a1, axiom, young(bob)).", "fof(a2, axiom, ![X]: (young(X) => smart(X)))."], "fof(c, conjecture, smart(bob)).", True),
    ("Dialect", "LaTeX", ["Young(bob)", r"\forall x (Young(x) \rightarrow Smart(x))"], "Smart(bob)", True),
    ("Dialect", "LaTeX ∨", [r"\forall x (A(x) \lor B(x))", r"\forall x (A(x) \to C(x))", r"\forall x (B(x) \to C(x))"], "C(k)", True),
    ("Dialect", "textbook ⊃ ≡ (∀x)", ["Young(bob)", "(∀x)(Young(x) ⊃ Smart(x))", "∀x (Smart(x) ≡ Clever(x))"], "Clever(bob)", True),
    ("Dialect", "FLD {A}{a}, (x):, v", ["(x): {C}x -> (¬{A}x v ¬{B}x)", "{C}{a}", "{A}{a}"], "¬{B}{a}", True),
    ("Dialect", "z3py ForAll/Implies/Or/Not", ["ForAll([x], Implies(Dog(x), Or(Brown(x), Black(x))))", "Dog(rex)", "Not(Brown(rex))"], "Black(rex)", True),
    ("Dialect", "Logic-LM / Pyke >>> && False", ["Big($x, True) && Red($x, False) >>> Kind($x, True)", "Big(Bob, True)", "Red(Bob, False)"], "Kind(Bob, True)", True),
    ("Dialect", "SMT-LIB assert/forall/=>", ["(declare-fun Dog (U) Bool)", "(assert (forall ((x U)) (=> (Dog x) (Mammal x))))", "(assert (Dog rex))"], "(Mammal rex)", True),
    ("Dialect", "Lean ∀ x, P x → Q x", ["∀ x y, Parent x y → ¬ Parent y x", "Parent tom bob"], "¬ Parent bob tom", True),
    ("Dialect", "set notation ∈ ⊆", ["Dog ⊆ Mammal", "rex ∈ Dog"], "rex ∈ Mammal", True),
    ("Dialect", "RuleTaker triples", ['("Bob" "is" "young" "+")', '((("someone" "is" "young" "+")) -> ("someone" "is" "smart" "+"))'], '("Bob" "is" "smart" "+")', True),
    ("Input format", "KB one string with '.'", "Young(bob). Kind(bob). ∀x (Young(x) → Smart(x)).", "Smart(bob)", True),
    ("Input format", "numbered premises", "1. Young(bob)\n2. ∀x (Young(x) → Smart(x))", "Smart(bob)", True),
    ("Input format", "JSON list string", '["Young(bob)", "∀x (Young(x) → Smart(x))"]', "Smart(bob)", True),
    ("Input format", "case Bob/bob, young/Young", ["young(Bob)", "∀x (Young(x) → Smart(x))"], "smart(bob)", True),
    ("Robustness", "one broken formula is dropped", ["Young(bob)", "%%% ???", "∀x (Young(x) → Smart(x))"], "Smart(bob)", True),
    ("Robustness", "garbage query -> fallback False", ["Young(bob)"], "%%%", False),
    ("Robustness", "alignment: typo y1984 -> 1984", ["∀x (StreamingService(x) → ¬Book(x))", "StreamingService(1984)", "Book(a) ∨ Book(b)"], "¬Book(y1984)", True),
    ("Robustness", "alignment refuses opposite meaning", ["InNorthernChina(beijing)", "Big(a) ∨ Big(b)"], "InSouthernChina(beijing)", False),
    ("Causal", "CounterBench English", "We know that Ziklo causes Blaf, Blaf causes Trune.", "Would Trune occur if not Ziklo instead of Ziklo?", False),
    ("Causal", "formal do()", ["Blaf := ¬Ziklo", "Vork := Blaf ∧ Trune", "Lumbo := Vork", "Trune"], "do(¬Ziklo) → Lumbo", True),
    ("Causal", "notation: =, V=1, P(Y=1 | do(X=1))", ["Blaf = ¬Ziklo", "Lumbo = Blaf", "Ziklo = 0"], "P(Lumbo=1 | do(Ziklo=1))", False),
    ("Causal", "notation: implications, Y_{¬X}", ["Ziklo → Blaf", "Blaf ∧ Trune → ¬Vork", "Trune"], "Vork_{¬Ziklo}", True),
    ("Causal", "notation: ← / <-, X □→ Y", ["Blaf ← ¬Ziklo", "Lumbo <- Blaf"], "Ziklo □→ Lumbo", False),
    ("Causal", "notation: causes, f(), Bern", ["Ziklo causes not Blaf", "Lumbo := f(Blaf)", "P(Ziklo) = 0.3"], "do(¬Ziklo) → Lumbo", True),
    ("Query vars", "free x in literal query = ∃", ["∀x (Device(x) → Connected(x))", "Device(modelxx)"], "Connected(x)", True),
    ("Query vars", "free x in rule query = ∀", ["∀x (A(x) → B(x))", "A(c)"], "A(x) → B(x)", True),
    ("Query vars", "free x in rule query not ∃", ["A(c)", "B(c)"], "A(x) → B(x)", False),
]
by_cat = {}
for cat, name, kb, q, exp in TESTS:
    d = predict_detail(kb, q)
    ok = d["answer"] == exp
    by_cat.setdefault(cat, []).append(ok)
    print(f"{'PASS' if ok else 'FAIL'}  [{cat:12s}] {name:40s} -> {d['answer']!s:5s} [{d['engine']}]")
print()
for cat, oks in by_cat.items():
    print(f"   {cat:12s} {sum(oks)}/{len(oks)}")
print(f"TOTAL: {sum(map(sum, by_cat.values()))}/{len(TESTS)} sanity tests passed")

PASS  [Reasoning   ] assignment example (2-step chain)        -> True  [backward-chaining (CWA)]
PASS  [Reasoning   ] not derivable -> False                   -> False [forward-chaining (Horn, OWA)]
PASS  [Reasoning   ] binary relation + unification            -> True  [backward-chaining (CWA)]
PASS  [Reasoning   ] reasoning by cases (∨)                   -> True  [resolution (OWA)]
PASS  [Reasoning   ] exclusive or                             -> True  [resolution (OWA)]
PASS  [Reasoning   ] existential (Skolem)                     -> True  [resolution (OWA)]
PASS  [Reasoning   ] transitivity                             -> True  [forward-chaining (Horn, OWA)]
PASS  [Reasoning   ] universal query                          -> True  [resolution (OWA)]
PASS  [Reasoning   ] tautology, empty KB                      -> True  [resolution (OWA)]
PASS  [Reasoning   ] equality                                 -> True  [resolution (OWA)]
PASS  [Semantics   ] closed world: unknown -> not             

## Section 8: Public Data as Logic

### 8a: RuleTaker English → formal logic (development adapter)
The test data is formal logic. The public RuleTaker contexts are English, so they are converted into the same
formal syntax with regular-expression templates and the whole pipeline is evaluated on the result.



In [26]:
import re

_VAR_WORDS = {'something', 'someone', 'it', 'they', 'he', 'she'}
_ADJ = re.compile(r"^(?P<s>.+?) (?:is|are) (?P<neg>not )?(?P<a>\w+)$")
_NEGREL = re.compile(r"^(?P<s>.+?) (?:does|do) not (?P<v>\w+) (?P<o>.+)$")
_REL = re.compile(r"^(?P<s>something|someone|it|they|[Tt]he \w+ \w+|[Tt]he \w+|(?!The\b)[A-Z]\w*) "
                  r"(?!the\b|is\b|are\b|not\b)(?P<v>\w+) (?P<o>.+)$")


def _entity(text):
    t = text.strip()
    if t.lower() in _VAR_WORDS:
        return 'x'
    t = re.sub(r'^the ', '', t, flags=re.I)
    return re.sub(r'\W+', '_', t.strip().lower())


def _verb(v, subj):
    v = v.lower()
    if subj.lower() != 'they' and v.endswith('s') and not v.endswith('ss'):
        v = v[:-1]                          # chases -> chase, sees -> see
    return v.capitalize()


def _clause(text, prev_subj=None):
    """'the rabbit chases the tiger' -> (literal_string, subject_text) or None."""
    text = text.strip()
    for attempt in (text, f"{prev_subj} is {text}" if prev_subj else None):
        if not attempt:
            continue
        m = _ADJ.match(attempt)
        if m:
            lit = f"{m['a'].capitalize()}({_entity(m['s'])})"
            return ('¬' if m['neg'] else '') + lit, m['s']
        m = _NEGREL.match(attempt)
        if m:
            return f"¬{_verb(m['v'], 'they')}({_entity(m['s'])}, {_entity(m['o'])})", m['s']
        m = _REL.match(attempt)
        if m:
            return f"{_verb(m['v'], m['s'])}({_entity(m['s'])}, {_entity(m['o'])})", m['s']
    return None


def _clauses(text):
    out, subj = [], None
    for part in re.split(r' and ', text):
        r = _clause(part, subj)
        if r is None:
            return None
        out.append(r[0])
        subj = r[1]
    return out


def _wrap(body, head):
    f = (' ∧ '.join(body) + ' → ' if body else '') + ' ∧ '.join(head)
    uses_var = re.search(r'[(, ]x[,)]', f) is not None
    return f"∀x ({f})" if uses_var else (f"({f})" if body else f)


def ruletaker_sentence(s):
    """One RuleTaker sentence -> formal formula string, or None if no template fits."""
    s = s.strip().rstrip('.').strip()
    m = re.match(r'^If (.+?) then (.+)$', s)
    if m:
        body, head = _clauses(m[1]), _clauses(m[2])
        return None if body is None or head is None else _wrap(body, head)
    m = re.match(r'^(?:All )?(.+?) (?:things|people|ones) are (not )?(\w+)$', s, flags=re.I)
    if m:
        adjs = [a for a in re.split(r',\s*|\s+and\s+|\s+', m[1].strip()) if a]
        body = [f"{a.capitalize()}(x)" for a in adjs]
        head = [('¬' if m[2] else '') + f"{m[3].capitalize()}(x)"]
        return _wrap(body, head)
    r = _clauses(s)
    return None if r is None else ' ∧ '.join(r)


def ruletaker_to_formal(context):
    sents = [x for x in re.split(r'(?<=\.)\s+', context.strip()) if x.strip()]
    out = []
    for x in sents:
        f = ruletaker_sentence(x)
        if f is None:
            return None
        out.append(f)
    return out

### 8b: Items
(knowledge base, query, gold label) items for the tune and report splits of Section 2b. RuleTaker: 500 tune and
3,000 report questions per depth configuration; FOLIO: train / validation; CounterBench: tune / held-out half.
The cell also reports how many FOLIO train formulas the parser reads.

In [27]:
# Splits come from Section 2: tune = exploration and choice of settings, report = accuracy only.
RT_SAMPLE_PER_CONFIG = 3000      # RuleTaker report items per depth config (None = whole report split)
RT_TUNE_PER_CONFIG = 500         # RuleTaker tune items per depth config (used by the experiments)


def ruletaker_items(df, per_config=None, seed=SEED):
    """RuleTaker (English) -> formal logic items (development adapter, Section 8a)."""
    df = df[~df["config"].str.contains("NatLang")]
    if per_config:
        df = df.sample(frac=1, random_state=seed).groupby("config").head(per_config)
    cache, items, failed = {}, [], 0
    for r in df.itertuples():
        if r.context not in cache:
            cache[r.context] = ruletaker_to_formal(r.context)
        kb, q = cache[r.context], ruletaker_sentence(r.question)
        if kb is None or q is None:
            failed += 1
            continue
        items.append((kb, q, r.label == "entailment", r.config))
    print(f"RuleTaker adapter: {len(items)}/{len(items) + failed} questions converted to formal logic")
    return items


def folio_items(df):
    return [(r["premises-FOL"], r["conclusion-FOL"], str(r["label"]) == "True", str(r["label"]))
            for _, r in df.iterrows()]


def counterbench_items(df):
    """CounterBench items with the English template as given in the dataset."""
    return [(r["given_info"], r["question"], r["answer"] == "yes", r["type"]) for _, r in df.iterrows()]


def counterbench_to_formal(given, question):
    """CounterBench (English template) -> formal causal logic: 'V := e', 'V ~ Bern(p)', observations,
    query 'do(..) → Y'. The actual X of 'not X instead of X' is written as an observation."""
    eq, pri, ev = parse_causal_text(given)
    Q = parse_causal_question(question)
    lit = lambda v, b: v if b else "¬" + v
    if Q["kind"] == "nested":
        do, factual = {Q["x"][0]: Q["x"][1], Q["v"][0]: Q["v"][1]}, {Q["x"][0]: not Q["x"][1]}
    else:
        do, factual = Q["do"][0], (Q["factual"] if Q["kind"] == "simple" else {})
    kb = [f"{v} := {fml_str(e)}" for v, e in eq.items()] + [f"{v} ~ Bern({p})" for v, p in pri.items()]
    kb += [lit(v, b) for v, b in dict(ev, **factual).items()]
    query = "do(" + ", ".join(lit(v, b) for v, b in do.items()) + f") → {Q['target']}"
    return kb, query


RT_ITEMS = ruletaker_items(RT_REPORT_DF, RT_SAMPLE_PER_CONFIG) if HAVE_RT else []                     # report
RT_TUNE_ITEMS = ruletaker_items(RT_TUNE_DF, RT_TUNE_PER_CONFIG) if HAVE_RT else []                   # tune
FOLIO_VAL = folio_items(folio["validation"]) if HAVE_FOLIO else []                                   # report
FOLIO_TRAIN = folio_items(folio["train"]) if HAVE_FOLIO and "train" in folio else []                  # tune
CB_ITEMS, CB_TUNE, CB_HELDOUT = [], [], []
if HAVE_CB:
    CB_ITEMS = counterbench_items(counterbench)                                                       # experiments
    CB_TUNE = counterbench_items(counterbench[counterbench.split == "tune"])                          # tune
    CB_HELDOUT = counterbench_items(counterbench[counterbench.split == "heldout"])                    # report
    ex = counterbench.iloc[0]
    print("\nEnglish -> formal causal logic:", ex["question"])
    print("   ", counterbench_to_formal(ex["given_info"], ex["question"]))
if FOLIO_TRAIN:                                   # parser coverage on the tune split
    n_f = n_drop = 0
    for kb, _, _, _ in FOLIO_TRAIN:
        st = parse_kb(kb)[2]
        n_f += st["formulas"]
        n_drop += len(st["dropped"])
    print(f"\nParser on FOLIO train premises: {n_f - n_drop}/{n_f} formulas parsed ({n_drop} dropped)")
print(f"\nItems: RuleTaker tune {len(RT_TUNE_ITEMS)} / report {len(RT_ITEMS)} | "
      f"FOLIO train {len(FOLIO_TRAIN)} / validation {len(FOLIO_VAL)} | "
      f"CounterBench tune {len(CB_TUNE)} / held-out {len(CB_HELDOUT)}")

RuleTaker adapter: 18000/18000 questions converted to formal logic
RuleTaker adapter: 3000/3000 questions converted to formal logic

English -> formal causal logic: Would Lumbo occur if not Ziklo instead of Ziklo?
    (['Blaf := Ziklo', 'Trune := Blaf', 'Vork := Trune', 'Lumbo := Vork', 'Ziklo'], 'do(¬Ziklo) → Lumbo')

Parser on FOLIO train premises: 5279/5286 formulas parsed (7 dropped)

Items: RuleTaker tune 3000 / report 18000 | FOLIO train 1001 / validation 203 | CounterBench tune 600 / held-out 600


## Section 9: Evaluation on the Public Data
Accuracy, time, errors and budget-outs for every set. The **combined accuracy** uses only the report sets
(RuleTaker test, FOLIO validation, CounterBench held-out); the tune sets (RuleTaker tune, FOLIO train, CounterBench
tune) are shown separately. Queries the engines
cannot answer (for example a causal cycle) are listed with the reason.

In [28]:
def evaluate(items, name, semantics=None, verbose=True):
    """items: [(kb, query, gold_bool, group), ...] -> (per-item DataFrame, accuracy, seconds).
    The cache is cleared first so that timings are comparable."""
    _kb_cached.cache_clear()
    rows = []
    t0 = time.perf_counter()
    for kb, q, gold, grp in items:
        d = predict_detail(kb, q, semantics)
        rows.append((grp, gold, d["answer"], d["engine"], d["status"]))
    secs = time.perf_counter() - t0
    res = pd.DataFrame(rows, columns=["group", "gold", "pred", "engine", "status"])
    acc = 100 * (res.gold == res.pred).mean() if len(res) else float("nan")
    if verbose:
        n_err = (res.engine == "fallback").sum()
        n_to = res.status.str.startswith(("timeout", "clause-limit", "step-limit")).sum()
        print(f"{name}: accuracy {acc:.2f}% on {len(res)} queries | {secs:.2f}s "
              f"({1000 * secs / max(len(res), 1):.2f} ms/query) | errors {n_err} | budget-outs {n_to}")
    return res, acc, secs


def per_group(res):
    return (res.assign(correct=res.gold == res.pred).groupby("group")["correct"]
               .agg(["mean", "count"]).rename(columns={"mean": "accuracy"}).round(4))


# report = sets whose accuracy is reported; tune = sets used to choose settings (excluded from the combined accuracy)
EVAL_SETS = [("RuleTaker (test)", RT_ITEMS, "report"),
             ("FOLIO v2 (validation)", FOLIO_VAL, "report"),
             ("CounterBench (held-out)", CB_HELDOUT, "report"),
             ("RuleTaker (tune)", RT_TUNE_ITEMS, "tune"),
             ("FOLIO v2 (train)", FOLIO_TRAIN, "tune"),
             ("CounterBench (tune)", CB_TUNE, "tune")]
EVAL, ITEMS_OF = {}, {}
for name, items, role in EVAL_SETS:
    if not items:
        print(f"{name}: data not loaded, evaluation skipped")
        continue
    res, acc, secs = evaluate(items, name)
    EVAL[name], ITEMS_OF[name] = res, items
    RESULTS[name] = (acc, len(res), secs, role)

if "RuleTaker (test)" in EVAL:
    print("\nRuleTaker by proof depth:")
    print(per_group(EVAL["RuleTaker (test)"]))
if "FOLIO v2 (validation)" in EVAL:
    print("\nFOLIO validation, gold label vs prediction:")
    print(pd.crosstab(EVAL["FOLIO v2 (validation)"].group, EVAL["FOLIO v2 (validation)"].pred))
cb_sets = [k for k in ("CounterBench (tune)", "CounterBench (held-out)") if k in EVAL]
if cb_sets:
    print("\nCounterBench by type (tune vs held-out):")
    print(pd.concat({k: per_group(EVAL[k])["accuracy"] for k in cb_sets}, axis=1).to_string())
if EVAL:
    print("\nEngine usage:")
    print(pd.concat({k: v.engine.value_counts() for k, v in EVAL.items()}, axis=1).fillna(0).astype(int).to_string())

# Queries the engines could not answer are listed with the reason (they are answered False)
print("\nEngine failures (answered False), with reason:")
n_fail = 0
for name, res in EVAL.items():
    for i in res.index[res.engine == "fallback"]:
        kb, q, gold, grp = ITEMS_OF[name][i]
        kb_txt = kb if isinstance(kb, str) else " | ".join(split_kb(kb))
        print(f"  [{name}] #{i} ({grp}) gold={gold} -> {res.status[i]}")
        print(f"      KB: {kb_txt[:150]}{'...' if len(kb_txt) > 150 else ''}")
        print(f"      Q : {q}")
        n_fail += 1
print(f"  total: {n_fail}" if n_fail else "  none")
if HAVE_CB:
    cyc = [(r.problem, r.split, r.question_id) for r in counterbench.itertuples()
           if "CausalCycleError" in predict_detail(r.given_info, r.question)["status"]]
    if cyc:
        probs = sorted({p for p, _, _ in cyc})
        print(f"\nCounterBench cyclic graphs: {len(cyc)} questions = {len(probs)} problem(s) x renamed copies "
              f"{probs} (split: {sorted({s for _, s, _ in cyc})}). A cycle in the structural equations leaves "
              f"do() without a unique answer; the binary output is False.")

# Combined accuracy over the report sets only (tune sets excluded)
rep = [(k, v) for k, v in RESULTS.items() if isinstance(v, tuple) and len(v) == 4 and v[3] == "report"]
if rep:
    n = sum(v[1] for _, v in rep)
    correct = sum(v[0] * v[1] / 100 for _, v in rep)
    print(f"\nCOMBINED ACCURACY ({', '.join(k for k, _ in rep)}): {100 * correct / n:.2f}% "
          f"({round(correct)}/{n})")

RuleTaker (test): accuracy 100.00% on 18000 queries | 24.69s (1.37 ms/query) | errors 0 | budget-outs 0
FOLIO v2 (validation): accuracy 85.22% on 203 queries | 2.74s (13.48 ms/query) | errors 0 | budget-outs 42
CounterBench (held-out): accuracy 93.33% on 600 queries | 0.09s (0.15 ms/query) | errors 5 | budget-outs 0
RuleTaker (tune): accuracy 100.00% on 3000 queries | 4.58s (1.53 ms/query) | errors 0 | budget-outs 0
FOLIO v2 (train): accuracy 81.12% on 1001 queries | 7.29s (7.28 ms/query) | errors 0 | budget-outs 167
CounterBench (tune): accuracy 94.17% on 600 queries | 0.09s (0.15 ms/query) | errors 0 | budget-outs 0

RuleTaker by proof depth:
            accuracy  count
group                      
depth-0          1.0   3000
depth-1          1.0   3000
depth-2          1.0   3000
depth-3          1.0   3000
depth-3ext       1.0   3000
depth-5          1.0   3000

FOLIO validation, gold label vs prediction:
pred       False  True 
group                  
False         59      3
True  

## Section 10: Experiments
The experiments are the evidence for the settings used by the router. Every setting is chosen on a **tune** split;
report splits appear only as reference columns. The cells run only with `RUN_EXPERIMENTS = True` (Section 1); an
error inside one experiment is reported and the notebook continues.


In [29]:
if RUN_EXPERIMENTS and HAVE_RT:
    try:
        from collections import defaultdict
        # Experiment 1 runs on the RuleTaker TUNE split: whole contexts (all questions of each context)
        rt = RT_TUNE_DF
        chosen = rt["context"].drop_duplicates().sample(min(800, rt["context"].nunique()), random_state=SEED)
        by_ctx = defaultdict(list)
        for r in rt[rt["context"].isin(set(chosen))].itertuples():
            by_ctx[tuple(ruletaker_to_formal(r.context))].append((ruletaker_sentence(r.question), r.label == "entailment"))
        fc_t = bc_t = 0.0
        agree = total = fc_ok = bc_ok = fc_steps = bc_steps = 0
        for kb, qs in by_ctx.items():
            rules = [x for f in parse_kb(list(kb))[0] for x in formula_to_rules(f)]
            lits = [_literal_list(parse_formula(q))[0] for q, _ in qs]
            t0 = time.perf_counter(); fc = RuleBase(rules); fa = [fc.holds(l) for l in lits]; fc_t += time.perf_counter() - t0
            t0 = time.perf_counter(); bc = BackwardChainer(rules); ba = [bc.holds(l) for l in lits]; bc_t += time.perf_counter() - t0
            fc_steps += fc.steps; bc_steps += bc.steps
            for (q, g), a, b in zip(qs, fa, ba):
                total += 1; agree += a == b; fc_ok += a == g; bc_ok += b == g
        print(f"{total} questions, {len(by_ctx)} contexts (~{total / len(by_ctx):.1f} questions per context)")
        print(f"Forward chaining : accuracy {100 * fc_ok / total:.2f}%  time {fc_t:.2f}s  rule firings {fc_steps}")
        print(f"Backward chaining: accuracy {100 * bc_ok / total:.2f}%  time {bc_t:.2f}s  sub-goals {bc_steps}")
        print(f"Identical answers: {agree}/{total}")
    except Exception as e:
        print(f'Experiment 1 stopped with an error: {type(e).__name__}: {e}')
else:
    print('Skipped: Experiment 1 (RUN_EXPERIMENTS = False or data missing)')

7867 questions, 800 contexts (~9.8 questions per context)
Forward chaining : accuracy 99.94%  time 0.29s  rule firings 25787
Backward chaining: accuracy 100.00%  time 0.07s  sub-goals 4756
Identical answers: 7862/7867


In [30]:
if RUN_EXPERIMENTS and HAVE_RT and HAVE_FOLIO:
    try:
        # Experiment 2: the semantics is chosen on the TUNE splits; the report splits are shown for reference only
        rows = []
        for sem in ["cwa", "owa", "auto"]:
            row = {"semantics": sem}
            for col, items in [("RuleTaker tune", RT_TUNE_ITEMS[:6000]), ("FOLIO train", FOLIO_TRAIN),
                               ("RuleTaker test (report only)", RT_ITEMS[:6000]), ("FOLIO val (report only)", FOLIO_VAL)]:
                row[col] = round(evaluate(items, col, sem, verbose=False)[1], 2) if items else None
            rows.append(row)
        print(pd.DataFrame(rows).to_string(index=False))
        print("\nThe choice is made on the tune columns; the report columns are shown for reference only.")
    except Exception as e:
        print(f'Experiment 2 stopped with an error: {type(e).__name__}: {e}')
else:
    print('Skipped: Experiment 2 (RUN_EXPERIMENTS = False or data missing)')

semantics  RuleTaker tune  FOLIO train  RuleTaker test (report only)  FOLIO val (report only)
      cwa          100.00        80.82                        100.00                    85.22
      owa           75.53        81.22                         76.08                    85.22
     auto          100.00        81.12                        100.00                    85.22

The choice is made on the tune columns; the report columns are shown for reference only.


In [31]:
if RUN_EXPERIMENTS and HAVE_FOLIO and len(FOLIO_TRAIN) > 0:
    try:
        saved = dict(CONFIG)
        rows = []
        budgets = [("time 0.3 s (wall clock)", {"max_given": None, "time_limit": 0.3})] + \
                  [(f"{mg} given clauses", {"max_given": mg, "time_limit": 2.0}) for mg in (20, 50, 100, 200)]
        for label, cfg in budgets:
            CONFIG.update(saved); CONFIG.update(cfg)
            _, a_tr, s_tr = evaluate(FOLIO_TRAIN, "train", verbose=False)
            _, a_va, s_va = evaluate(FOLIO_VAL, "val", verbose=False)
            rows.append({"budget": label, "train acc (tune)": round(a_tr, 2), "val acc (report)": round(a_va, 2),
                         "train s": round(s_tr, 2), "val s": round(s_va, 2)})
        CONFIG.update(saved)
        print(pd.DataFrame(rows).to_string(index=False))

        print()
        rows = []
        for align in [False, True]:
            CONFIG["align_query"] = align
            _, a_tr, _ = evaluate(FOLIO_TRAIN, "train", verbose=False)
            _, a_va, _ = evaluate(FOLIO_VAL, "val", verbose=False)
            rows.append({"query alignment": align, "FOLIO train acc (tune)": round(a_tr, 2), "FOLIO val acc (report)": round(a_va, 2)})
        CONFIG.update(saved)
        print(pd.DataFrame(rows).to_string(index=False))

        print()
        rows = []
        for mode in ["forall", "auto"]:
            CONFIG["query_free_vars"] = mode
            _, a_tr, _ = evaluate(FOLIO_TRAIN, "train", verbose=False)
            _, a_va, _ = evaluate(FOLIO_VAL, "val", verbose=False)
            rows.append({"free variable in query": {"forall": "always ∀", "auto": "∃ for literal queries, ∀ for rules"}[mode],
                         "FOLIO train acc (tune)": round(a_tr, 2), "FOLIO val acc (report)": round(a_va, 2)})
        CONFIG.update(saved)
        print(pd.DataFrame(rows).to_string(index=False))
    except Exception as e:
        print(f'Experiment 3 stopped with an error: {type(e).__name__}: {e}')
else:
    print('Skipped: Experiment 3 (RUN_EXPERIMENTS = False or data missing)')

                 budget  train acc (tune)  val acc (report)  train s  val s
time 0.3 s (wall clock)             81.12             85.71    47.07  13.17
       20 given clauses             79.82             85.22     3.55   1.47
       50 given clauses             81.12             85.22     6.32   3.68
      100 given clauses             81.12             85.22    19.71   9.21
      200 given clauses             81.12             85.22    55.04  18.77

 query alignment  FOLIO train acc (tune)  FOLIO val acc (report)
           False                   80.52                   84.24
            True                   81.12                   85.22

            free variable in query  FOLIO train acc (tune)  FOLIO val acc (report)
                          always ∀                   80.92                   84.73
∃ for literal queries, ∀ for rules                   81.12                   85.22


In [32]:
if RUN_EXPERIMENTS and HAVE_RT and HAVE_FOLIO and HAVE_CB:
    try:
        _KEEP = {"and", "or", "not", "xor", "implies", "iff", "if", "then", "true", "false", "forall", "exists",
                 "all", "some", "every", "each"} | set("xyzuvw")
        _ASCII = [("∀", " forall "), ("∃", " exists "), ("¬", " ~"), ("∧", " & "), ("∨", " | "),
                  ("→", " -> "), ("⟷", " <-> "), ("↔", " <-> "), ("⊕", " xor ")]


        _CB_WORDS = {"We", "Would", "Assume", "Bern"}


        def rename_symbols(kb, query, seed, only_capitalized=False):
            """Give every predicate/constant a random name (consistently in KB and query); the logic is unchanged.
            only_capitalized: for CounterBench English, rename only the capitalised variable names, not template words."""
            rng, mapping = random.Random(seed), {}

            def sub(m):
                w = m.group(0)
                if w.lower() in _KEEP or (only_capitalized and (w[0].islower() or w in _CB_WORDS)):
                    return w
                k = w.lower()
                if k not in mapping:
                    mapping[k] = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(8))
                new = mapping[k]
                return new.capitalize() if w[0].isupper() else new
            pat = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")
            if isinstance(kb, str) and "We know that" in kb:
                return pat.sub(sub, kb), pat.sub(sub, query)
            return [pat.sub(sub, f) for f in split_kb(kb)], pat.sub(sub, query)


        def to_ascii(s):
            for a, b in _ASCII:
                s = s.replace(a, b)
            return s


        rows = []
        for name, items in [("RuleTaker", RT_ITEMS[:3000]), ("FOLIO val", FOLIO_VAL), ("CounterBench", CB_ITEMS)]:
            base, a0, _ = evaluate(items, name, verbose=False)
            variants = {
                "renamed symbols": [(*rename_symbols(kb, q, i, name == "CounterBench"), g, grp)
                                    for i, (kb, q, g, grp) in enumerate(items)],
                "ASCII syntax": [([to_ascii(f) for f in split_kb(kb)], to_ascii(q), g, grp) for kb, q, g, grp in items],
                "shuffled premises": [(random.Random(i).sample(split_kb(kb), len(split_kb(kb))), q, g, grp)
                                      for i, (kb, q, g, grp) in enumerate(items)],
            }
            row = {"dataset": name, "original": round(a0, 2)}
            for vname, vitems in variants.items():
                if name == "CounterBench" and vname != "renamed symbols":
                    continue                                   # English template: ASCII rewriting / shuffling does not apply
                res, a, _ = evaluate(vitems, vname, verbose=False)
                row[vname] = round(a, 2)
                row[vname + " (same answer)"] = f"{(res.pred.values == base.pred.values).sum()}/{len(res)}"
            rows.append(row)
        print(pd.DataFrame(rows).fillna("-").to_string(index=False))
    except Exception as e:
        print(f'Experiment 4 stopped with an error: {type(e).__name__}: {e}')
else:
    print('Skipped: Experiment 4 (RUN_EXPERIMENTS = False or data missing)')

     dataset  original  renamed symbols renamed symbols (same answer) ASCII syntax ASCII syntax (same answer) shuffled premises shuffled premises (same answer)
   RuleTaker    100.00           100.00                     3000/3000        100.0                  3000/3000             100.0                       3000/3000
   FOLIO val     85.22            84.24                       201/203        85.22                    203/203             85.22                         203/203
CounterBench     93.75            93.75                     1200/1200            -                          -                 -                               -


In [33]:
if RUN_EXPERIMENTS:
    try:
        import itertools as it
        R = random.Random(SEED)
        ATOMS = ["A", "B", "C", "D"]


        def rnd_formula(d):
            if d == 0 or R.random() < .25:
                a = ("atom", R.choice(ATOMS), ())
                return ("not", a) if R.random() < .3 else a
            op = R.choice(["and", "or", "imp", "iff", "xor", "not"])
            return ("not", rnd_formula(d - 1)) if op == "not" else (op, rnd_formula(d - 1), rnd_formula(d - 1))


        def truth(f, m):
            return _eval(f, m)                                   # Boolean evaluator of the SCM (Section 5e)


        def tt_entails(kb, q):
            """Brute force: check every model (2^4)."""
            for vals in it.product([False, True], repeat=len(ATOMS)):
                m = dict(zip(ATOMS, vals))
                if all(truth(f, m) for f in kb) and not truth(q, m):
                    return False
            return True


        agree = n = entailed = 0
        for _ in range(400):
            kb = [rnd_formula(3) for _ in range(R.randint(1, 3))]
            q = rnd_formula(3)
            if tt_entails(kb, ("false",)):
                continue                                         # contradictory KB: different policy (set of support), skipped
            gold = tt_entails(kb, q)
            got = predict([fml_str(f) for f in kb], fml_str(q), semantics="owa")
            n += 1; agree += got == gold; entailed += gold
        print(f"Router (owa) vs truth table: {agree}/{n} same  (entailed cases: {entailed})")
    except Exception as e:
        print(f'Experiment 5 stopped with an error: {type(e).__name__}: {e}')
else:
    print('Skipped: Experiment 5 (RUN_EXPERIMENTS = False or data missing)')

Router (owa) vs truth table: 346/346 same  (entailed cases: 79)


In [34]:
if RUN_EXPERIMENTS and HAVE_CB:
    try:
        from collections import defaultdict
        counts = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))      # qtype -> hypothesis -> split -> correct
        for _, r in counterbench.iterrows():
            try:
                eq, pri, ev = parse_causal_text(r["given_info"]); Q = parse_causal_question(r["question"]); scm = SCM(eq, pri)
            except ValueError:                                                     # cyclic graph: no hypothesis applies
                continue
            gold, sp = r["answer"] == "yes", r["split"]
            if Q["kind"] == "nested":
                (x, xv), (v, vv) = Q["x"], Q["v"]
                hyps = {"H1: do(X, V)": ({x: not xv}, [{x: xv, v: vv}]),
                        "H2: do(V) only": ({x: not xv}, [{v: vv}]),
                        "H3: V := V_{X=0}, X factual": ({x: not xv}, [lambda s, ro, d: {v: s.evaluate(ro, {x: xv}).get(v, vv)}])}
                for h, (fact, dos) in hyps.items():
                    p = scm.counterfactual(Q["target"], fact, dos)
                    counts["nested"][h][sp] += (p is not None and (p >= 0.5) == gold)
            elif r["type"] == "conditional":
                for h, (fe, ce) in {"observation in factual world": (dict(ev, **Q["factual"]), None),
                                    "observation in counterfactual world": (Q["factual"], ev)}.items():
                    p = scm.counterfactual(Q["target"], fe, Q["do"], ce)
                    counts["conditional"][h][sp] += (p is not None and (p >= 0.5) == gold)

        n_split = counterbench.groupby(["type", "split"]).size()
        rows = []
        for qtype, hyps in counts.items():
            tune = {h: 100 * c["tune"] / n_split[(qtype, "tune")] for h, c in hyps.items()}
            best = max(tune, key=tune.get)
            for h, c in hyps.items():
                rows.append({"type": qtype, "hypothesis": h, "tune acc": round(tune[h], 1),
                             "held-out acc": round(100 * c["heldout"] / n_split[(qtype, "heldout")], 1),
                             "chosen (best on tune)": "<--" if h == best else ""})
        print(pd.DataFrame(rows).to_string(index=False))
        print("\nThe chosen semantics is the one implemented in causal_answer (Section 5e); the held-out column is its unbiased estimate.")
    except Exception as e:
        print(f'Experiment 6 stopped with an error: {type(e).__name__}: {e}')
else:
    print('Skipped: Experiment 6 (RUN_EXPERIMENTS = False or data missing)')

       type                          hypothesis  tune acc  held-out acc chosen (best on tune)
conditional        observation in factual world      36.0          32.0                      
conditional observation in counterfactual world      92.0          80.0                   <--
     nested                        H1: do(X, V)      92.0          96.0                   <--
     nested                      H2: do(V) only      76.0          76.0                      
     nested         H3: V := V_{X=0}, X factual      60.0          72.0                      

The chosen semantics is the one implemented in causal_answer (Section 5e); the held-out column is its unbiased estimate.


In [35]:
if RUN_EXPERIMENTS and HAVE_CB:
    try:
        # The notation of causal models in the test data is not specified. Every CounterBench question is
        # rewritten in 7 equation styles x 6 query styles and run through the full pipeline (predict).
        def _asc(s):
            return s.replace("¬", "not ").replace("∧", " and ").replace("∨", " or ").replace("  ", " ")


        EQ_STYLES = {
            "V := e":     lambda v, e: f"{v} := {fml_str(e)}",
            "V ≔ e":      lambda v, e: f"{v} ≔ {fml_str(e)}",
            "V = e (words)": lambda v, e: f"{v} = {_asc(fml_str(e))}",
            "V := f(e)":  lambda v, e: f"{v} := f({fml_str(e)})",
            "V ↔ e":      lambda v, e: f"{v} ↔ {fml_str(e)}",
            "V <- e (words)": lambda v, e: f"{v} <- {_asc(fml_str(e))}",
            "e → V":      lambda v, e: f"({fml_str(e)}) → {v}",
        }
        _lit = lambda v, b: v if b else "¬" + v
        _asg = lambda v, b: f"{v}={int(b)}"
        Q_STYLES = {
            "do(¬X) → Y":     lambda y, do: "do(" + ", ".join(_lit(*i) for i in do) + f") → {y}",
            "P(Y=1|do(X=0))": lambda y, do: f"P({y}=1 | do(" + ", ".join(_asg(*i) for i in do) + "))",
            "Y_{¬X}":         lambda y, do: f"{y}_{{" + ", ".join(_lit(*i) for i in do) + "}",
            "¬X □→ Y":        lambda y, do: " ∧ ".join(_lit(*i) for i in do) + f" □→ {y}",
            "[do(¬X)] Y":     lambda y, do: "[do(" + ", ".join(_lit(*i) for i in do) + f")] {y}",
            "Y | do(X=0)":    lambda y, do: f"{y} | do(" + ", ".join(_asg(*i) for i in do) + ")",
        }
        OBS_STYLES = [lambda v, b: _lit(v, b), lambda v, b: f"{v} = {int(b)}"]           # 'Trune' / 'Trune = 1'
        PRIOR_STYLES = [lambda v, p: f"{v} ~ Bern({p})", lambda v, p: f"P({v}) = {p}"]

        cb_items = []
        for r in counterbench.itertuples():
            eq, pri, ev = parse_causal_text(r.given_info)
            Q = parse_causal_question(r.question)
            do = {Q["x"][0]: Q["x"][1], Q["v"][0]: Q["v"][1]} if Q["kind"] == "nested" else Q["do"][0]
            cb_items.append((r.answer == "yes", eq, pri, ev, Q["target"], list(do.items())))

        t0 = time.perf_counter()
        table = {}
        for i, (en, ef) in enumerate(EQ_STYLES.items()):
            for j, (qn, qf) in enumerate(Q_STYLES.items()):
                ok = 0
                for k, (gold, eq, pri, ev, y, do) in enumerate(cb_items):
                    obs, prior = OBS_STYLES[(i + j + k) % 2], PRIOR_STYLES[(i + j + k) % 2]
                    kb = [ef(v, e) for v, e in eq.items()] + [prior(v, p) for v, p in pri.items()] + \
                         [obs(v, b) for v, b in ev.items()]
                    ok += predict(kb, qf(y, do)) == gold
                table.setdefault(en, {})[qn] = round(100 * ok / len(cb_items), 2)
        print(pd.DataFrame(table).T.to_string())
        print(f"\n{len(EQ_STYLES) * len(Q_STYLES)} notations x {len(cb_items)} questions | {time.perf_counter() - t0:.1f}s")
        print("Example:", [EQ_STYLES['e → V'](v, e) for v, e in list(cb_items[0][1].items())[:2]],
              Q_STYLES["¬X □→ Y"](cb_items[0][4], cb_items[0][5]))
    except Exception as e:
        print(f'Experiment 7 stopped with an error: {type(e).__name__}: {e}')
else:
    print('Skipped: Experiment 7 (RUN_EXPERIMENTS = False or data missing)')

                do(¬X) → Y  P(Y=1|do(X=0))  Y_{¬X}  ¬X □→ Y  [do(¬X)] Y  Y | do(X=0)
V := e               93.75           93.75   93.75    93.75       93.75        93.75
V ≔ e                93.75           93.75   93.75    93.75       93.75        93.75
V = e (words)        93.75           93.75   93.75    93.75       93.75        93.75
V := f(e)            93.75           93.75   93.75    93.75       93.75        93.75
V ↔ e                93.75           93.75   93.75    93.75       93.75        93.75
V <- e (words)       93.75           93.75   93.75    93.75       93.75        93.75
e → V                93.75           93.75   93.75    93.75       93.75        93.75

42 notations x 1200 questions | 34.8s
Example: ['(Ziklo) → Blaf', '(Blaf) → Trune'] ¬Ziklo □→ Lumbo


In [36]:
if RUN_EXPERIMENTS:
    try:
        # Outside check on public logic data that was not used to tune any setting:
        #   ProofWriter OWA (test, 600): RuleTaker-style data under the open world (True / False / Unknown).
        #   ProverQA is used as the private test in Section 12 and is therefore not repeated here.
        import glob, json, os, urllib.request


        def load_proofwriter_owa():
            local = "export_data/proofwriter_owa_test.json"
            if os.path.exists(local):
                data = json.load(open(local))
            else:
                url = "https://raw.githubusercontent.com/opendatalab/ProverGen/main/logic_data/ProofWriter/test.json"
                data = json.loads(urllib.request.urlopen(url, timeout=60).read().decode("utf-8"))
            items = []
            for r in data:
                statement = r["question"].split("?", 1)[1].strip()
                items.append((ruletaker_to_formal(r["context"]), ruletaker_sentence(statement), r["answer"] == "A",
                              "ProofWriter OWA"))
            return items


        unseen = []
        for name, loader in [("ProofWriter OWA", load_proofwriter_owa)]:
            try:
                its = loader()
                unseen += its
                print(f"{name}: {len(its)} questions")
            except Exception as e:
                print(f"{name}: could not be loaded ({type(e).__name__}: {str(e)[:80]}), skipped")

        if unseen:
            saved = dict(CONFIG)
            runs = {}
            for label, sem, cfg in [("router (final)", None, {}),
                                    ("router, CWA on all symbols (before)", None, {"cwa_kb_vocab_only": False}),
                                    ("forced open world", "owa", {})]:
                CONFIG.update(saved); CONFIG.update(cfg)
                res, _, _ = evaluate(unseen, label, semantics=sem, verbose=False)
                res["ok"] = res.gold == res.pred
                runs[label] = res.groupby("group")["ok"].mean().mul(100).round(2)
            CONFIG.update(saved)
            table = pd.DataFrame(runs)
            table["n"] = pd.Series([g for *_, g in unseen]).value_counts()
            print(table.to_string())
    except Exception as e:
        print(f'Experiment 8 stopped with an error: {type(e).__name__}: {e}')
else:
    print('Skipped: Experiment 8 (RUN_EXPERIMENTS = False or data missing)')

ProofWriter OWA: 600 questions
                 router (final)  router, CWA on all symbols (before)  forced open world    n
group                                                                                       
ProofWriter OWA            84.5                                 83.5               99.5  600


## Section 11: Reasoning Traces
Forward chaining: derivation tree. Resolution: resolution steps up to □. Causal: abduction → action → prediction.

In [37]:
if HAVE_FOLIO and HAVE_CB:
    def show(kb, q, title):
        d = predict_detail(kb, q, want_trace=True)
        print(f"── {title}")
        print(f"KB    : {kb if isinstance(kb, str) else split_kb(kb)}")
        print(f"Query : {q}")
        print(f"Answer: {d['answer']}   [{d['engine']} | {d['status']}]")
        for line in d["trace"]:
            print("     ", line)
        print()


    def causal_trace(given, question):
        """Show Pearl's three steps for a counterfactual."""
        eq, pri, ev = parse_causal_text(given)
        Q = parse_causal_question(question)
        print("── CounterBench: counterfactual (Pearl's three steps)")
        print("Given   :", given)
        print("Question:", question)
        print("Equations:", {v: fml_str(f) for v, f in eq.items()}, "| priors:", pri, "| observed:", ev)
        scm = SCM(eq, pri)
        fact = Q["factual"] if Q["kind"] == "simple" else {Q["x"][0]: not Q["x"][1]} if Q["kind"] == "nested" else {}
        do = Q["do"][0] if Q["kind"] != "nested" else {Q["x"][0]: Q["x"][1], Q["v"][0]: Q["v"][1]}
        cf_mode = bool(ev) and all(v in scm.roots for v in do)      # conditional: observation in the counterfactual world
        for roots, w in scm.worlds(fact if cf_mode else dict(ev, **fact)):
            val = scm.evaluate(roots, do)
            keep = not cf_mode or all(val.get(k) == b for k, b in ev.items())
            print(f"  1) abduction  : roots {roots} (weight {w:.2f})")
            print(f"  2) action     : do{do}  (their equations are removed)")
            print(f"  3) prediction : {val}" + ("" if keep else "   <- inconsistent with the observation, world discarded"))
        ans, p = causal_predict(given, question)
        print(f"Answer: {'yes' if ans else 'no'} (P = {p})\n")


    show(["Young(bob)", "Kind(bob)", "∀x (Young(x) → Smart(x))", "∀x (Smart(x) ∧ Kind(x) → Successful(x))"],
         "Successful(bob)", "Assignment example: forward chaining proof tree")
    show(ruletaker_to_formal("Fiona is quiet. Bob is kind. Bob is young. Kind, young things are not smart. "
                             "If something is quiet and not smart then it is nice."),
         "Nice(fiona)", "RuleTaker style: negation as failure (closed world)")
    show(["∀x (Drinks(x) → Dependent(x))", "∀x (Drinks(x) ⊕ Jokes(x))", "∀x (Jokes(x) → ¬Unaware(x))",
          "Unaware(rina)"], "Dependent(rina)", "FOLIO style: resolution refutation (open world)")
    # first FOLIO validation question proved by resolution
    for kb, q, g, grp in FOLIO_VAL:
        d = predict_detail(kb, q)
        if d["engine"].startswith("resolution") and d["answer"] and g and 4 <= len(d["trace"]) <= 14:
            show(kb, q, f"FOLIO v2 validation question (gold: {grp})")
            break
    causal_trace(counterbench.iloc[0]["given_info"], counterbench.iloc[0]["question"])
    cond = counterbench[counterbench.type == "conditional"].iloc[0]
    causal_trace(cond["given_info"], cond["question"])
else:
    print('Skipped: reasoning traces on dataset items (data missing)')

── Assignment example: forward chaining proof tree
KB    : ['Young(bob)', 'Kind(bob)', '∀x (Young(x) → Smart(x))', '∀x (Smart(x) ∧ Kind(x) → Successful(x))']
Query : Successful(bob)
Answer: True   [forward-chaining (CWA) | closure]
      Successful(bob)   [rule 3: Smart(x) ∧ Kind(x) → Successful(x)]
          Smart(bob)   [rule 2: Young(x) → Smart(x)]
              Young(bob)   [given fact]
          Kind(bob)   [given fact]

── RuleTaker style: negation as failure (closed world)
KB    : ['Quiet(fiona)', 'Kind(bob)', 'Young(bob)', '∀x (Kind(x) ∧ Young(x) → ¬Smart(x))', '∀x (Quiet(x) ∧ ¬Smart(x) → Nice(x))']
Query : Nice(fiona)
Answer: True   [forward-chaining (CWA) | closure]
      Nice(fiona)   [rule 4: Quiet(x) ∧ ¬Smart(x) → Nice(x)]
          Quiet(fiona)   [given fact]
          ¬Smart(fiona)   [not derivable → true by negation-as-failure]

── FOLIO style: resolution refutation (open world)
KB    : ['∀x (Drinks(x) → Dependent(x))', '∀x (Drinks(x) ⊕ Jokes(x))', '∀x (Jokes(x) → ¬Unaw

## Section 12: Test Files → `submission.csv`
Two logic-only test files are answered and their accuracy is reported:
1. **Private test** `private_test.csv`. If the file is not present, it is built from ProverQA (dev: easy, medium,
   hard; first-order logic premises and conclusion, labels verified by the Prover9 prover), with the answers in
   `private_test_labels.csv`. ProverQA was used once during development as an outside check and revealed one routing
   error (the closed world now covers only the knowledge base's own vocabulary); no setting was tuned on it.
   Predictions → `submission.csv`.
2. **Public logic test** `public_test.csv`, built from the held-out public data (RuleTaker test, FOLIO validation,
   CounterBench held-out), answers in `public_test_labels.csv`. Predictions → `submission_public.csv`.

Without either file a built-in logic test is used. Every input is checked to be formal logic.

The loader reads CSV, JSON, JSONL or parquet and detects the column names (id: `id`, `example_id`, ...;
knowledge base: `context`, `premises-FOL`, `kb`, ...; query: `statement`, `conclusion-FOL`, `query`, ...;
an optional label column). Output: `id,label` with the label exactly `True` or `False`, one row per input row.
Every query is counted as proved, saturated, budget-out or fallback; fallbacks are printed with the reason.

In [38]:
# Test files (all inputs are formal logic):
#   private test : TEST_PATH. If it does not exist and USE_PROVERQA_AS_PRIVATE_TEST is True, it is built from
#                  ProverQA (public FOL data, labels verified by the Prover9 prover) with labels in PRIVATE_LABELS_PATH.
#   public test  : logic-only test from the held-out public data (RuleTaker test, FOLIO validation, CounterBench
#                  held-out), labels in PUBLIC_LABELS_PATH.
#   built-in test: used only if neither of the two is available.
# submission.csv holds the predictions for the private test (or, without one, for the next available test).
import glob

PUBLIC_TEST_RT_PER_CONFIG = 200      # RuleTaker questions per depth config in the public test

# (type, knowledge base, query, label). Labels: classical entailment, except "rule base" (closed world, as in RuleTaker).
BUILTIN_TEST = [
    ("rule base", ["Young(bob)", "Kind(bob)", "Quiet(anne)", "∀x (Young(x) → Smart(x))", "∀x (Smart(x) ∧ Kind(x) → Successful(x))"], "Successful(bob)", True),
    ("rule base", ["Young(bob)", "Kind(bob)", "Quiet(anne)", "∀x (Young(x) → Smart(x))", "∀x (Smart(x) ∧ Kind(x) → Successful(x))"], "Successful(anne)", False),
    ("rule base", ["Quiet(anne)", "Big(bob)", "∀x (Big(x) ∧ ¬Quiet(x) → Strong(x))"], "Strong(bob)", True),
    ("rule base", ["Quiet(anne)", "Big(bob)", "∀x (Big(x) ∧ ¬Quiet(x) → Strong(x))"], "¬Strong(anne)", True),
    ("rule base", ["Red(dave)", "Big(erin)", "Red(erin)", "∀x (Red(x) → Round(x))", "∀x (Round(x) ∧ ¬Big(x) → Nice(x))"], "Nice(erin)", False),
    ("rule base", ["Red(dave)", "Big(erin)", "Red(erin)", "∀x (Red(x) → Round(x))", "∀x (Round(x) ∧ ¬Big(x) → Nice(x))"], "Nice(dave)", True),
    ("rule base", ["Chases(cat, mouse)", "∀x (Chases(x, mouse) → Fast(x))", "∀x (Fast(x) → Visits(x, dog))"], "Visits(cat, dog)", True),
    ("rule base", ["∀x (Kind(x) ∧ Young(x) → ¬Smart(x))", "Kind(bob)", "Young(bob)", "Smart(dave)"], "Smart(bob)", False),
    ("first-order", ["∀x (Drinks(x) → Dependent(x))", "∀x (Drinks(x) ⊕ Jokes(x))", "∀x (Jokes(x) → ¬Unaware(x))", "Unaware(rina)"], "Dependent(rina)", True),
    ("first-order", ["∀x (Bird(x) → Fly(x))", "Bird(tweety)", "¬Fly(rex)"], "¬Bird(rex)", True),
    ("first-order", ["∀x (Bird(x) → Fly(x))", "Bird(tweety)", "¬Fly(rex)"], "Fly(polly)", False),
    ("first-order", ["∀x (Dog(x) → Brown(x) ∨ Black(x))", "Dog(rex)", "¬Brown(rex)"], "Black(rex)", True),
    ("first-order", ["∀x (Dog(x) → Brown(x) ∨ Black(x))", "Dog(rex)"], "Black(rex)", False),
    ("first-order", ["∀x ∃y Loves(x, y)", "∀x ∀y (Loves(x, y) → Happy(y))", "Person(ann)"], "∃z Happy(z)", True),
    ("first-order", ["∀x ∃y Loves(x, y)"], "∃y ∀x Loves(x, y)", False),
    ("first-order", ["∀x (Student(x) ↔ ¬Employed(x))", "Employed(sam)"], "¬Student(sam)", True),
    ("first-order", ["∃x (Cat(x) ∧ Black(x))", "∀x (Cat(x) → Pet(x))"], "∃x (Pet(x) ∧ Black(x))", True),
    ("first-order", ["Tall(a)", "a = b"], "Tall(b)", True),
    ("first-order", ["∀x (Human(x) → Mortal(x))", "Human(socrates)"], "Mortal(socrates) ∧ ¬Mortal(plato)", False),
    ("propositional", ["Rain → Wet", "Sprinkler → Wet", "Rain ∨ Sprinkler"], "Wet", True),
    ("propositional", ["Rain ⊕ Snow", "Snow"], "¬Rain", True),
    ("propositional", ["Rain → Wet"], "Wet → Rain", False),
    ("causal", ["Blaf := ¬Ziklo", "Vork := Blaf ∧ Trune", "Lumbo := Vork", "Ziklo", "Trune"], "do(¬Ziklo) → Lumbo", True),
    ("causal", ["Blaf := Ziklo", "Trune := Blaf", "Lumbo := Trune", "Ziklo"], "do(¬Ziklo) → Lumbo", False),
    ("causal", ["Gonk := Hupt ∨ Dask", "Crov := Gonk ∧ ¬Dask", "Hupt", "Dask"], "do(¬Dask) → Crov", True),
    ("causal", ["Gonk := Hupt ∨ Dask", "Crov := Gonk ∧ ¬Dask", "Hupt", "Dask"], "do(¬Hupt, ¬Dask) → Gonk", False),
]


def _pick(cols, names):
    low = {c.lower(): c for c in cols}
    for n in names:
        if n.lower() in low:
            return low[n.lower()]
    return None


def _to_bool(v):
    return str(v).strip().lower() in ("true", "1", "yes", "entailment", "proved", "t")


def load_test_file(path):
    """CSV / JSON / JSONL / parquet. Column names are detected automatically; a label column is kept if present."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        df = pd.read_csv(path)
    elif ext == ".jsonl":
        df = pd.read_json(path, lines=True)
    elif ext == ".json":
        try:
            df = pd.read_json(path)
        except ValueError:
            df = pd.read_json(path, lines=True)
    elif ext == ".parquet":
        df = pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")
    cols = list(df.columns)
    id_c = _pick(cols, ["id", "example_id", "question_id", "qid", "idx"])
    kb_c = _pick(cols, ["context", "premises-FOL", "premises_fol", "premises", "kb", "knowledge_base",
                        "theory", "facts_rules", "given_info", "facts"])
    q_c = _pick(cols, ["statement", "conclusion-FOL", "conclusion_fol", "conclusion", "query", "question",
                       "hypothesis", "goal"])
    lab_c = _pick(cols, ["label", "gold", "answer", "flag", "target"])
    if kb_c is None or q_c is None:
        raise ValueError(f"Knowledge-base / query columns not found: {cols}")
    print(f"Columns -> id: {id_c} | knowledge base: {kb_c} | query: {q_c} | label: {lab_c} | rows: {len(df)}")
    out = pd.DataFrame({"id": df[id_c].astype(str) if id_c else [str(i) for i in range(len(df))],
                        "kb": df[kb_c].map(lambda v: v if isinstance(v, str) else list(v)),
                        "query": df[q_c].astype(str)})
    if lab_c:
        out["gold"] = df[lab_c].map(_to_bool)
    return out


_LOGIC_MARK = re.compile(r"[()∀∃¬∧∨→↔⊕:=]")


def looks_like_logic(text):
    """Formal logic, not a natural-language sentence: logic symbols / brackets, or a single bare atom."""
    t = str(text).strip()
    return bool(_LOGIC_MARK.search(t)) or bool(re.fullmatch(r"[¬~]?\w+", t))


def write_test(rows, labels, test_path, labels_path):
    pd.DataFrame(rows, columns=["id", "context", "statement"]).to_csv(test_path, index=False)
    pd.DataFrame(labels, columns=["id", "label", "source"]).to_csv(labels_path, index=False)


def build_proverqa_test(test_path, labels_path, per_level=PROVERQA_PER_LEVEL):
    """ProverQA (dev: easy / medium / hard) -> logic-only test file + labels file."""
    from huggingface_hub import snapshot_download
    root = snapshot_download("opendatalab/ProverQA", repo_type="dataset")
    rng = random.Random(SEED)
    rows, labels = [], []
    for path in sorted(glob.glob(os.path.join(root, "**", "*.json"), recursive=True)):
        if "train" in os.path.relpath(path, root).lower():
            continue
        level = os.path.splitext(os.path.basename(path))[0]
        with open(path, encoding="utf-8") as fh:
            records = [r for r in json.load(fh) if r.get("nl2fol") and r.get("conclusion_fol")]
        for r in rng.sample(records, min(per_level, len(records))):
            fol = r["nl2fol"]
            premises = [p for p in (fol.values() if isinstance(fol, dict) else fol) if isinstance(p, str) and p.strip()]
            qid = f"pq{len(rows) + 1:04d}"
            rows.append({"id": qid, "context": json.dumps(premises, ensure_ascii=False),
                         "statement": r["conclusion_fol"]})
            gold = str(r["answer"]).strip().upper().startswith("A")      # A) True; B) False / C) Uncertain -> False
            labels.append({"id": qid, "label": "True" if gold else "False", "source": f"ProverQA {level}"})
    if not rows:
        raise RuntimeError(f"No ProverQA records with FOL found under {root}")
    write_test(rows, labels, test_path, labels_path)
    print(f"Private test built from ProverQA: {test_path} ({len(rows)} questions) + labels: {labels_path}")


def build_public_test(test_path, labels_path):
    """Held-out public data -> logic-only test file (id, context, statement) + separate labels file."""
    rows, labels = [], []

    def add(prefix, kb, q, gold, source):
        i = f"{prefix}{len(rows):05d}"
        kb = kb if isinstance(kb, list) else split_kb(kb)
        rows.append({"id": i, "context": json.dumps(kb, ensure_ascii=False), "statement": q})
        labels.append({"id": i, "label": "True" if gold else "False", "source": source})
    per_cfg = defaultdict(int)
    for kb, q, gold, cfg in RT_ITEMS:
        if per_cfg[cfg] < PUBLIC_TEST_RT_PER_CONFIG:
            per_cfg[cfg] += 1
            add("rt", kb, q, gold, "RuleTaker test")
    for kb, q, gold, _ in FOLIO_VAL:
        add("fo", kb, q, gold, "FOLIO validation")
    if HAVE_CB:
        for r in counterbench[counterbench.split == "heldout"].itertuples():
            kb, q = counterbench_to_formal(r.given_info, r.question)
            add("cb", kb, q, r.answer == "yes", "CounterBench held-out")
    write_test(rows, labels, test_path, labels_path)
    print(f"Public logic test built: {test_path} ({len(rows)} questions) + labels: {labels_path}")


def build_builtin_test(test_path, labels_path):
    rows = [{"id": f"b{i:03d}", "context": json.dumps(kb, ensure_ascii=False), "statement": q}
            for i, (_, kb, q, _) in enumerate(BUILTIN_TEST)]
    labels = [{"id": f"b{i:03d}", "label": str(lab), "source": typ} for i, (typ, _, _, lab) in enumerate(BUILTIN_TEST)]
    write_test(rows, labels, test_path, labels_path)
    print(f"Built-in logic test written to {test_path} ({len(rows)} questions)")


def outcome(d):
    s = d["status"].split()[0] if d["status"] else ""
    if d["engine"] == "fallback":
        return "fallback (input/engine error)"
    if s in ("step-limit", "timeout", "clause-limit"):
        return "budget-out (not proved)"
    if s == "saturated":
        return "saturated (not entailed)"
    return "proved / derived" if d["answer"] else "not derived"


def run_test_file(name, test_path, labels_path, out_path):
    """Predict every row of a logic test file, write id,label predictions, report the accuracy if labels exist."""
    print(f"\n========== {name}: {test_path}")
    df = load_test_file(test_path)
    kb_lines = df["kb"].map(lambda kb: kb if isinstance(kb, list) else split_kb(kb))
    non_logic = sum(not looks_like_logic(f) for fs in kb_lines for f in fs) + sum(not looks_like_logic(q) for q in df["query"])
    print(f"Input format check: {'all inputs are formal logic' if non_logic == 0 else f'{non_logic} inputs do not look like formal logic'}")
    _kb_cached.cache_clear()
    t0 = time.perf_counter()
    details = [predict_detail(r.kb, r.query) for r in df.itertuples()]
    secs = time.perf_counter() - t0
    preds = ["True" if d["answer"] else "False" for d in details]
    sub = pd.DataFrame({"id": df["id"], "label": preds})
    assert len(sub) == len(df), "every input row must have exactly one prediction"
    sub.to_csv(out_path, index=False)
    print(f"{out_path} written: {len(sub)} predictions | {secs:.2f}s ({1000 * secs / max(len(sub), 1):.2f} ms/query)")
    print("Engine :", pd.Series([d["engine"] for d in details]).value_counts().to_dict())
    print("Outcome:", pd.Series([outcome(d) for d in details]).value_counts().to_dict())
    for i, d in zip(df["id"], details):
        if d["engine"] == "fallback":
            print(f"  fallback {i}: {d['status']}")
    labels = None
    if labels_path and os.path.exists(labels_path):
        labels = pd.read_csv(labels_path, dtype={"id": str})
    elif "gold" in df:
        labels = pd.DataFrame({"id": df["id"], "label": df["gold"].map(str), "source": name})
    acc = None
    if labels is not None:
        m = labels.assign(id=labels["id"].astype(str)).merge(sub, on="id", suffixes=("_gold", "_pred"))
        m["ok"] = m["label_gold"].map(_to_bool) == m["label_pred"].map(_to_bool)
        acc = 100 * m.ok.mean()
        print(f"ACCURACY ({name}): {acc:.2f}% ({m.ok.sum()}/{len(m)})")
        if "source" in m:
            print(m.groupby("source").ok.agg(["mean", "count"]).assign(mean=lambda d: (100 * d["mean"]).round(2)).to_string())
    else:
        print("No labels available for this file: accuracy is computed by the evaluator.")
    return {"name": name, "file": test_path, "predictions": out_path, "n": len(sub), "seconds": secs, "accuracy": acc}


# ---- 1) private test (ProverQA if no private file is present) ----
private_labels = None
if not os.path.exists(TEST_PATH) and USE_PROVERQA_AS_PRIVATE_TEST:
    try:
        build_proverqa_test(TEST_PATH, PRIVATE_LABELS_PATH)
    except Exception as e:
        print(f"ProverQA could not be loaded ({type(e).__name__}: {str(e)[:100]})")
if os.path.exists(TEST_PATH):
    private_labels = next((p for p in (PRIVATE_LABELS_PATH, "proverqa_answer_key.csv") if os.path.exists(p)), None)

# ---- 2) public logic test from held-out public data ----
have_public = bool(RT_ITEMS or FOLIO_VAL or HAVE_CB)
if have_public:
    build_public_test(PUBLIC_TEST_PATH, PUBLIC_LABELS_PATH)

# ---- 3) run the test files ----
TEST_RUNS = []
if os.path.exists(TEST_PATH):
    TEST_RUNS.append(run_test_file("private test", TEST_PATH, private_labels, SUBMISSION_PATH))
if have_public:
    out = "submission_public.csv" if TEST_RUNS else SUBMISSION_PATH
    TEST_RUNS.append(run_test_file("public logic test", PUBLIC_TEST_PATH, PUBLIC_LABELS_PATH, out))
if not TEST_RUNS:
    build_builtin_test(PUBLIC_TEST_PATH, PUBLIC_LABELS_PATH)
    TEST_RUNS.append(run_test_file("built-in logic test", PUBLIC_TEST_PATH, PUBLIC_LABELS_PATH, SUBMISSION_PATH))
RESULTS["test runs"] = TEST_RUNS

print("\n========== Test files")
print(pd.DataFrame([{"test": r["name"], "file": r["file"], "questions": r["n"],
                     "accuracy": None if r["accuracy"] is None else round(r["accuracy"], 2),
                     "inference (s)": round(r["seconds"], 2), "predictions": r["predictions"]}
                    for r in TEST_RUNS]).to_string(index=False))
print(f"\n{SUBMISSION_PATH}:")
print(pd.read_csv(SUBMISSION_PATH).head().to_string(index=False))

Public logic test built: public_test.csv (2003 questions) + labels: public_test_labels.csv

========== private test: private_test.csv
Columns -> id: id | knowledge base: context | query: statement | label: None | rows: 300
Input format check: all inputs are formal logic
submission.csv written: 300 predictions | 0.66s (2.21 ms/query)
Engine : {'resolution (OWA)': 283, 'backward-chaining (CWA)': 11, 'forward-chaining (Horn, OWA)': 6}
Outcome: {'saturated (not entailed)': 196, 'proved / derived': 91, 'not derived': 12, 'budget-out (not proved)': 1}
ACCURACY (private test): 99.33% (298/300)
                  mean  count
source                       
ProverQA easy    100.0    100
ProverQA hard    100.0    100
ProverQA medium   98.0    100

========== public logic test: public_test.csv
Columns -> id: id | knowledge base: context | query: statement | label: None | rows: 2003
Input format check: all inputs are formal logic
submission_public.csv written: 2003 predictions | 6.71s (3.35 ms/query)

## Section 13: Final Summary

In [39]:
print("=" * 86)
print(f"{'Dataset':30s}{'Role':>8s}{'Accuracy':>10s}{'Queries':>9s}{'Time (s)':>10s}{'ms/query':>10s}")
print("-" * 86)
sets = [(k, v) for k, v in RESULTS.items() if isinstance(v, tuple) and len(v) == 4]
for k, (acc, n, secs, role) in sets:
    print(f"{k:30s}{role:>8s}{acc:9.2f}%{n:9d}{secs:10.2f}{1000 * secs / n:10.2f}")
rep = [v for _, v in sets if v[3] == "report"]
print("-" * 86)
if rep:
    n = sum(v[1] for v in rep)
    print(f"{'COMBINED ACCURACY (report sets)':38s}{100 * sum(v[0] * v[1] / 100 for v in rep) / n:9.2f}%{n:9d}"
          f"{sum(v[2] for v in rep):10.2f}")
    print(f"{'MACRO AVERAGE (report sets)':38s}{sum(v[0] for v in rep) / len(rep):9.2f}%   (equal weight per dataset)")
    print("(tune sets were used to choose settings and are not included in the combined accuracy)")
else:
    print("Public datasets were not loaded, so no public accuracy is reported.")
print("-" * 86)
for r in RESULTS.get("test runs", []):
    acc = "n/a (no labels)" if r["accuracy"] is None else f"{r['accuracy']:.2f}%"
    print(f"{r['name'] + ' (' + r['file'] + ')':48s} accuracy {acc:>10s} | {r['n']:5d} questions | "
          f"inference {r['seconds']:.2f}s -> {r['predictions']}")
print("=" * 86)

Dataset                           Role  Accuracy  Queries  Time (s)  ms/query
--------------------------------------------------------------------------------------
RuleTaker (test)                report   100.00%    18000     24.69      1.37
FOLIO v2 (validation)           report    85.22%      203      2.74     13.48
CounterBench (held-out)         report    93.33%      600      0.09      0.15
RuleTaker (tune)                  tune   100.00%     3000      4.58      1.53
FOLIO v2 (train)                  tune    81.12%     1001      7.29      7.28
CounterBench (tune)               tune    94.17%      600      0.09      0.15
--------------------------------------------------------------------------------------
COMBINED ACCURACY (report sets)           99.63%    18803     27.52
MACRO AVERAGE (report sets)               92.85%   (equal weight per dataset)
(tune sets were used to choose settings and are not included in the combined accuracy)
-----------------------------------------------